# IC-Arb Loss — BraTS 2020 · RT-DETR Detection + SAM Segmentation

**A complete, reproducible Google Colab research pipeline** for brain-tumor
detection and detection-guided segmentation.

- **Detector:** RT-DETR (Ultralytics) on 2D axial pseudo-RGB BraTS slices
- **Proposed loss:** *IoU–Coverage Arbitration Loss* (**IC-Arb**),
  `L = α·(1−IoU) + (1−α)·(1−C_p)` with prediction coverage `C_p = A_int / A_pred`
- **Segmenter:** SAM (ViT-B/L/H) prompted by detector boxes, with a cached
  ground-truth-box **upper-bound** reference
- **Plan:** a fixed **37-run** experiment registry (8 IoU baselines + 22 IC-Arb
  + 5 segmentation-loss baselines + 2 classification-loss ablations)
- **Storage:** Google Drive as durable state; Colab local disk as disposable cache

> **This notebook is fully self-contained.** All custom classes, functions,
> registries, tests, runners, evaluators and reporting live in these cells.
> It does **not** import any custom `.py` helper module. Restarting the runtime
> and reopening this `.ipynb` (with Drive mounted) is sufficient to continue.

### How to use (quick start)
1. Run **Section 00** and edit the configuration to your Drive paths / data source.
2. Run Sections **01–04** (mount, install, verify, stage dashboard).
3. Run Sections **05–11** once to build the dataset (persistent, reused later).
4. Run Sections **12–15** to install & *verify* the custom criterion.
5. Set `SMOKE_TEST = True` in Section 00 and run the smoke test (Section 28-ish).
6. Launch **Run 01** or all pending runs from the training runner (Section 17).

Detailed step-by-step guides are in the **README / Usage** section at the end.


## 00 · Configuration

There are **two cells**: a short **CONTROL PANEL** (the handful of settings you
actually change) and an **ADVANCED** cell (sensible defaults you can usually leave
alone). Edit the control panel, then run everything top-to-bottom.

**Data behaviour**
- `DATA_SOURCE = "kaggle"` → the dataset is downloaded **into this Colab session**
  (fast local disk) and processed from there. The *processed* slices are saved to
  your Drive, so a later session reuses them without re-downloading.
- `DATA_SOURCE = "drive"` → the notebook first **checks your Drive** for the
  dataset; if it is there it is reused, otherwise it downloads it (into Drive).

**First run vs. continue** — set `EXECUTION_MODE`. You never have to guess: run
`show_status_and_plan()` (Section 26) and it prints exactly what is done, what is
pending, and what the next launch will do.


In [1]:
# =========================================================================== #
#                          CONTROL PANEL  (edit these)                        #
# =========================================================================== #

# 1) WHERE IS THE DATA?
DATA_SOURCE = "kaggle"        # "kaggle" -> download into THIS Colab session
                              # "drive"  -> look in your Drive; download only if missing

# 2) YOUR DRIVE PROJECT FOLDER (all durable results are stored here)
PROJECT_ROOT = "/content/drive/MyDrive/icarb_brats_rtdetr_sam"

# 3) ONLY FOR DATA_SOURCE=="drive": where the extracted BraTS folder is / will be
DRIVE_DATASET_PATH = "/content/drive/MyDrive/datasets/brats2020"

# 4) ARE YOU STARTING FRESH OR CONTINUING?
EXECUTION_MODE = "auto"       # "auto"      -> detect automatically (recommended)
                              # "first_run" -> build dataset and start from the beginning
                              # "continue"  -> reuse everything done; resume/skip as needed

# 5) DO A TINY 3-PATIENT DRY RUN FIRST?  (recommended before the real 37 runs)
SMOKE_TEST = False

# 6) WHICH RUNS TO EXECUTE WHEN YOU LAUNCH THE PIPELINE
RUN_SELECTION = "all_pending" # "all_pending" | "single:5" | "range:9-20"

# =========================================================================== #
print("CONTROL PANEL:  data =", DATA_SOURCE, "| mode =", EXECUTION_MODE,
      "| smoke =", SMOKE_TEST, "| runs =", RUN_SELECTION)


CONTROL PANEL:  data = kaggle | mode = auto | smoke = False | runs = all_pending


In [2]:
# === 00.1 ADVANCED settings (defaults are fine for most users) =========== #
KAGGLE_DATASET = "awsaf49/brats20-dataset-training-validation"
DRIVE_MOUNT_POINT = "/content/drive"
# For DATA_SOURCE=="kaggle": raw data is downloaded/extracted HERE (Colab session, disposable).
RAW_LOCAL_DIR = "/content/brats_raw"
DATASET_ARCHIVE_PATH = ""     # optional path to a pre-downloaded .zip
LOCAL_SCRATCH = "/content/icarb_local_cache"
PRIMARY_SELECTION_METRIC = "val_map50_95"
SAM_EVALUATION_POLICY = "deferred_all_runs"   # "all_runs"|"selected_runs"|"deferred_all_runs"

# --- Derive runner controls from the control panel (do not edit) ----------
# Kept in sync with Section 16.1 (which re-exports the authoritative TOTAL_RUNS).
TOTAL_RUNS = 37

def _parse_run_selection(sel):
    sel = str(sel).strip()
    if sel.startswith("single:"):
        return "single", int(sel.split(":")[1]), [1, TOTAL_RUNS]
    if sel.startswith("range:"):
        a, b = sel.split(":")[1].split("-"); return "range", 1, [int(a), int(b)]
    return "all_pending", 1, [1, TOTAL_RUNS]

RUN_MODE, SELECTED_RUN_ID, RUN_RANGE = _parse_run_selection(RUN_SELECTION)

# --- Force / reuse flags (advanced; EXECUTION_MODE sets sane defaults) -----
FORCE_DATASET_DOWNLOAD = False
FORCE_DATASET_EXTRACTION = False
FORCE_REBUILD_PATIENT_SPLIT = False
FORCE_REBUILD_PREPROCESSED_DATA = False
FORCE_REVALIDATE_DATASET = False
REUSE_EXISTING_DRIVE_ARTIFACTS = True
FAST_REUSE_VALIDATION = True

# EXECUTION_MODE maps to resume / rerun behaviour:
#   first_run -> do not resume half-finished trainings, just run the selection
#   continue  -> resume interrupted trainings and skip completed ones
#   auto      -> resume if available; state is auto-detected and reported
RESUME_IF_AVAILABLE = (EXECUTION_MODE != "first_run")
FORCE_RERUN = False

# --- Per-stage execution mode: "auto" | "run" | "skip" ---------------------
# "auto" reuses a verified artifact or runs the stage; this is what you want.
STAGE_MODE = {
    "dataset_download":   "auto", "dataset_extract":    "auto",
    "patient_discovery":  "auto", "patient_split":      "auto",
    "preprocessing":      "auto", "slice_generation":   "auto",
    "dataset_integrity":  "auto", "gt_box_sam_cache":   "auto",
    "training":           "run",  "detection_evaluation": "auto",
    "sam_evaluation":     "auto", "reporting":          "auto",
}
print(f"Advanced config resolved. RUN_MODE={RUN_MODE} SELECTED={SELECTED_RUN_ID} "
      f"RANGE={RUN_RANGE} RESUME={RESUME_IF_AVAILABLE}")


Advanced config resolved. RUN_MODE=all_pending SELECTED=1 RANGE=[1, 37] RESUME=True


In [3]:
# === 00.2 Structured configuration models (dataclasses, fully serializable) #
from __future__ import annotations
import dataclasses, json, hashlib, os
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Any, Optional, Tuple

def _sha256_of_obj(obj) -> str:
    payload = json.dumps(obj, sort_keys=True, default=str).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

@dataclass
class PathConfig:
    project_root: str = PROJECT_ROOT
    drive_mount: str = DRIVE_MOUNT_POINT
    local_scratch: str = LOCAL_SCRATCH
    def __post_init__(self):
        r = self.project_root
        self.data_dir = os.path.join(r, "data")
        self.raw_dir = os.path.join(self.data_dir, "raw")
        self.processed_dir = os.path.join(self.data_dir, "processed")   # Drive: metadata + durable media archive
        self.metadata_dir = os.path.join(self.processed_dir, "metadata")
        # Durable single-file media archive on Drive (Drive reads/writes ONE big file fast,
        # unlike 36k small PNGs which are latency-bound over Drive FUSE).
        self.media_archive = os.path.join(self.processed_dir, "media_archive.tar")
        # Media (images/labels/masks) live on LOCAL disk for fast training/eval I/O.
        # They are archived to Drive after generation and restored from the archive on reopen.
        self.local_media = os.path.join(self.local_scratch, "media")
        self.images_dir = os.path.join(self.local_media, "images")
        self.labels_dir = os.path.join(self.local_media, "labels")
        self.masks_dir = os.path.join(self.local_media, "masks")
        self.dataset_yaml = os.path.join(self.local_media, "dataset.yaml")
        self.stage_manifests = os.path.join(r, "stage_manifests")
        self.experiments = os.path.join(r, "experiments")
        self.experiments_repeated = os.path.join(r, "experiments_repeated")
        self.registry = os.path.join(r, "registry")
        self.sam_shared = os.path.join(r, "sam_evaluation")
        self.reports = os.path.join(r, "reports")
        self.figures = os.path.join(r, "figures")
        self.snapshots = os.path.join(r, "notebook_snapshots")
        self.shared = os.path.join(r, "shared_artifacts")

@dataclass
class DatasetConfig:
    source: str = DATA_SOURCE
    kaggle_dataset: str = KAGGLE_DATASET
    drive_dataset_path: str = DRIVE_DATASET_PATH
    archive_path: str = DATASET_ARCHIVE_PATH
    # BraTS2020 training root folder name inside the archive
    brats_subdir: str = "BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData"
    modalities: Tuple[str, ...] = ("flair", "t1ce", "t2", "seg")
    channel_map: Tuple[str, str, str] = ("flair", "t1ce", "t2")   # R,G,B
    split_train: float = 0.70
    split_val: float = 0.15
    split_test: float = 0.15
    split_seed: int = 1337
    stratify: bool = False

@dataclass
class PreprocessConfig:
    normalization: str = "zscore"        # zscore|percentile_minmax|robust_zscore|none
    clip_low_pct: float = 1.0
    clip_high_pct: float = 99.0
    plane: str = "axial"                 # axial (default)
    min_brain_pixels: int = 500
    min_tumor_pixels: int = 10           # positive threshold
    flag_tiny_positive: bool = True      # flag, never silently relabel
    neg_to_pos_ratio: float = 0.5
    hard_negative_band: int = 3          # neighbouring negatives around tumor range
    negative_seed: int = 1337

@dataclass
class AugmentConfig:
    hsv: bool = False                    # DISABLED (channels are MRI modalities)
    mosaic: float = 0.0                  # disabled by default
    fliplr: float = 0.5
    degrees: float = 5.0
    translate: float = 0.05
    scale: float = 0.1
    intensity_jitter: float = 0.0        # mild MRI-appropriate; 0 by default

@dataclass
class ModelConfig:
    variant: str = "rtdetr-l.pt"         # Ultralytics RT-DETR-L
    pretrained: bool = True
    imgsz: int = 640
    num_classes: int = 1                 # whole-tumor single class

@dataclass
class TrainConfig:
    optimizer: str = "AdamW"
    lr0: float = 0.0005
    weight_decay: float = 0.0001
    batch: int = 12
    epochs: int = 100
    patience: int = 50
    amp: bool = True
    grad_clip: float = 0.0               # 0 -> Ultralytics default
    grad_accumulate: int = 1
    workers: int = 2
    cache: bool = False
    seed: int = 1337
    #seed: int = 62
    # Loss coefficients frozen for the primary comparison
    overlap_loss_weight: float = 3.0
    l1_loss_weight: float = 5.0          # Ultralytics DETR default bbox gain
    cls_loss_weight: float = 1.0

@dataclass
class SAMConfig:
    model_type: str = "vit_b"            # vit_b|vit_l|vit_h
    checkpoint_name: str = "sam_vit_b_01ec64.pth"
    checkpoint_url: str = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"
    device: str = "cuda"
    multimask_output: bool = False

@dataclass
class EvalConfig:
    conf_threshold: float = 0.25         # frozen from validation for fair comparison
    iou_match_threshold: float = 0.50
    nms_iou: float = 0.7
    orig_slice_hw: Tuple[int, int] = (240, 240)

@dataclass
class SmokeConfig:
    enabled: bool = SMOKE_TEST
    n_patients: int = 3
    epochs: int = 1

@dataclass
class MasterConfig:
    paths: PathConfig = field(default_factory=PathConfig)
    dataset: DatasetConfig = field(default_factory=DatasetConfig)
    preprocess: PreprocessConfig = field(default_factory=PreprocessConfig)
    augment: AugmentConfig = field(default_factory=AugmentConfig)
    model: ModelConfig = field(default_factory=ModelConfig)
    train: TrainConfig = field(default_factory=TrainConfig)
    sam: SAMConfig = field(default_factory=SAMConfig)
    evaluation: EvalConfig = field(default_factory=EvalConfig)
    smoke: SmokeConfig = field(default_factory=SmokeConfig)

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)
    def to_json(self, indent=2) -> str:
        return json.dumps(self.to_dict(), indent=indent, default=str)
    def fingerprint(self) -> str:
        return _sha256_of_obj(self.to_dict())

CFG = MasterConfig()
print("MasterConfig built. fingerprint =", CFG.fingerprint()[:16])
print(CFG.to_json()[:600], "...")


MasterConfig built. fingerprint = da1287357effe4b7
{
  "paths": {
    "project_root": "/content/drive/MyDrive/icarb_brats_rtdetr_sam",
    "drive_mount": "/content/drive",
    "local_scratch": "/content/icarb_local_cache"
  },
  "dataset": {
    "source": "kaggle",
    "kaggle_dataset": "awsaf49/brats20-dataset-training-validation",
    "drive_dataset_path": "/content/drive/MyDrive/datasets/brats2020",
    "archive_path": "",
    "brats_subdir": "BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData",
    "modalities": [
      "flair",
      "t1ce",
      "t2",
      "seg"
    ],
    "channel_map": [
      "flair",
      "t1ce",
      "t2"
     ...


## 01 · Mount Google Drive

Drive is the durable store. Local Colab disk is disposable cache.


In [4]:
import os, sys, time
IN_COLAB = "google.colab" in sys.modules

def mount_drive():
    if not IN_COLAB:
        print("[drive] Not running in Colab -> skipping mount. PROJECT_ROOT will be used as-is.")
        return False
    from google.colab import drive
    if not os.path.ismount(CFG.paths.drive_mount):
        drive.mount(CFG.paths.drive_mount, force_remount=False)
    ok = os.path.ismount(CFG.paths.drive_mount) or os.path.isdir(CFG.paths.drive_mount)
    print(f"[drive] mounted at {CFG.paths.drive_mount}: {ok}")
    return ok

DRIVE_OK = mount_drive()

# Create the durable project skeleton (idempotent).
def ensure_project_tree():
    p = CFG.paths
    for d in [p.project_root, p.data_dir, p.raw_dir, p.processed_dir, p.images_dir,
              p.labels_dir, p.masks_dir, p.metadata_dir, p.stage_manifests,
              p.experiments, p.experiments_repeated, p.registry, p.sam_shared,
              p.reports, p.figures, p.snapshots, p.shared, p.local_scratch]:
        os.makedirs(d, exist_ok=True)
    for split in ("train", "val", "test"):
        for base in (p.images_dir, p.labels_dir, p.masks_dir):
            os.makedirs(os.path.join(base, split), exist_ok=True)
    print("[tree] project skeleton ready under", p.project_root)

ensure_project_tree()


Mounted at /content/drive
[drive] mounted at /content/drive: True
[tree] project skeleton ready under /content/drive/MyDrive/icarb_brats_rtdetr_sam


## 02 · Install dependencies

Standard third-party packages only. Pinned where it matters for reproducibility.


In [5]:
# Idempotent install. On Colab the base image already ships torch/cv2/numpy.
REQUIRED = {
    "ultralytics": "ultralytics>=8.1.0",
    "nibabel": "nibabel",
    "segment_anything": "git+https://github.com/facebookresearch/segment-anything.git",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "pandas": "pandas",
    "openpyxl": "openpyxl",            # xlsx export
    "tqdm": "tqdm",
    "psutil": "psutil",
}

def _pip_install(spec):
    import subprocess
    print(f"[pip] installing {spec} ...", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", spec], check=False)

def ensure_deps():
    import importlib
    for modname, spec in REQUIRED.items():
        try:
            importlib.import_module(modname if modname != "sklearn" else "sklearn")
        except Exception:
            _pip_install(spec)
    # kaggle only needed when downloading from kaggle
    if CFG.dataset.source == "kaggle":
        try:
            import kaggle  # noqa
        except Exception:
            _pip_install("kaggle")
    print("[pip] dependency check complete.", flush=True)

if not SMOKE_TEST or IN_COLAB:
    ensure_deps()
else:
    print("[pip] local dry mode: skipping installs (assume environment prepared).")


[pip] installing ultralytics>=8.1.0 ...
[pip] installing git+https://github.com/facebookresearch/segment-anything.git ...
You must authenticate before you can call the Kaggle API.
Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication
[pip] installing kaggle ...
[pip] dependency check complete.


## 03 · Environment verification

Print and record every version the study depends on (Section 2, item 1).


In [6]:
import platform, json, subprocess

def _try_ver(import_name, attr="__version__"):
    try:
        m = __import__(import_name)
        return getattr(m, attr, "unknown")
    except Exception as e:
        return f"MISSING ({type(e).__name__})"

def collect_environment() -> dict:
    env = {}
    env["python"] = platform.python_version()
    env["platform"] = platform.platform()
    try:
        import torch
        env["torch"] = torch.__version__
        env["cuda_available"] = torch.cuda.is_available()
        env["cuda"] = torch.version.cuda
        env["cudnn"] = torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else None
        if torch.cuda.is_available():
            env["gpu_name"] = torch.cuda.get_device_name(0)
            env["gpu_mem_gb"] = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
    except Exception as e:
        env["torch"] = f"MISSING ({e})"
    env["torchvision"] = _try_ver("torchvision")
    env["ultralytics"] = _try_ver("ultralytics")
    env["numpy"] = _try_ver("numpy")
    env["cv2"] = _try_ver("cv2")
    env["nibabel"] = _try_ver("nibabel")
    env["pandas"] = _try_ver("pandas")
    env["scipy"] = _try_ver("scipy")
    try:
        import sklearn; env["scikit_learn"] = sklearn.__version__
    except Exception as e:
        env["scikit_learn"] = f"MISSING ({e})"
    try:
        import segment_anything; env["segment_anything"] = getattr(segment_anything, "__version__", "installed")
    except Exception as e:
        env["segment_anything"] = f"MISSING ({e})"
    return env

ENVIRONMENT = collect_environment()
print(json.dumps(ENVIRONMENT, indent=2))


Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
{
  "python": "3.13.15",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "torch": "2.11.0+cu128",
  "cuda_available": true,
  "cuda": "12.8",
  "cudnn": 91900,
  "gpu_name": "NVIDIA A100-SXM4-40GB",
  "gpu_mem_gb": 42.41,
  "torchvision": "0.26.0+cu128",
  "ultralytics": "8.4.131",
  "numpy": "2.1.3",
  "cv2": "4.14.0",
  "nibabel": "5.4.2",
  "pandas": "2.2.3",
  "scipy": "1.16.3",
  "scikit_learn": "1.6.1",
  "segment_anything": "installed"
}


## 04 · Persistent stage manager, atomic I/O, observability

The pipeline has expensive one-time stages (download, extract, split, preprocess,
slice, integrity, GT-box SAM cache). Each is a **verified, independently
controllable** stage with a completion flag *and* a manifest. A flag without a
valid matching manifest is never trusted. Writes are atomic (temp + rename).


In [7]:
# === 04.1 Atomic I/O + hashing utilities ================================= #
import os, json, tempfile, hashlib, time, gc

def atomic_write_text(path: str, text: str):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    d = os.path.dirname(path) or "."
    fd, tmp = tempfile.mkstemp(dir=d, suffix=".tmp")
    try:
        with os.fdopen(fd, "w") as f:
            f.write(text)
            f.flush(); os.fsync(f.fileno())
        os.replace(tmp, path)
    finally:
        if os.path.exists(tmp):
            os.remove(tmp)

def atomic_write_json(path: str, obj):
    atomic_write_text(path, json.dumps(obj, indent=2, default=str))

def read_json(path: str, default=None):
    try:
        with open(path) as f:
            return json.load(f)
    except Exception:
        return default

def sha256_file(path: str, chunk=1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def touch_flag(path: str):
    atomic_write_text(path, f"done at {time.strftime('%Y-%m-%d %H:%M:%S')}\n")

print("[io] atomic I/O helpers ready.")


[io] atomic I/O helpers ready.


In [8]:
# === 04.2 Observability: heartbeat, timers, resource summary ============= #
import time, gc

def now_hms():
    return time.strftime("%H:%M:%S")

def fmt_dur(seconds: float) -> str:
    seconds = int(max(0, seconds))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

def resource_summary() -> str:
    parts = []
    try:
        import psutil
        vm = psutil.virtual_memory()
        parts.append(f"RAM={vm.used/1e9:.1f}/{vm.total/1e9:.1f}GB")
    except Exception:
        pass
    try:
        import torch
        if torch.cuda.is_available():
            parts.append(f"GPU={torch.cuda.memory_allocated()/1e9:.1f}/"
                         f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")
    except Exception:
        pass
    return " | ".join(parts)

class Heartbeat:
    """Emit a status line every `every` seconds regardless of inner progress."""
    def __init__(self, stage: str, every: float = 15.0, log_path: str = None):
        self.stage = stage; self.every = every; self.log_path = log_path
        self.t0 = time.time(); self.last = 0.0
    def beat(self, **fields):
        t = time.time()
        if t - self.last < self.every:
            return
        self.last = t
        detail = " | ".join(f"{k}={v}" for k, v in fields.items())
        line = (f"[{now_hms()}] Stage={self.stage} | {detail} | "
                f"Elapsed={fmt_dur(t - self.t0)} | {resource_summary()}")
        print(line, flush=True)
        if self.log_path:
            try:
                with open(self.log_path, "a") as f:
                    f.write(line + "\n")
            except Exception:
                pass

class Timer:
    def __init__(self, name): self.name = name
    def __enter__(self):
        self.t0 = time.time()
        print(f"[{now_hms()}] >>> START {self.name}", flush=True)
        return self
    def __exit__(self, *a):
        self.dt = time.time() - self.t0
        print(f"[{now_hms()}] <<< DONE  {self.name} in {fmt_dur(self.dt)} | {resource_summary()}", flush=True)

def free_memory():
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass

print("[obs] observability helpers ready. Sample:", now_hms(), "|", resource_summary())


[obs] observability helpers ready. Sample: 14:54:53 | RAM=2.2/89.6GB | GPU=0.0/42.4GB


In [9]:
# === 04.3 Stage manager: modes, flags, manifests, verification =========== #
STAGE_FLAG_NAMES = {
    "dataset_download":   "DATASET_DOWNLOAD_DONE.flag",
    "dataset_extract":    "DATASET_EXTRACTION_DONE.flag",
    "patient_discovery":  "PATIENT_DISCOVERY_DONE.flag",
    "patient_split":      "PATIENT_SPLIT_DONE.flag",
    "preprocessing":      "PREPROCESSING_DONE.flag",
    "slice_generation":   "SLICE_GENERATION_DONE.flag",
    "dataset_integrity":  "DATASET_INTEGRITY_DONE.flag",
    "gt_box_sam_cache":   "GT_BOX_SAM_CACHE_DONE.flag",
}

class StageManager:
    """Central controller for persistent stages.

    Each stage owns a flag file and a manifest JSON under stage_manifests/.
    A stage is 'valid' only if BOTH the flag and a parseable manifest exist and
    the manifest's declared outputs are present. `decide()` maps the requested
    mode ('auto'|'run'|'skip') + validity into a resolved action.
    """
    def __init__(self, cfg, stage_mode: dict):
        self.cfg = cfg
        self.mode = dict(stage_mode)
        self.mdir = cfg.paths.stage_manifests
        os.makedirs(self.mdir, exist_ok=True)

    def flag_path(self, stage):
        return os.path.join(self.mdir, STAGE_FLAG_NAMES.get(stage, f"{stage}_DONE.flag"))
    def manifest_path(self, stage):
        return os.path.join(self.mdir, f"{stage}_manifest.json")

    def is_valid(self, stage) -> (bool, str):
        if not os.path.exists(self.flag_path(stage)):
            return False, "no completion flag"
        man = read_json(self.manifest_path(stage))
        if man is None:
            return False, "manifest missing or unparseable"
        for out in man.get("outputs", []):
            if not os.path.exists(out):
                return False, f"declared output missing: {out}"
        return True, "flag+manifest+outputs present"

    def decide(self, stage) -> dict:
        mode = self.mode.get(stage, "auto")
        valid, reason = self.is_valid(stage)
        if mode == "run":
            action = "run"; why = "mode=run (forced execute/rebuild)"
        elif mode == "skip":
            if valid:
                action = "reuse"; why = "mode=skip and artifact verified"
            else:
                action = "block"; why = f"mode=skip but artifact INVALID: {reason}"
        else:  # auto
            if valid and REUSE_EXISTING_DRIVE_ARTIFACTS:
                action = "reuse"; why = f"mode=auto, valid artifact ({reason})"
            else:
                action = "run"; why = f"mode=auto, must execute ({reason})"
        return {"stage": stage, "mode": mode, "action": action, "reason": why,
                "valid": valid, "flag": self.flag_path(stage),
                "manifest": self.manifest_path(stage)}

    def complete(self, stage, config: dict, inputs: list, outputs: list,
                 extra: dict = None, hashes: dict = None):
        """Write manifest atomically, then the flag. Order matters: flag last."""
        man = {
            "stage": stage,
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "config": config,
            "inputs": inputs,
            "outputs": outputs,
            "hashes": hashes or {},
            "validation": "ok",
        }
        if extra:
            man.update(extra)
        atomic_write_json(self.manifest_path(stage), man)
        touch_flag(self.flag_path(stage))
        return man

STAGE = StageManager(CFG, STAGE_MODE)
print("[stage] StageManager ready. Stages:", list(STAGE_FLAG_NAMES))


[stage] StageManager ready. Stages: ['dataset_download', 'dataset_extract', 'patient_discovery', 'patient_split', 'preprocessing', 'slice_generation', 'dataset_integrity', 'gt_box_sam_cache']


In [10]:
# === 04.4 Stage-status dashboard ========================================= #
def stage_status_table():
    rows = []
    for stage in STAGE_FLAG_NAMES:
        d = STAGE.decide(stage)
        man = read_json(STAGE.manifest_path(stage)) or {}
        rows.append({
            "stage": stage,
            "requested": d["mode"],
            "resolved": d["action"].upper(),
            "valid": "yes" if d["valid"] else "no",
            "last_done": man.get("timestamp", "-"),
            "reason": d["reason"],
        })
    return rows

def print_stage_dashboard():
    rows = stage_status_table()
    hdr = f'{"STAGE":<20}{"REQ":<7}{"RESOLVED":<10}{"VALID":<7}{"LAST DONE":<21}REASON'
    print("=" * len(hdr)); print("STAGE STATUS DASHBOARD"); print("=" * len(hdr))
    print(hdr); print("-" * len(hdr))
    for r in rows:
        print(f'{r["stage"]:<20}{r["requested"]:<7}{r["resolved"]:<10}'
              f'{r["valid"]:<7}{r["last_done"]:<21}{r["reason"]}')
    print("=" * len(hdr))

print_stage_dashboard()


STAGE STATUS DASHBOARD
STAGE               REQ    RESOLVED  VALID  LAST DONE            REASON
-----------------------------------------------------------------------
dataset_download    auto   RUN       no     2026-08-27 18:27:06  mode=auto, must execute (declared output missing: /content/brats_raw/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData)
dataset_extract     auto   RUN       no     2026-08-27 18:27:08  mode=auto, must execute (declared output missing: /content/brats_raw/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData)
patient_discovery   auto   REUSE     yes    2026-08-27 18:27:09  mode=auto, valid artifact (flag+manifest+outputs present)
patient_split       auto   REUSE     yes    2026-07-24 08:32:29  mode=auto, valid artifact (flag+manifest+outputs present)
preprocessing       auto   REUSE     yes    2026-07-24 11:46:20  mode=auto, valid artifact (flag+manifest+outputs present)
slice_generation    auto   REUSE     yes    2026-07-24 11:46:20  mode=auto, valid artifa

## 05 · Dataset acquisition — one-time, source-aware

- **`DATA_SOURCE="kaggle"`** → the dataset is downloaded + extracted into this
  **Colab session** (`RAW_LOCAL_DIR`, fast disposable disk) and processed there.
  Because the *processed* slices are saved to Drive, a later session does **not**
  re-download (preprocessing is reused by fingerprint).
- **`DATA_SOURCE="drive"`** → the notebook first **checks Drive**
  (`DRIVE_DATASET_PATH`) for the extracted tree or an archive; if present it is
  reused/extracted, otherwise it downloads from Kaggle **into Drive** so it
  persists.

Acquisition is verified by flag + manifest + on-disk inventory before reuse; a
skipped stage never means "assume success". Only BraTS-2020 **training** patients
(with segmentation masks) are used.


In [11]:
import glob, shutil, zipfile, subprocess

def _search_roots():
    """Directories that might contain BraTS20_Training_xxx/, in priority order."""
    if CFG.dataset.source == "kaggle":
        bases = [RAW_LOCAL_DIR, CFG.paths.raw_dir]
    else:  # drive
        bases = [CFG.dataset.drive_dataset_path, CFG.paths.raw_dir]
    out = []
    for b in bases:
        if not b:
            continue
        out += [os.path.join(b, CFG.dataset.brats_subdir), b]
    return out

def _brats_root():
    """Resolve the folder that directly contains BraTS20_Training_xxx/ subdirs."""
    for c in _search_roots():
        if c and os.path.isdir(c) and glob.glob(os.path.join(c, "BraTS20_Training_*")):
            return c
    return None

def _count_patient_dirs(root):
    return len(glob.glob(os.path.join(root, "BraTS20_Training_*"))) if root else 0

def _kaggle_download(dest):
    os.makedirs(dest, exist_ok=True)
    print(f"[05] Downloading BraTS from Kaggle into {dest} (needs ~/.kaggle/kaggle.json)...", flush=True)
    r = subprocess.run(["kaggle", "datasets", "download", "-d", CFG.dataset.kaggle_dataset,
                        "-p", dest, "--unzip"], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-800:]); print(r.stderr[-800:])
        raise RuntimeError("Kaggle download failed. Upload kaggle.json to ~/.kaggle/ (chmod 600).")

def _extract_archive(archive, dest):
    print(f"[05] Extracting {archive} -> {dest}", flush=True)
    os.makedirs(dest, exist_ok=True)
    with zipfile.ZipFile(archive) as zf:
        hb = Heartbeat("dataset_extract", every=8)
        members = zf.namelist()
        for i, m in enumerate(members):
            zf.extract(m, dest); hb.beat(file=f"{i+1}/{len(members)}")

def stage_acquire_dataset():
    """Make the raw BraTS tree available according to DATA_SOURCE. Returns its root."""
    dl = STAGE.decide("dataset_download"); ex = STAGE.decide("dataset_extract")
    print(f"[05] source={CFG.dataset.source} | download: {dl['action'].upper()} | extract: {ex['action'].upper()}")
    for d in (dl, ex):
        if d["action"] == "block":
            raise RuntimeError(f"Stage {d['stage']} blocked: {d['reason']}")

    root = _brats_root()
    force = FORCE_DATASET_DOWNLOAD or FORCE_DATASET_EXTRACTION or dl["action"] == "run"
    if root and _count_patient_dirs(root) > 0 and not force:
        print(f"[05] Reusing extracted dataset at {root} ({_count_patient_dirs(root)} patients).")
    else:
        if CFG.dataset.source == "kaggle":
            # Download into the Colab SESSION (fast, disposable).
            target = RAW_LOCAL_DIR
            if CFG.dataset.archive_path and os.path.exists(CFG.dataset.archive_path):
                _extract_archive(CFG.dataset.archive_path, target)
            else:
                _kaggle_download(target)
            root = _brats_root()
        else:  # drive
            target = CFG.dataset.drive_dataset_path
            if root and _count_patient_dirs(root) > 0:
                pass  # already there
            elif CFG.dataset.archive_path and os.path.exists(CFG.dataset.archive_path):
                _extract_archive(CFG.dataset.archive_path, target)
            else:
                # not in Drive -> download from Kaggle INTO Drive so it persists.
                print("[05] Dataset not found in Drive; downloading from Kaggle into Drive.")
                _kaggle_download(target)
            root = _brats_root()

    if root is None or _count_patient_dirs(root) == 0:
        raise RuntimeError("Dataset acquisition failed: no BraTS20_Training_* folders found. "
                           "Check DATA_SOURCE / DRIVE_DATASET_PATH / kaggle.json.")

    n = _count_patient_dirs(root)
    where = "colab-session" if CFG.dataset.source == "kaggle" else "drive"
    STAGE.complete("dataset_download", {"source": CFG.dataset.source, "location": where,
                   "kaggle": CFG.dataset.kaggle_dataset}, inputs=[], outputs=[root], extra={"patient_dirs": n})
    STAGE.complete("dataset_extract", {"root": root}, inputs=[root], outputs=[root], extra={"patient_dirs": n})
    print(f"[05] Acquisition complete ({where}). root={root} patients={n}")
    return root

# Non-destructive on import: just resolve if already present (no download here).
BRATS_ROOT = _brats_root()
print("[05] BRATS_ROOT (pre-resolved):", BRATS_ROOT,
      "\n     -> call prepare_dataset_stages() to acquire/build when needed.")


[05] BRATS_ROOT (pre-resolved): None 
     -> call prepare_dataset_stages() to acquire/build when needed.


## 06 · Patient discovery & modality validation

Discover annotated patients, verify all modalities exist, dimensions match, affines are compatible, mask exists, and volumes are readable (no NaN/Inf). Excluded patients are logged with reasons.


In [12]:
import numpy as np

MODALITY_SUFFIX = {"flair": "_flair.nii", "t1": "_t1.nii", "t1ce": "_t1ce.nii",
                   "t2": "_t2.nii", "seg": "_seg.nii"}

def _find_modality(patient_dir, pid, mod):
    for ext in (".nii", ".nii.gz"):
        cand = os.path.join(patient_dir, f"{pid}{MODALITY_SUFFIX[mod][:-4]}{ext}")
        if os.path.exists(cand):
            return cand
    hits = glob.glob(os.path.join(patient_dir, f"*{mod}.nii*"))
    return hits[0] if hits else None

def discover_patients(root):
    import nibabel as nib
    pdirs = sorted(glob.glob(os.path.join(root, "BraTS20_Training_*")))
    valid, excluded = [], []
    hb = Heartbeat("patient_discovery", every=8)
    needed = ["flair", "t1ce", "t2", "seg"]
    for i, pd in enumerate(pdirs):
        pid = os.path.basename(pd)
        rec = {"patient_id": pid, "dir": pd}
        try:
            paths = {m: _find_modality(pd, pid, m) for m in needed}
            missing = [m for m, p in paths.items() if p is None]
            if missing:
                excluded.append({**rec, "reason": f"missing modalities: {missing}"}); continue
            shapes, affines = {}, {}
            for m, p in paths.items():
                img = nib.load(p); shapes[m] = tuple(img.shape); affines[m] = img.affine
            ref = shapes["flair"]
            if any(s != ref for s in shapes.values()):
                excluded.append({**rec, "reason": f"shape mismatch {shapes}"}); continue
            aff_ref = affines["flair"]
            if any(not np.allclose(a, aff_ref, atol=1e-3) for a in affines.values()):
                excluded.append({**rec, "reason": "affine mismatch"}); continue
            rec.update({"paths": paths, "shape": ref})
            valid.append(rec)
        except Exception as e:
            excluded.append({**rec, "reason": f"read error: {e}"})
        hb.beat(patient=f"{i+1}/{len(pdirs)}", valid=len(valid), excl=len(excluded))
    print(f"[06] discovered {len(valid)} valid, {len(excluded)} excluded of {len(pdirs)}")
    return valid, excluded

print("[06] discovery helpers ready (call discover_patients(BRATS_ROOT)).")


[06] discovery helpers ready (call discover_patients(BRATS_ROOT)).


## 07 · Patient-level split (no leakage)

Split **patients** (never slices) into train/val/test with a fixed seed. Automatic leakage check aborts preprocessing if any patient appears in more than one split. Saves `patient_split.{csv,json}` + `split_summary.json`. The **test split is strictly locked**.


In [13]:
import csv, random

def make_patient_split(valid_patients):
    d = STAGE.decide("patient_split")
    print(f"[07] patient_split: {d['action'].upper()} ({d['reason']})")
    split_csv = os.path.join(CFG.paths.metadata_dir, "patient_split.csv")
    split_json = os.path.join(CFG.paths.metadata_dir, "patient_split.json")
    if d["action"] == "reuse" and not FORCE_REBUILD_PATIENT_SPLIT:
        data = read_json(split_json)
        if data:
            print(f"[07] reusing existing split: "
                  f"{sum(len(v) for v in data.values())} patients")
            return data
    if d["action"] == "block":
        raise RuntimeError(d["reason"])

    ids = sorted(p["patient_id"] for p in valid_patients)
    rng = random.Random(CFG.dataset.split_seed)
    rng.shuffle(ids)
    n = len(ids); n_tr = int(round(n * CFG.dataset.split_train))
    n_va = int(round(n * CFG.dataset.split_val))
    split = {"train": ids[:n_tr], "val": ids[n_tr:n_tr + n_va], "test": ids[n_tr + n_va:]}

    # leakage check
    seen = {}
    for s, members in split.items():
        for pid in members:
            if pid in seen:
                raise AssertionError(f"LEAKAGE: {pid} in {seen[pid]} and {s}")
            seen[pid] = s
    assert sum(len(v) for v in split.values()) == n, "split lost patients"

    os.makedirs(CFG.paths.metadata_dir, exist_ok=True)
    atomic_write_json(split_json, split)
    with open(split_csv, "w", newline="") as f:
        w = csv.writer(f); w.writerow(["patient_id", "split"])
        for s, members in split.items():
            for pid in members: w.writerow([pid, s])
    split_hash = sha256_text(json.dumps(split, sort_keys=True))
    summary = {"counts": {s: len(v) for s, v in split.items()}, "seed": CFG.dataset.split_seed,
               "ratios": [CFG.dataset.split_train, CFG.dataset.split_val, CFG.dataset.split_test],
               "split_hash": split_hash}
    atomic_write_json(os.path.join(CFG.paths.metadata_dir, "split_summary.json"), summary)
    STAGE.complete("patient_split", {"seed": CFG.dataset.split_seed}, inputs=[],
                   outputs=[split_json, split_csv], hashes={"split_hash": split_hash})
    print(f"[07] split: train={len(split['train'])} val={len(split['val'])} test={len(split['test'])} "
          f"hash={split_hash[:12]}")
    return split

print("[07] split helpers ready.")


[07] split helpers ready.


## 08 · Preprocessing registry + deterministic fingerprint

Per-modality intensity normalization on **non-zero brain voxels** only. Modes:
`zscore` (default), `percentile_minmax`, `robust_zscore`, `none`. A preprocessing
**fingerprint** (source identity + split + modality order + normalization +
clipping + plane + thresholds + policies + code hash) governs reuse: processed
data is reused only on an exact fingerprint match. **No HSV augmentation** —
channels are MRI modalities.

**Fast-storage design (important).** Training and evaluation read image files on
*every* epoch. Over a Drive-mounted filesystem, 36k small PNGs are latency-bound
(a few files/sec) which makes training — and even the integrity check —
impractically slow. So the generated media (images/labels/masks) is written to
**local disk** (`LOCAL_SCRATCH/media`, fast) and persisted to Drive as **one
archive file** (`data/processed/media_archive.tar`) — Drive reads/writes a single
big file quickly. On reopen, the media is **restored from that one archive** in
seconds instead of copying thousands of small files. Metadata, manifests and the
archive stay on Drive (durable); the local media is a disposable fast cache.


In [14]:
def normalize_nonzero(vol, mode, clip_lo, clip_hi):
    """Normalize only non-zero brain voxels; preserve zero background."""
    vol = vol.astype(np.float32)
    brain = vol > 0
    if brain.sum() == 0 or mode == "none":
        return vol
    vals = vol[brain]
    if mode == "zscore":
        mu, sd = vals.mean(), vals.std() + 1e-8
        out = np.zeros_like(vol); out[brain] = (vol[brain] - mu) / sd
    elif mode == "robust_zscore":
        med = np.median(vals); iqr = np.subtract(*np.percentile(vals, [75, 25])) + 1e-8
        out = np.zeros_like(vol); out[brain] = (vol[brain] - med) / iqr
    elif mode == "percentile_minmax":
        lo, hi = np.percentile(vals, [clip_lo, clip_hi]); hi = max(hi, lo + 1e-8)
        out = np.zeros_like(vol); out[brain] = np.clip((vol[brain] - lo) / (hi - lo), 0, 1)
    else:
        raise ValueError(f"unknown normalization mode {mode}")
    return out

def to_uint8_channel(norm_vol_slice, clip_lo, clip_hi):
    """Map a normalized 2D slice to uint8 [0,255], preserving zero background."""
    s = norm_vol_slice.astype(np.float32)
    brain = s != 0
    if brain.sum() == 0:
        return np.zeros_like(s, dtype=np.uint8)
    vals = s[brain]
    lo, hi = np.percentile(vals, [clip_lo, clip_hi]); hi = max(hi, lo + 1e-8)
    out = np.zeros_like(s, dtype=np.float32)
    out[brain] = np.clip((s[brain] - lo) / (hi - lo), 0, 1) * 255.0
    return out.astype(np.uint8)

def preprocessing_fingerprint(split_hash, code_hash):
    payload = {
        "source": CFG.dataset.kaggle_dataset if CFG.dataset.source == "kaggle" else CFG.dataset.drive_dataset_path,
        "split_hash": split_hash,
        "modality_order": list(CFG.dataset.channel_map),
        "normalization": CFG.preprocess.normalization,
        "clip": [CFG.preprocess.clip_low_pct, CFG.preprocess.clip_high_pct],
        "plane": CFG.preprocess.plane,
        "min_brain_pixels": CFG.preprocess.min_brain_pixels,
        "min_tumor_pixels": CFG.preprocess.min_tumor_pixels,
        "neg_to_pos_ratio": CFG.preprocess.neg_to_pos_ratio,
        "hard_negative_band": CFG.preprocess.hard_negative_band,
        "negative_seed": CFG.preprocess.negative_seed,
        "code_hash": code_hash,
    }
    return sha256_text(json.dumps(payload, sort_keys=True)), payload

print("[08] preprocessing registry ready. modes: zscore|percentile_minmax|robust_zscore|none")


[08] preprocessing registry ready. modes: zscore|percentile_minmax|robust_zscore|none


## 09 · Slice extraction: pseudo-RGB, tight boxes, whole-tumor masks

For each axial `z`: build pseudo-RGB `(R=FLAIR, G=T1ce, B=T2)`, whole-tumor mask
`seg > 0` (labels 1∪2∪4 → one class), tight 2D bbox (no padding), YOLO label,
and binary mask PNG. Negatives are sampled at the configured ratio incl. hard
negatives near tumor slices. Tiny positives (`< min_tumor_pixels`) are **flagged**,
never silently relabeled. Filenames preserve patient/slice/split.


In [15]:
def tight_bbox_from_mask(mask):
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return None
    x1, x2 = xs.min(), xs.max(); y1, y2 = ys.min(), ys.max()
    return int(x1), int(y1), int(x2), int(y2)

def yolo_line_from_bbox(x1, y1, x2, y2, W, H, cls=0):
    cx = (x1 + x2 + 1) / 2.0 / W; cy = (y1 + y2 + 1) / 2.0 / H
    w = (x2 - x1 + 1) / W; h = (y2 - y1 + 1) / H
    return f"{cls} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}"

def generate_slices_for_patient(rec, split, pp, code_hash, writer_rows):
    """Return list of slice metadata dicts; writes PNG image/mask + label files."""
    import nibabel as nib
    try:
        import cv2
    except Exception:
        cv2 = None
    from PIL import Image
    paths = rec["paths"]
    vols = {m: nib.load(paths[m]).get_fdata() for m in ("flair", "t1ce", "t2")}
    seg = nib.load(paths["seg"]).get_fdata()
    if not np.isfinite(seg).all():
        seg = np.nan_to_num(seg)
    norm = {m: normalize_nonzero(np.nan_to_num(vols[m]), pp.normalization, pp.clip_low_pct, pp.clip_high_pct)
            for m in vols}
    Z = seg.shape[2]
    ch = CFG.dataset.channel_map
    positive_z, all_meta = [], []

    # First pass: identify positives and brain coverage
    slice_info = []
    for z in range(Z):
        seg_z = seg[:, :, z]
        # wt = seg_z > 0
        wt = np.isin(np.rint(seg_z), (1, 4))     # TC = nekroz + enhancing
        brain = (norm[ch[0]][:, :, z] != 0) | (norm[ch[1]][:, :, z] != 0) | (norm[ch[2]][:, :, z] != 0)
        brain_px = int(brain.sum()); tumor_px = int(wt.sum())
        slice_info.append({"z": z, "brain_px": brain_px, "tumor_px": tumor_px, "wt": wt})
        if tumor_px > 0:
            positive_z.append(z)

    # Negative selection: hard negatives near tumor band + random negatives
    rng = random.Random(pp.negative_seed + hash(rec["patient_id"]) % 10000)
    neg_candidates = [s["z"] for s in slice_info
                      if s["tumor_px"] == 0 and s["brain_px"] >= pp.min_brain_pixels]
    hard = set()
    for z in positive_z:
        for dz in range(1, pp.hard_negative_band + 1):
            for zz in (z - dz, z + dz):
                if 0 <= zz < Z and slice_info[zz]["tumor_px"] == 0 and slice_info[zz]["brain_px"] >= pp.min_brain_pixels:
                    hard.add(zz)
    n_pos = len([z for z in positive_z if slice_info[z]["tumor_px"] >= 1])
    n_neg_target = int(round(n_pos * pp.neg_to_pos_ratio))
    chosen_neg = set(list(hard)[:n_neg_target])
    remaining = [z for z in neg_candidates if z not in chosen_neg]
    rng.shuffle(remaining)
    for z in remaining:
        if len(chosen_neg) >= n_neg_target: break
        chosen_neg.add(z)

    def _save(z, is_pos):
        si = slice_info[z]
        if si["brain_px"] < pp.min_brain_pixels and not is_pos:
            return None
        H, W = seg.shape[0], seg.shape[1]
        rgb = np.stack([to_uint8_channel(norm[ch[0]][:, :, z], pp.clip_low_pct, pp.clip_high_pct),
                        to_uint8_channel(norm[ch[1]][:, :, z], pp.clip_low_pct, pp.clip_high_pct),
                        to_uint8_channel(norm[ch[2]][:, :, z], pp.clip_low_pct, pp.clip_high_pct)], axis=-1)
        base = f"{rec['patient_id']}_z{z:03d}"
        img_p = os.path.join(CFG.paths.images_dir, split, base + ".png")
        lbl_p = os.path.join(CFG.paths.labels_dir, split, base + ".txt")
        msk_p = os.path.join(CFG.paths.masks_dir, split, base + "_mask.png")
        Image.fromarray(rgb).save(img_p)
        wt = si["wt"].astype(np.uint8) * 255
        Image.fromarray(wt).save(msk_p)
        tumor_px = si["tumor_px"]; tiny = 0 < tumor_px < pp.min_tumor_pixels
        if is_pos and tumor_px > 0:
            bb = tight_bbox_from_mask(si["wt"])
            line = yolo_line_from_bbox(*bb, W, H) if bb else ""
            atomic_write_text(lbl_p, line + ("\n" if line else ""))
        else:
            atomic_write_text(lbl_p, "")  # empty label = negative
        meta = {"patient_id": rec["patient_id"], "split": split, "z": z, "image": img_p,
                "label": lbl_p, "mask": msk_p, "positive": int(is_pos and tumor_px > 0),
                "tumor_px": tumor_px, "brain_px": si["brain_px"], "tiny_positive": int(tiny),
                "H": H, "W": W}
        writer_rows.append(meta)
        return meta

    for z in positive_z:
        _save(z, True)
    for z in sorted(chosen_neg):
        _save(z, False)
    return len([z for z in positive_z]), len(chosen_neg)

print("[09] slice generation helpers ready.")


[09] slice generation helpers ready.


In [16]:
# === Local-media staging with a single durable Drive archive ============= #
# Rationale: training/eval read image files EVERY epoch. Over Drive FUSE, 36k
# small PNGs are latency-bound (~a few files/sec) which makes training and even
# integrity impractically slow. So media lives on LOCAL disk (fast) and is
# persisted to Drive as ONE archive file (fast big-file I/O). This makes the
# pipeline fast on Colab and any mounted-storage environment.
def _local_media_count():
    n = 0
    for s in ("train", "val", "test"):
        n += len(glob.glob(os.path.join(CFG.paths.images_dir, s, "*.png")))
    return n

def archive_media_to_drive():
    import tarfile
    os.makedirs(os.path.dirname(CFG.paths.media_archive), exist_ok=True)
    tmp = CFG.paths.media_archive + ".tmp"
    with Timer("archive media -> Drive (one file)"):
        with tarfile.open(tmp, "w") as tf:   # no gzip: PNGs already compressed, save CPU
            for sub in ("images", "labels", "masks"):
                d = os.path.join(CFG.paths.local_media, sub)
                if os.path.isdir(d):
                    tf.add(d, arcname=sub)
        os.replace(tmp, CFG.paths.media_archive)
    print(f"[08/09] media archived to Drive: {CFG.paths.media_archive} "
          f"({os.path.getsize(CFG.paths.media_archive)/1e6:.0f} MB)")

def restore_media_from_drive():
    import tarfile
    if not os.path.exists(CFG.paths.media_archive):
        return False
    os.makedirs(CFG.paths.local_media, exist_ok=True)
    with Timer("restore media archive -> local (one file)"):
        with tarfile.open(CFG.paths.media_archive, "r:*") as tf:
            tf.extractall(CFG.paths.local_media)
    ok = _local_media_count() > 0
    print(f"[08/09] media restored to local: {CFG.paths.local_media} (images={_local_media_count()})")
    return ok

def ensure_local_media_present():
    """Make the media available on local disk: reuse if already there, else
    restore from the Drive archive. Returns True if media is present."""
    if _local_media_count() > 0:
        return True
    return restore_media_from_drive()

def stage_preprocess_and_slice(valid_patients, split):
    d_pp = STAGE.decide("preprocessing"); d_sl = STAGE.decide("slice_generation")
    print(f"[08/09] preprocessing: {d_pp['action'].upper()} ({d_pp['reason']})")
    slices_csv = os.path.join(CFG.paths.metadata_dir, "slices.csv")
    code_hash = sha256_text("preproc_v2_localmedia")  # bump when preprocessing code changes
    split_hash = read_json(os.path.join(CFG.paths.metadata_dir, "split_summary.json"), {}).get("split_hash", "")
    fp, fp_payload = preprocessing_fingerprint(split_hash, code_hash)

    stored = read_json(os.path.join(CFG.paths.metadata_dir, "preprocessing_config.json"))
    if (d_pp["action"] == "reuse" and stored and stored.get("fingerprint") == fp
            and os.path.exists(slices_csv) and not FORCE_REBUILD_PREPROCESSED_DATA):
        # Reuse only if the media is (or can be made) present on local disk.
        if ensure_local_media_present():
            print(f"[08/09] fingerprint match ({fp[:12]}) — reusing processed data (media local).")
            return read_csv_dicts(slices_csv)
        print("[08/09] media archive missing/empty — rebuilding processed data.")
    if d_pp["action"] == "block":
        raise RuntimeError(d_pp["reason"])

    by_id = {p["patient_id"]: p for p in valid_patients}
    rows = []
    hb = Heartbeat("preprocessing", every=10, log_path=os.path.join(CFG.paths.metadata_dir, "preprocess.log"))
    total = sum(len(v) for v in split.values()); done = 0
    with Timer("preprocessing + slice generation (writing to LOCAL disk)"):
        for s in ("train", "val", "test"):
            for pid in split[s]:
                if pid not in by_id:
                    continue
                np_, nn_ = generate_slices_for_patient(by_id[pid], s, CFG.preprocess, code_hash, rows)
                done += 1
                hb.beat(split=s, patient=f"{done}/{total}", pos=np_, neg=nn_)
    write_csv_dicts(slices_csv, rows)
    atomic_write_json(os.path.join(CFG.paths.metadata_dir, "preprocessing_config.json"),
                      {"fingerprint": fp, "payload": fp_payload, "config": asdict(CFG.preprocess)})
    # Persist media to Drive as ONE archive so a later session can restore it fast.
    archive_media_to_drive()
    STAGE.complete("preprocessing", fp_payload, inputs=[], outputs=[slices_csv, CFG.paths.media_archive],
                   hashes={"fingerprint": fp})
    STAGE.complete("slice_generation", {"n_slices": len(rows)}, inputs=[slices_csv],
                   outputs=[slices_csv, CFG.paths.media_archive])
    print(f"[08/09] wrote {len(rows)} slices to LOCAL disk + archived to Drive. fingerprint={fp[:12]}")
    return rows

def write_csv_dicts(path, rows):
    if not rows:
        atomic_write_text(path, ""); return
    keys = list(rows[0].keys())
    import io
    buf = io.StringIO(); w = csv.DictWriter(buf, fieldnames=keys); w.writeheader()
    for r in rows:
        w.writerow({k: r.get(k, "") for k in keys})
    atomic_write_text(path, buf.getvalue())

def read_csv_dicts(path):
    if not os.path.exists(path):
        return []
    with open(path) as f:
        return list(csv.DictReader(f))

print("[08/09] preprocessing orchestrator ready.")


[08/09] preprocessing orchestrator ready.


In [17]:
def write_dataset_yaml():
    # Media lives on LOCAL disk (fast training I/O); dataset.yaml points there.
    yaml_txt = (
        f"# BraTS2020 whole-tumor detection (pseudo-RGB axial slices)\n"
        f"path: {CFG.paths.local_media}\n"
        f"train: images/train\n"
        f"val: images/val\n"
        f"test: images/test\n"
        f"nc: 1\n"
        # f"names:\n  0: tumor\n"
        f"names:\n  0: tumor_core\n"
    )
    atomic_write_text(CFG.paths.dataset_yaml, yaml_txt)
    print("[yaml] wrote", CFG.paths.dataset_yaml)
    return CFG.paths.dataset_yaml

print("[yaml] dataset.yaml writer ready.")


[yaml] dataset.yaml writer ready.


## 10 · Dataset integrity tests

Automated checks: image↔label pairing, normalized coords in [0,1], positive w/h, box↔mask agreement, empty-label↔negative-mask, no duplicate image hash across splits, no patient leakage, valid PNGs, 3 channels, matching dims, valid class id, dataset.yaml paths. Produces a human-readable report and **aborts** on critical failures.


In [18]:
def run_integrity_tests(split):
    d = STAGE.decide("dataset_integrity")
    print(f"[10] integrity: {d['action'].upper()} ({d['reason']})")
    report = {"checks": {}, "critical_failures": [], "warnings": []}
    from PIL import Image
    def add(name, ok, critical=True, detail=""):
        report["checks"][name] = {"ok": bool(ok), "detail": detail}
        if not ok and critical:
            report["critical_failures"].append(f"{name}: {detail}")
        elif not ok:
            report["warnings"].append(f"{name}: {detail}")

    hashes = {}
    leak = False
    seen_patient_split = {}
    n_imgs = 0
    coord_ok = True; wh_ok = True; box_mask_ok = True; chan_ok = True; dim_ok = True; cls_ok = True
    empty_neg_ok = True; pos_has_label = True
    for s in ("train", "val", "test"):
        img_dir = os.path.join(CFG.paths.images_dir, s)
        for img_p in sorted(glob.glob(os.path.join(img_dir, "*.png"))):
            n_imgs += 1
            base = os.path.basename(img_p)[:-4]
            pid = "_".join(base.split("_")[:-1])
            if pid in seen_patient_split and seen_patient_split[pid] != s:
                leak = True
            seen_patient_split[pid] = s
            lbl_p = os.path.join(CFG.paths.labels_dir, s, base + ".txt")
            msk_p = os.path.join(CFG.paths.masks_dir, s, base + "_mask.png")
            if not os.path.exists(lbl_p):
                pos_has_label = False; continue
            try:
                im = Image.open(img_p); arr = np.array(im)
                if arr.ndim != 3 or arr.shape[2] != 3: chan_ok = False
                h = sha256_file(img_p); hashes.setdefault(h, []).append(f"{s}/{base}")
            except Exception:
                add("valid_png", False, True, img_p); continue
            lines = [ln for ln in open(lbl_p).read().splitlines() if ln.strip()]
            mask = np.array(Image.open(msk_p)) if os.path.exists(msk_p) else None
            if not lines:
                if mask is not None and mask.sum() > 0:
                    empty_neg_ok = False
            for ln in lines:
                parts = ln.split()
                if len(parts) != 5: coord_ok = False; continue
                c, cx, cy, w, ww = parts
                if int(float(c)) != 0: cls_ok = False
                vals = list(map(float, (cx, cy, w, ww)))
                if any(v < 0 or v > 1 for v in vals): coord_ok = False
                if vals[2] <= 0 or vals[3] <= 0: wh_ok = False
                if mask is not None and mask.sum() > 0:
                    ys, xs = np.where(mask > 0)
                    H, W = mask.shape[:2]
                    bx1 = (vals[0] - vals[2] / 2) * W; bx2 = (vals[0] + vals[2] / 2) * W
                    by1 = (vals[1] - vals[3] / 2) * H; by2 = (vals[1] + vals[3] / 2) * H
                    if abs(bx1 - xs.min()) > 2 or abs(bx2 - (xs.max() + 1)) > 2 or \
                       abs(by1 - ys.min()) > 2 or abs(by2 - (ys.max() + 1)) > 2:
                        box_mask_ok = False
    dup = {h: v for h, v in hashes.items() if len(set(x.split("/")[0] for x in v)) > 1}
    add("image_label_pairing", pos_has_label)
    add("coords_in_unit_range", coord_ok)
    add("positive_wh", wh_ok)
    add("box_matches_mask", box_mask_ok, critical=False)
    add("empty_label_is_negative", empty_neg_ok)
    add("no_cross_split_duplicate", len(dup) == 0, detail=f"{len(dup)} duplicates")
    add("no_patient_leakage", not leak)
    add("three_channels", chan_ok)
    add("valid_class_id", cls_ok)
    add("dataset_yaml_exists", os.path.exists(CFG.paths.dataset_yaml))
    report["n_images"] = n_imgs
    atomic_write_json(os.path.join(CFG.paths.metadata_dir, "data_integrity_report.json"), report)
    # human-readable
    lines = ["DATASET INTEGRITY REPORT", f"images: {n_imgs}", ""]
    for name, r in report["checks"].items():
        lines.append(f"  [{'PASS' if r['ok'] else 'FAIL'}] {name} {r['detail']}")
    if report["critical_failures"]:
        lines.append("\nCRITICAL FAILURES:")
        lines += [f"  - {x}" for x in report["critical_failures"]]
    txt = "\n".join(lines); print(txt)
    if report["critical_failures"]:
        raise AssertionError("Dataset integrity CRITICAL failures — aborting training.")
    STAGE.complete("dataset_integrity", {}, inputs=[CFG.paths.dataset_yaml],
                   outputs=[os.path.join(CFG.paths.metadata_dir, "data_integrity_report.json")])
    return report

print("[10] integrity tests ready.")


[10] integrity tests ready.


In [ ]:
# Seed tekrarlarını ayrı bir ağaca yaz: orijinal 37 run'a dokunulmaz.
import os
# SEED_TAG = f"seed_{CFG.train.seed}"
SEED_TAG = f"tc_seed_{CFG.train.seed}"
_repeat_root = os.path.join(CFG.paths.experiments_repeated, SEED_TAG)

# TC dataseti tamamen ayri agaclara: WT verisi ve manifestleri oldugu gibi kalir
CFG.paths.local_media    = os.path.join(CFG.paths.local_scratch, "media_tc")
CFG.paths.images_dir     = os.path.join(CFG.paths.local_media, "images")
CFG.paths.labels_dir     = os.path.join(CFG.paths.local_media, "labels")
CFG.paths.masks_dir      = os.path.join(CFG.paths.local_media, "masks")
CFG.paths.dataset_yaml   = os.path.join(CFG.paths.local_media, "dataset.yaml")
CFG.paths.media_archive  = os.path.join(CFG.paths.processed_dir, "media_archive_tc.tar")
CFG.paths.metadata_dir   = os.path.join(CFG.paths.processed_dir, "metadata_tc")
CFG.paths.stage_manifests = os.path.join(CFG.paths.project_root, "stage_manifests_tc")
os.makedirs(CFG.paths.metadata_dir, exist_ok=True)
os.makedirs(CFG.paths.stage_manifests, exist_ok=True)

CFG.paths.experiments = _repeat_root
CFG.paths.registry    = os.path.join(_repeat_root, "registry")
CFG.paths.reports     = os.path.join(_repeat_root, "reports")
CFG.paths.figures     = os.path.join(_repeat_root, "figures")

for _d in (CFG.paths.experiments, CFG.paths.registry,
           CFG.paths.reports, CFG.paths.figures):
    os.makedirs(_d, exist_ok=True)

print("[repeat] çıktı ->", CFG.paths.experiments)
assert "experiments_repeated" in CFG.paths.experiments, "koruma: orijinal ağaca yazılıyor!"


[repeat] çıktı -> /content/drive/MyDrive/icarb_brats_rtdetr_sam/experiments_repeated/seed_62


## 11 · Sample visualization

Sanity-check a few slices: pseudo-RGB with the ground-truth box and mask overlaid. Saved to `figures/`.


In [ ]:
def visualize_samples(n=6, split="train"):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from PIL import Image
    imgs = sorted(glob.glob(os.path.join(CFG.paths.images_dir, split, "*.png")))
    # prefer positives
    pos = []
    for p in imgs:
        base = os.path.basename(p)[:-4]
        lbl = os.path.join(CFG.paths.labels_dir, split, base + ".txt")
        if os.path.exists(lbl) and open(lbl).read().strip():
            pos.append(p)
    show = (pos or imgs)[:n]
    if not show:
        print("[11] no slices to visualize yet."); return None
    cols = 3; rows = (len(show) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.array(axes).reshape(-1)
    for ax in axes: ax.axis("off")
    for i, p in enumerate(show):
        base = os.path.basename(p)[:-4]
        arr = np.array(Image.open(p)); H, W = arr.shape[:2]
        axes[i].imshow(arr); axes[i].set_title(base, fontsize=8)
        lbl = os.path.join(CFG.paths.labels_dir, split, base + ".txt")
        if os.path.exists(lbl):
            for ln in open(lbl).read().splitlines():
                if not ln.strip(): continue
                _, cx, cy, w, h = map(float, ln.split())
                x1 = (cx - w / 2) * W; y1 = (cy - h / 2) * H
                axes[i].add_patch(plt.Rectangle((x1, y1), w * W, h * H, fill=False, edgecolor="lime", lw=1.5))
    out = os.path.join(CFG.paths.figures, f"sample_slices_{split}.png")
    fig.tight_layout(); fig.savefig(out, dpi=150); plt.close(fig)
    print("[11] saved", out)
    return out

print("[11] visualization ready.")


[11] visualization ready.


## 12 · Bounding-box loss registry

A centralized, extensible registry of IoU-family regression losses plus the
proposed **IC-Arb**. Every loss operates on **xyxy** boxes and returns a
**per-pair** tensor `(N,)` so reduction/normalization is owned by the criterion,
never hidden. Each entry carries metadata (reference, default params, box format,
epsilon, notes) and is unit-tested in Section 13.

`L_IC-Arb = α·(1 − IoU) + (1 − α)·(1 − C_p)`, `C_p = A_int / A_pred`.
Target coverage `C_t = A_int / A_gt` is reported separately (it is *not* optimized).

### 12.2b Segmentation-literature losses, exactly formulated on boxes

Dice, Tversky / Focal Tversky, Boundary / Generalized Surface and Lovász-Hinge are
normally defined over pixel masks. A box **is** a mask, and for two axis-aligned
rectangles every quantity these losses need is available in closed form:

- region measures — `TP = A_int`, `FP = A_pred − A_int`, `FN = A_gt − A_int`;
- surface measures — the per-edge outward overshoot and inward shortfall,
  normalised by the enclosing-box diagonal.

So no rasterisation, sampling or discretisation is involved: the implementations are
exact, scale-invariant and differentiable w.r.t. the predicted coordinates, and they
plug into the same `_get_loss_bbox` slot as the IoU family — the comparison against
IC-Arb stays strictly single-factor.

Two properties are worth stating up front, both asserted in Section 13:

- **Dice / Tversky / Focal Tversky have zero gradient for fully disjoint boxes**
  (`TP = 0`). This is the same defect that motivated GIoU. Here the frozen L1 term
  (weight 5.0) and the untouched Hungarian GIoU matching cost still pull such pairs
  together; Boundary, Generalized Surface and Lovász-Hinge do not have the defect
  because they carry an explicit surface-distance term.
- **`lovasz_hinge(κ=0)` equals `2·(1 − IoU)` exactly.** The Lovász extension must
  agree with the set function it extends whenever the margins are binary, so this
  identity is the correctness proof for the implementation. Note also that the
  Lovász-hinge scale is ≈2× the IoU family (margin 2 for a fully wrong region).

Lovász-**Hinge** is used rather than Lovász-**Softmax** because the detection task has a
single foreground class; the softmax variant is the multi-class form and degenerates
here (Berman et al., CVPR 2018).


In [ ]:
# === 12.1 Geometry primitives & coverage terms (xyxy, per-pair) ========== #
import math
import torch

BOX_EPS = 1e-7

def _area_xyxy(b):
    w = (b[..., 2] - b[..., 0]).clamp(min=0); h = (b[..., 3] - b[..., 1]).clamp(min=0)
    return w * h

def _inter_xyxy(p, g):
    x1 = torch.max(p[..., 0], g[..., 0]); y1 = torch.max(p[..., 1], g[..., 1])
    x2 = torch.min(p[..., 2], g[..., 2]); y2 = torch.min(p[..., 3], g[..., 3])
    return (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)

def _iou_parts(p, g, eps=BOX_EPS):
    ap = _area_xyxy(p); ag = _area_xyxy(g); inter = _inter_xyxy(p, g)
    union = ap + ag - inter + eps
    return inter, union, ap, ag, inter / union

def _enclosing(p, g):
    return (torch.min(p[..., 0], g[..., 0]), torch.min(p[..., 1], g[..., 1]),
            torch.max(p[..., 2], g[..., 2]), torch.max(p[..., 3], g[..., 3]))

def _wh_c(b):
    w = (b[..., 2] - b[..., 0]).clamp(min=0); h = (b[..., 3] - b[..., 1]).clamp(min=0)
    return w, h, (b[..., 0] + b[..., 2]) / 2, (b[..., 1] + b[..., 3]) / 2

def prediction_coverage(p, g, eps=BOX_EPS):
    "C_p = A_int / A_pred"
    return _inter_xyxy(p, g) / (_area_xyxy(p) + eps)

def target_coverage(p, g, eps=BOX_EPS):
    "C_t = A_int / A_gt"
    return _inter_xyxy(p, g) / (_area_xyxy(g) + eps)

def xywh2xyxy_t(b):
    x, y, w, h = b[..., 0], b[..., 1], b[..., 2], b[..., 3]
    return torch.stack([x - w / 2, y - h / 2, x + w / 2, y + h / 2], dim=-1)


In [ ]:
# === 12.2 Loss functions (per-pair, differentiable, numerically stable) == #
def iou_loss(p, g, eps=BOX_EPS, **_):
    return 1.0 - _iou_parts(p, g, eps)[4]

def giou_loss(p, g, eps=BOX_EPS, **_):
    inter, union, _, _, iou = _iou_parts(p, g, eps)
    cx1, cy1, cx2, cy2 = _enclosing(p, g)
    c = (cx2 - cx1).clamp(min=0) * (cy2 - cy1).clamp(min=0) + eps
    return 1.0 - (iou - (c - union) / c)

def diou_loss(p, g, eps=BOX_EPS, **_):
    _, _, _, _, iou = _iou_parts(p, g, eps)
    cx1, cy1, cx2, cy2 = _enclosing(p, g)
    c2 = (cx2 - cx1) ** 2 + (cy2 - cy1) ** 2 + eps
    _, _, pcx, pcy = _wh_c(p); _, _, gcx, gcy = _wh_c(g)
    return 1.0 - (iou - ((pcx - gcx) ** 2 + (pcy - gcy) ** 2) / c2)

def ciou_loss(p, g, eps=BOX_EPS, **_):
    _, _, _, _, iou = _iou_parts(p, g, eps)
    cx1, cy1, cx2, cy2 = _enclosing(p, g)
    c2 = (cx2 - cx1) ** 2 + (cy2 - cy1) ** 2 + eps
    pw, ph, pcx, pcy = _wh_c(p); gw, gh, gcx, gcy = _wh_c(g)
    rho2 = (pcx - gcx) ** 2 + (pcy - gcy) ** 2
    v = (4 / math.pi ** 2) * torch.pow(torch.atan(gw / (gh + eps)) - torch.atan(pw / (ph + eps)), 2)
    with torch.no_grad():
        alpha = v / (1 - iou + v + eps)
    return 1.0 - (iou - (rho2 / c2 + alpha * v))

def eiou_loss(p, g, eps=BOX_EPS, **_):
    _, _, _, _, iou = _iou_parts(p, g, eps)
    cx1, cy1, cx2, cy2 = _enclosing(p, g)
    cw = cx2 - cx1; ch = cy2 - cy1; c2 = cw ** 2 + ch ** 2 + eps
    pw, ph, pcx, pcy = _wh_c(p); gw, gh, gcx, gcy = _wh_c(g)
    dis = ((pcx - gcx) ** 2 + (pcy - gcy) ** 2) / c2
    asp = (pw - gw) ** 2 / (cw ** 2 + eps) + (ph - gh) ** 2 / (ch ** 2 + eps)
    return 1.0 - (iou - dis - asp)

def siou_loss(p, g, eps=BOX_EPS, **_):
    _, _, _, _, iou = _iou_parts(p, g, eps)
    pw, ph, pcx, pcy = _wh_c(p); gw, gh, gcx, gcy = _wh_c(g)
    cw = gcx - pcx; ch = gcy - pcy
    sigma = torch.sqrt(cw ** 2 + ch ** 2 + eps)
    sa = torch.abs(ch) / sigma; sb = torch.abs(cw) / sigma
    sa = torch.where(sa > sb, sb, sa)
    angle = torch.cos(2 * (torch.asin(sa.clamp(-1 + eps, 1 - eps)) - math.pi / 4))
    ex1, ey1, ex2, ey2 = _enclosing(p, g)
    cwid = (ex2 - ex1) + eps; chei = (ey2 - ey1) + eps
    gamma = 2 - angle
    dist = (1 - torch.exp(-gamma * (cw / cwid) ** 2)) + (1 - torch.exp(-gamma * (ch / chei) ** 2))
    theta = 4.0
    ow = torch.abs(pw - gw) / (torch.max(pw, gw) + eps)
    oh = torch.abs(ph - gh) / (torch.max(ph, gh) + eps)
    shape = torch.pow(1 - torch.exp(-ow), theta) + torch.pow(1 - torch.exp(-oh), theta)
    return 1.0 - (iou - (dist + shape) / 2)

def alpha_iou_loss(p, g, alpha=3.0, eps=BOX_EPS, **_):
    return 1.0 - torch.pow(_iou_parts(p, g, eps)[4].clamp(min=eps), alpha)

def wiou_loss(p, g, eps=BOX_EPS, **_):
    _, _, _, _, iou = _iou_parts(p, g, eps)
    l_iou = 1.0 - iou
    ex1, ey1, ex2, ey2 = _enclosing(p, g)
    _, _, pcx, pcy = _wh_c(p); _, _, gcx, gcy = _wh_c(g)
    rho2 = (pcx - gcx) ** 2 + (pcy - gcy) ** 2
    r = torch.exp(rho2 / ((ex2 - ex1) ** 2 + (ey2 - ey1) ** 2 + eps).detach())
    return r * l_iou

def piou_loss(p, g, eps=BOX_EPS, **_):
    _, _, _, _, iou = _iou_parts(p, g, eps)
    gw, gh, _, _ = _wh_c(g)
    P = (torch.abs(p[..., 0] - g[..., 0]) / (gw + eps) + torch.abs(p[..., 2] - g[..., 2]) / (gw + eps)
         + torch.abs(p[..., 1] - g[..., 1]) / (gh + eps) + torch.abs(p[..., 3] - g[..., 3]) / (gh + eps)) / 4.0
    return (1.0 - iou) + (1.0 - torch.exp(-(P ** 2)))

def icarb_loss(p, g, alpha=0.5, eps=BOX_EPS, **_):
    "IC-Arb: alpha*(1-IoU) + (1-alpha)*(1-C_p)"
    _, _, ap, _, iou = _iou_parts(p, g, eps)
    c_p = _inter_xyxy(p, g) / (ap + eps)
    return alpha * (1.0 - iou) + (1.0 - alpha) * (1.0 - c_p)

def icarb_components(p, g, alpha=0.5, eps=BOX_EPS):
    "Return (iou_component, coverage_component, total), each (N,)."
    _, _, ap, _, iou = _iou_parts(p, g, eps)
    c_p = _inter_xyxy(p, g) / (ap + eps)
    ic = alpha * (1.0 - iou); cc = (1.0 - alpha) * (1.0 - c_p)
    return ic, cc, ic + cc


In [ ]:
# === 12.2b Region / boundary / Lovasz losses (box-level, per-pair) ====== #
# These families come from the SEGMENTATION literature, where they act on pixel
# masks. Here every "mask" is an axis-aligned box, so the region measures
# (TP/FP/FN areas) and the surface distances have EXACT closed forms - no
# rasterisation, no sampling, no discretisation error. Each loss below is
# per-pair, scale-invariant and differentiable w.r.t. the predicted box, so it
# drops into the same `_get_loss_bbox` slot as the IoU-family baselines and the
# comparison against IC-Arb stays strictly single-factor.

def _confusion_areas(p, g):
    "Exact (TP, FP, FN, A_pred, A_gt) areas for two axis-aligned boxes."
    inter = _inter_xyxy(p, g); ap = _area_xyxy(p); ag = _area_xyxy(g)
    return inter, (ap - inter).clamp(min=0), (ag - inter).clamp(min=0), ap, ag


def _surface_terms(p, g, eps=BOX_EPS):
    """Normalised outward / inward edge displacements between two rectangles.

    over  = total distance the prediction spills OUTSIDE the gt boundary
    short = total distance the prediction falls SHORT of the gt boundary
    Both are divided by the enclosing-box diagonal (scale invariant) and both
    are exactly 0 iff the boxes coincide. `over + short` is the L1 symmetric
    surface (boundary) distance between the two rectangle contours.
    """
    ex1, ey1, ex2, ey2 = _enclosing(p, g)
    diag = torch.sqrt((ex2 - ex1).clamp(min=0) ** 2 + (ey2 - ey1).clamp(min=0) ** 2 + eps)
    over = ((g[..., 0] - p[..., 0]).clamp(min=0) + (p[..., 2] - g[..., 2]).clamp(min=0)
            + (g[..., 1] - p[..., 1]).clamp(min=0) + (p[..., 3] - g[..., 3]).clamp(min=0))
    short = ((p[..., 0] - g[..., 0]).clamp(min=0) + (g[..., 2] - p[..., 2]).clamp(min=0)
             + (p[..., 1] - g[..., 1]).clamp(min=0) + (g[..., 3] - p[..., 3]).clamp(min=0))
    return over / (diag + eps), short / (diag + eps), diag


# --- Dice ---------------------------------------------------------------- #
def dice_loss(p, g, smooth=0.0, eps=BOX_EPS, **_):
    "L = 1 - 2*TP / (A_pred + A_gt). smooth=0 keeps the loss scale-invariant."
    tp, fp, fn, ap, ag = _confusion_areas(p, g)
    return 1.0 - (2.0 * tp + smooth) / (ap + ag + smooth).clamp(min=eps)


# --- Tversky / Focal Tversky --------------------------------------------- #
def tversky_index(p, g, alpha=0.3, beta=0.7, eps=BOX_EPS):
    "TI = TP / (TP + alpha*FP + beta*FN). alpha weights precision, beta recall."
    tp, fp, fn, _, _ = _confusion_areas(p, g)
    return tp / (tp + alpha * fp + beta * fn).clamp(min=eps)


def tversky_loss(p, g, alpha=0.3, beta=0.7, eps=BOX_EPS, **_):
    return 1.0 - tversky_index(p, g, alpha, beta, eps)


def focal_tversky_loss(p, g, alpha=0.3, beta=0.7, gamma=0.75, eps=BOX_EPS, **_):
    "FTL = (1 - TI)**gamma. Abraham & Khan use exponent 1/gamma_paper = 0.75."
    ti = tversky_index(p, g, alpha, beta, eps)
    return torch.pow((1.0 - ti).clamp(min=eps), gamma)


# --- Boundary / surface -------------------------------------------------- #
def boundary_loss(p, g, eps=BOX_EPS, **_):
    "Mean normalised rectangle surface distance (Hausdorff-style boundary term)."
    over, short, _ = _surface_terms(p, g, eps)
    return (over + short) / 4.0


def generalized_surface_loss(p, g, lam=0.5, eps=BOX_EPS, **_):
    "GSL = (1-lam)*Dice + lam*Boundary. lam=0 -> Dice, lam=1 -> pure boundary."
    return (1.0 - lam) * dice_loss(p, g, eps=eps) + lam * boundary_loss(p, g, eps)


# --- Lovasz-hinge -------------------------------------------------------- #
def _lovasz_grad_weighted(y_sorted, w_sorted, eps=BOX_EPS):
    """Measure-weighted Lovasz gradient of the Jaccard loss (last dim = elements).

    Generalises Berman et al.'s per-pixel `lovasz_grad` to elements carrying an
    arbitrary measure w (here: region areas instead of unit-area pixels).
    """
    gts = (y_sorted * w_sorted).sum(-1, keepdim=True)
    cum_pos = torch.cumsum(y_sorted * w_sorted, dim=-1)
    cum_neg = torch.cumsum((1.0 - y_sorted) * w_sorted, dim=-1)
    jac = 1.0 - (gts - cum_pos) / (gts + cum_neg).clamp(min=eps)
    return torch.cat([jac[..., :1], jac[..., 1:] - jac[..., :-1]], dim=-1)


def lovasz_hinge_loss(p, g, kappa=1.0, eps=BOX_EPS, **_):
    """Exact Lovasz extension of the Jaccard loss over the {TP, FP, FN} partition.

    A box pair partitions the plane into four measurable regions. TN and TP are
    classified correctly with margin >= 1, so their hinge errors are 0 and they
    sort last; only TP is kept (it still carries measure for the Jaccard grad).
    FP / FN carry hinge error 2 + kappa * (normalised surface overshoot), which
    is what makes the loss keep pushing after the boxes stop overlapping.

    kappa = 0 reduces EXACTLY to 2*(1 - IoU) - the Lovasz extension agrees with
    the set function it extends on binary margins. That identity is asserted in
    Section 13 and is the correctness proof for this implementation.
    """
    tp, fp, fn, _, _ = _confusion_areas(p, g)
    over, short, _ = _surface_terms(p, g, eps)
    w = torch.stack([tp, fp, fn], dim=-1)
    w = w / w.sum(-1, keepdim=True).clamp(min=eps)          # scale invariance
    y = torch.stack([torch.ones_like(tp), torch.zeros_like(fp), torch.ones_like(fn)], dim=-1)
    z = torch.zeros_like(tp)
    err = torch.stack([z, 2.0 + kappa * over, 2.0 + kappa * short], dim=-1)
    err_sorted, perm = torch.sort(err, dim=-1, descending=True, stable=True)
    grad = _lovasz_grad_weighted(torch.gather(y, -1, perm), torch.gather(w, -1, perm), eps)
    return (err_sorted * grad).sum(-1)


In [ ]:
# === 12.3 The registry object =========================================== #
class LossRegistry:
    """Extensible registry: add a loss without touching the training loop."""
    def __init__(self):
        self._d = {}
    def register(self, name, fn, display, reference, default_params, box_format,
                 requires_overlap, eps, notes, family="iou", disjoint_min=0.9):
        # family      : "iou" | "region" | "boundary" | "lovasz" — grouping for reports
        # disjoint_min: documented lower bound of L(disjoint pair), asserted in Sec. 13.
        #               Not every family saturates at ~1 the way the IoU family does.
        self._d[name] = {
            "registry_name": name, "display_name": display, "reference": reference,
            "callable": fn, "default_params": dict(default_params),
            "box_format": box_format, "requires_overlap": requires_overlap,
            "eps": eps, "expected_return_shape": "(N,) per-pair", "notes": notes,
            "family": family, "disjoint_min": float(disjoint_min),
        }
    def list(self):
        return sorted(self._d)
    def get(self, name):
        if name not in self._d:
            raise KeyError(f"loss '{name}' not registered. Available: {self.list()}")
        return self._d[name]
    def validate_params(self, name, params):
        meta = self.get(name)
        merged = dict(meta["default_params"]); merged.update(params or {})
        if name == "icarb":
            a = merged.get("alpha", 0.5)
            if not (0.0 <= float(a) <= 1.0):
                raise ValueError(f"IC-Arb alpha must be in [0,1], got {a}")
        if name == "alpha_iou":
            if float(merged.get("alpha", 3.0)) <= 0:
                raise ValueError("alpha_iou alpha must be > 0")
        if name in ("tversky", "focal_tversky"):
            a, b = float(merged.get("alpha", 0.3)), float(merged.get("beta", 0.7))
            if a < 0 or b < 0 or (a + b) <= 0:
                raise ValueError(f"tversky needs alpha,beta >= 0 and alpha+beta > 0, got {a},{b}")
            if name == "focal_tversky" and float(merged.get("gamma", 0.75)) <= 0:
                raise ValueError("focal_tversky gamma must be > 0")
        if name == "dice" and float(merged.get("smooth", 0.0)) < 0:
            raise ValueError("dice smooth must be >= 0")
        if name == "gsl":
            lam = float(merged.get("lam", 0.5))
            if not (0.0 <= lam <= 1.0):
                raise ValueError(f"gsl lam must be in [0,1], got {lam}")
        if name == "lovasz_hinge" and float(merged.get("kappa", 1.0)) < 0:
            raise ValueError("lovasz_hinge kappa must be >= 0")
        return merged
    def make(self, name, params=None):
        meta = self.get(name); merged = self.validate_params(name, params)
        fn = meta["callable"]
        return (lambda p, g: fn(p, g, **merged)), merged
    def to_json_dict(self):
        out = {}
        for k, v in self._d.items():
            vv = {kk: vv2 for kk, vv2 in v.items() if kk != "callable"}
            out[k] = vv
        return out

LOSS_REGISTRY = LossRegistry()
_R = LOSS_REGISTRY.register
_R("giou", giou_loss, "GIoU", "Rezatofighi et al., CVPR 2019", {}, "xyxy", True, BOX_EPS, "Generalized IoU.")
_R("diou", diou_loss, "DIoU", "Zheng et al., AAAI 2020", {}, "xyxy", False, BOX_EPS, "Distance IoU.")
_R("ciou", ciou_loss, "CIoU", "Zheng et al., AAAI 2020 / TCYB 2021", {}, "xyxy", False, BOX_EPS, "Complete IoU (aspect-ratio term, alpha detached).")
_R("eiou", eiou_loss, "EIoU", "Zhang et al., Neurocomputing 2022", {}, "xyxy", False, BOX_EPS, "Efficient IoU (explicit w/h penalties).")
_R("siou", siou_loss, "SIoU", "Gevorgyan, arXiv 2022", {}, "xyxy", False, BOX_EPS, "SCYLLA IoU (angle+distance+shape).")
_R("alpha_iou", alpha_iou_loss, "Alpha-IoU", "He et al., NeurIPS 2021", {"alpha": 3.0}, "xyxy", True, BOX_EPS, "1 - IoU**alpha.")
_R("wiou", wiou_loss, "WIoU", "Tong et al., arXiv 2023", {}, "xyxy", False, BOX_EPS, "Wise-IoU v1 (detached enclosing size).")
_R("piou", piou_loss, "PIoU", "Liu et al., arXiv 2023", {}, "xyxy", True, BOX_EPS, "Powerful-IoU v1 corner-distance penalty.")
_R("icarb", icarb_loss, "IC-Arb", "Proposed (this study)", {"alpha": 0.5}, "xyxy", True, BOX_EPS,
   "IoU-Coverage Arbitration: alpha*(1-IoU)+(1-alpha)*(1-C_p). C_p asymmetric; report C_t separately.")

# --- Segmentation-literature losses, exact box formulations (Sec. 12.2b) --
_R("dice", dice_loss, "Dice", "Milletari et al., 3DV 2016; Sudre et al., DLMIA 2017",
   {"smooth": 0.0}, "xyxy", True, BOX_EPS,
   "Box Dice: 1 - 2*TP/(A_p+A_g). Equals Tversky(0.5,0.5). Symmetric in FP/FN. "
   "CAVEAT: gradient vanishes for fully disjoint boxes (TP=0); the frozen L1 term "
   "(weight 5.0) and the unchanged Hungarian GIoU matching cost still pull them together.",
   family="region", disjoint_min=0.9)
_R("tversky", tversky_loss, "Tversky", "Salehi et al., MLMI 2017",
   {"alpha": 0.3, "beta": 0.7}, "xyxy", True, BOX_EPS,
   "TP/(TP+alpha*FP+beta*FN). beta>alpha penalises misses (recall-oriented), the "
   "mirror image of IC-Arb, whose C_p term penalises over-prediction. Same disjoint caveat as Dice.",
   family="region", disjoint_min=0.9)
_R("focal_tversky", focal_tversky_loss, "Focal Tversky", "Abraham & Khan, ISBI 2019",
   {"alpha": 0.3, "beta": 0.7, "gamma": 0.75}, "xyxy", True, BOX_EPS,
   "(1-TI)**gamma; gamma=0.75 is the paper's 1/gamma with gamma=4/3. gamma<1 up-weights "
   "hard (low-overlap) pairs. gamma=1 reduces to Tversky. Same disjoint caveat as Dice.",
   family="region", disjoint_min=0.9)
_R("boundary", boundary_loss, "Boundary / Surface", "Kervadec et al., MIDL 2019 (exact rectangle-surface form)",
   {}, "xyxy", False, BOX_EPS,
   "Mean normalised rectangle surface distance. Contour-based, NOT region-based: it keeps "
   "a non-zero gradient when the boxes are disjoint, but it is bounded (~0.59 for a far pair) "
   "and is designed to be combined with a region term - see gsl.",
   family="boundary", disjoint_min=0.40)
_R("gsl", generalized_surface_loss, "Generalized Surface",
   "Celaya et al., Machine Learning: Science and Technology 2024",
   {"lam": 0.5}, "xyxy", True, BOX_EPS,
   "(1-lam)*Dice + lam*Boundary with a fixed lam (no schedule, so all runs stay comparable). "
   "Restores the disjoint-pair gradient that pure Dice loses.",
   family="boundary", disjoint_min=0.60)
_R("lovasz_hinge", lovasz_hinge_loss, "Lovasz-Hinge", "Berman et al., CVPR 2018",
   {"kappa": 1.0}, "xyxy", True, BOX_EPS,
   "Exact Lovasz extension of the Jaccard loss over the {TP,FP,FN} area partition, with "
   "hinge margins 2 + kappa*(normalised surface overshoot). kappa=0 collapses to 2*(1-IoU) "
   "exactly (asserted in Sec. 13). Binary/foreground task -> hinge is the correct variant; "
   "Lovasz-Softmax is the multi-class form and degenerates for a single class. "
   "NOTE: its scale is ~2x the IoU family (margin 2 for a fully wrong region).",
   family="lovasz", disjoint_min=0.9)

print("Registered losses:", LOSS_REGISTRY.list())
for _fam in ("iou", "region", "boundary", "lovasz"):
    _names = [n for n in LOSS_REGISTRY.list() if LOSS_REGISTRY.get(n)["family"] == _fam]
    if _names: print(f"  {_fam:9s}: {_names}")


Registered losses: ['alpha_iou', 'boundary', 'ciou', 'dice', 'diou', 'eiou', 'focal_tversky', 'giou', 'gsl', 'icarb', 'lovasz_hinge', 'piou', 'siou', 'tversky', 'wiou']
  iou      : ['alpha_iou', 'ciou', 'diou', 'eiou', 'giou', 'icarb', 'piou', 'siou', 'wiou']
  region   : ['dice', 'focal_tversky', 'tversky']
  boundary : ['boundary', 'gsl']
  lovasz   : ['lovasz_hinge']


## 13 · Loss unit tests

Every loss is tested on canonical box configurations, gradient finiteness, and IC-Arb's defining properties. This cell **aborts** if any test fails.


In [ ]:
def _b(t): return torch.tensor(t, dtype=torch.float64)

def run_loss_unit_tests(verbose=False):
    failures = []
    def ck(cond, msg):
        if not cond: failures.append(msg)
        if verbose: print(("  ok  : " if cond else "  FAIL: ") + msg)

    names = LOSS_REGISTRY.list()
    ident = _b([[10., 10., 30., 30.]])
    for n in names:
        fn, _ = LOSS_REGISTRY.make(n)
        # identical boxes -> ~0
        ck(abs(fn(ident, ident.clone()).item()) < 1e-4, f"{n}: identical~0")
        # non-overlapping -> finite, above the family-specific documented floor
        dmin = LOSS_REGISTRY.get(n).get("disjoint_min", 0.9)
        v = fn(_b([[0,0,10,10]]), _b([[50,50,60,60]])).item()
        ck(torch.isfinite(torch.tensor(v)) and v >= dmin, f"{n}: disjoint finite (>= {dmin})")
        # partial overlap finite
        v = fn(_b([[0,0,20,20]]), _b([[10,10,30,30]])).item()
        ck(torch.isfinite(torch.tensor(v)), f"{n}: partial finite")
        # containment both ways finite
        ck(torch.isfinite(fn(_b([[5,5,25,25]]), _b([[0,0,40,40]]))).all(), f"{n}: pred-in-gt")
        ck(torch.isfinite(fn(_b([[0,0,40,40]]), _b([[5,5,25,25]]))).all(), f"{n}: gt-in-pred")
        # degenerate / invalid boxes finite
        ck(torch.isfinite(fn(_b([[10,10,10,10]]), _b([[0,0,20,20]]))).all(), f"{n}: zero-area finite")
        ck(torch.isfinite(fn(_b([[30,30,10,10]]), _b([[0,0,20,20]]))).all(), f"{n}: inverted finite")
        # small & large-coordinate boxes
        ck(torch.isfinite(fn(_b([[0,0,1,1]]), _b([[0,0,2,2]]))).all(), f"{n}: tiny finite")
        ck(torch.isfinite(fn(_b([[1e4,1e4,1e4+50,1e4+50]]), _b([[1e4+10,1e4+10,1e4+70,1e4+70]]))).all(), f"{n}: large-coord finite")
        # float16 forward finite
        ck(torch.isfinite(fn(_b([[2,3,18,22]]).half(), _b([[10,10,30,30]]).half()).float()).all(), f"{n}: fp16 finite")
        # finite non-zero gradient through predicted coords
        pr = _b([[2,3,18,22]]).clone().requires_grad_(True)
        fn(pr, _b([[10,10,30,30]])).sum().backward()
        ck(torch.isfinite(pr.grad).all() and pr.grad.abs().sum() > 0, f"{n}: grad finite&nonzero")
        # monotone: identical < partial overlap < disjoint
        l_id = fn(ident, ident.clone()).item()
        l_pa = fn(_b([[0,0,20,20]]), _b([[10,10,30,30]])).item()
        l_dj = fn(_b([[0,0,10,10]]), _b([[50,50,60,60]])).item()
        ck(l_id < l_pa < l_dj, f"{n}: monotone ({l_id:.4f} < {l_pa:.4f} < {l_dj:.4f})")
        # Scale invariance: training feeds normalised 0-1 boxes, these tests use pixels.
        # The 12.2b families are exact (clamped denominators) -> 1e-5; the IoU family
        # adds an ABSOLUTE eps to its denominators, which is a ~1e-4 relative effect at
        # normalised scale. That is pre-existing and deliberately left untouched, so the
        # IoU family is only checked for gross scale dependence.
        tol = 1e-3 if LOSS_REGISTRY.get(n).get("family", "iou") == "iou" else 1e-5
        pa, ga = _b([[2,3,18,22]]), _b([[10,10,30,30]])
        ck(abs(fn(pa, ga).item() - fn(pa / 640.0, ga / 640.0).item()) < tol,
           f"{n}: scale-invariant within {tol} (px vs normalised)")

    # IC-Arb specific
    p = _b([[2,3,18,22]]); g = _b([[10,10,30,30]])
    ck(abs(icarb_loss(p, g, alpha=1.0).item() - iou_loss(p, g).item()) < 1e-6, "icarb alpha=1 == 1-IoU")
    ck(abs(icarb_loss(p, g, alpha=0.0).item() - (1 - prediction_coverage(p, g).item())) < 1e-6, "icarb alpha=0 == 1-C_p")
    gfix = _b([[10,10,20,20]])
    ck(prediction_coverage(_b([[0,0,40,40]]), gfix).item() < prediction_coverage(gfix, gfix).item(),
       "enlarging prediction lowers C_p")
    under = _b([[12,12,15,15]])
    ck(abs(prediction_coverage(under, gfix).item() - 1.0) < 1e-4 and target_coverage(under, gfix).item() < 0.5,
       "undersized-inside: C_p~1 but C_t low (asymmetric limitation)")
    ck(abs(icarb_loss(p, g, 0.2).item() - icarb_loss(p, g, 0.8).item()) > 1e-3, "alpha changes loss")
    ic, cc, tot = icarb_components(p, g, 0.35)
    ck((ic + cc - tot).abs().max().item() < 1e-9, "icarb components sum to total")
    ck(abs(giou_loss(p, g).item() - icarb_loss(p, g, 0.5).item()) > 1e-3, "baseline vs icarb differ")

    # --- Section 12.2b families: closed-form identities that pin the implementations --
    ck(abs(tversky_loss(p, g, 0.5, 0.5).item() - dice_loss(p, g).item()) < 1e-9,
       "tversky(0.5,0.5) == dice")
    ck(abs(focal_tversky_loss(p, g, 0.3, 0.7, 1.0).item() - tversky_loss(p, g, 0.3, 0.7).item()) < 1e-9,
       "focal_tversky(gamma=1) == tversky")
    ck(focal_tversky_loss(p, g, gamma=0.75).item() > focal_tversky_loss(p, g, gamma=1.5).item(),
       "focal_tversky gamma<1 up-weights a hard pair")
    ck(tversky_loss(p, g, 0.1, 0.9).item() != tversky_loss(p, g, 0.9, 0.1).item(),
       "tversky alpha/beta asymmetry is active")
    ck(abs(generalized_surface_loss(p, g, lam=0.0).item() - dice_loss(p, g).item()) < 1e-12,
       "gsl(lam=0) == dice")
    ck(abs(generalized_surface_loss(p, g, lam=1.0).item() - boundary_loss(p, g).item()) < 1e-12,
       "gsl(lam=1) == boundary")
    ck(abs(boundary_loss(ident, ident.clone()).item()) < 1e-12, "boundary == 0 for identical boxes")
    # THE Lovasz correctness proof: the Lovasz extension must agree with the set
    # function it extends whenever the margins are binary -> exactly 2*(1-IoU).
    for _pp, _gg in [(p, g), (ident, ident.clone()),
                     (_b([[0,0,20,20]]), _b([[10,10,30,30]])),
                     (_b([[0,0,10,10]]), _b([[50,50,60,60]])),
                     (_b([[5,5,25,25]]), _b([[0,0,40,40]])),
                     (_b([[0,0,40,40]]), _b([[5,5,25,25]]))]:
        ck(abs(lovasz_hinge_loss(_pp, _gg, kappa=0.0).item() - 2.0 * iou_loss(_pp, _gg).item()) < 1e-6,
           "lovasz_hinge(kappa=0) == 2*(1-IoU)")
    ck(lovasz_hinge_loss(p, g, kappa=1.0).item() > lovasz_hinge_loss(p, g, kappa=0.0).item(),
       "lovasz_hinge kappa adds the surface term")
    # p == g must be a minimum of every new loss (no spurious optimum shift)
    for _n in ("dice", "tversky", "focal_tversky", "boundary", "gsl", "lovasz_hinge"):
        _f, _ = LOSS_REGISTRY.make(_n)
        _base = _b([[10., 10., 30., 30.]]); _l0 = _f(_base, _base.clone()).item()
        _worse = all(_f(_base + _d, _base.clone()).item() >= _l0 - 1e-9
                     for _d in [_b([[.1,0,0,0]]), _b([[-.1,0,0,0]]), _b([[0,0,.1,0]]),
                                _b([[0,0,-.1,0]]), _b([[.1,.1,.1,.1]]), _b([[-.1,-.1,-.1,-.1]])])
        ck(_worse, f"{_n}: p == g is a local minimum")
    return failures

_LOSS_FAILS = run_loss_unit_tests(verbose=SMOKE_TEST)
if _LOSS_FAILS:
    for m in _LOSS_FAILS: print("  FAIL:", m)
    raise AssertionError(f"{len(_LOSS_FAILS)} loss unit tests FAILED — aborting.")
print(f"LOSS UNIT TESTS PASSED ({len(LOSS_REGISTRY.list())} losses).")

# Persist the registry JSON snapshot (run-independent shared artifact).
try:
    atomic_write_json(os.path.join(CFG.paths.registry, "loss_registry.json"),
                      LOSS_REGISTRY.to_json_dict())
    print("  saved loss_registry.json")
except Exception as _e:
    print("  (registry snapshot skipped:", _e, ")")


LOSS UNIT TESTS PASSED (15 losses).
  saved loss_registry.json


## 14 · RT-DETR custom-criterion integration

We **audit** the installed Ultralytics RT-DETR loss, then integrate the custom
overlap loss by *subclassing* `RTDETRDetectionLoss` and overriding **only**
`_get_loss_bbox` — replacing the post-matching IoU term (`loss_giou`) while
leaving the **L1 term untouched**. Injection uses a version-compatible
monkeypatch of `RTDETRDetectionModel.init_criterion`, which the model's `loss()`
method is guaranteed to call. This is **not** a fragile callback.

**Integration scope (documented):** the override replaces the post-matching
IoU-family regression term for the main decoder layer, all **auxiliary**
decoder-layer losses, and the **denoising** query losses (every path that calls
`_get_loss_bbox`). The **Hungarian matching cost** (`HungarianMatcher`, which uses
its own GIoU cost) is left unchanged, as is the L1 (`loss_bbox`) term.


In [ ]:
# === 14.1 Audit the installed RT-DETR loss path ========================= #
import inspect
def audit_rtdetr_loss():
    report = {}
    try:
        from ultralytics.models.utils.loss import DETRLoss, RTDETRDetectionLoss
        from ultralytics.nn.tasks import RTDETRDetectionModel
        report["DETRLoss"] = "found"
        report["RTDETRDetectionLoss"] = "found"
        report["has__get_loss_bbox"] = hasattr(DETRLoss, "_get_loss_bbox")
        sig = inspect.signature(DETRLoss._get_loss_bbox)
        report["_get_loss_bbox_sig"] = str(sig)
        report["init_criterion_src"] = inspect.getsource(RTDETRDetectionModel.init_criterion).strip()
        report["compatible"] = report["has__get_loss_bbox"] and \
            set(["pred_bboxes", "gt_bboxes", "postfix"]).issubset(set(sig.parameters))
    except Exception as e:
        report["error"] = f"{type(e).__name__}: {e}"; report["compatible"] = False
    return report

RTDETR_AUDIT = audit_rtdetr_loss()
print(json.dumps({k: v for k, v in RTDETR_AUDIT.items() if k != "init_criterion_src"}, indent=2))
print("\ninit_criterion source:\n", RTDETR_AUDIT.get("init_criterion_src", "N/A"))
assert RTDETR_AUDIT.get("compatible"), "Installed RT-DETR loss is INCOMPATIBLE with the integration — aborting."


{
  "DETRLoss": "found",
  "RTDETRDetectionLoss": "found",
  "has__get_loss_bbox": true,
  "_get_loss_bbox_sig": "(self, pred_bboxes: 'torch.Tensor', gt_bboxes: 'torch.Tensor', postfix: 'str' = '') -> 'dict[str, torch.Tensor]'",
  "compatible": true
}

init_criterion source:
 def init_criterion(self):
        """Initialize the loss criterion for the RTDETRDetectionModel."""
        from ultralytics.models.utils.loss import RTDETRDetectionLoss

        return RTDETRDetectionLoss(nc=self.nc, use_vfl=True)


In [ ]:
# === 14.2 Custom criterion subclass ===================================== #
from ultralytics.models.utils.loss import RTDETRDetectionLoss

# Module-level spec read by the patched init_criterion (set per run).
ACTIVE_CRITERION_SPEC = {"overlap_name": "giou", "overlap_params": {}, "overlap_weight": 3.0,
                        "cls_loss": "vfl"}

# Classification-head loss selector -> upstream DETRLoss flags.
#   vfl : Varifocal          (RT-DETR default; use_fl=True, use_vfl=True)
#   fl  : Focal cross-entropy(use_fl=True,  use_vfl=False)
#   bce : plain (binary) cross-entropy, BCEWithLogitsLoss (use_fl=False)
# Verified against ultralytics DETRLoss._get_loss_class: `if self.fl: ... vfl/fl`,
# `else: BCEWithLogitsLoss`. "vfl" reproduces the behaviour of Runs 01-30 byte for byte.
CLS_LOSS_FLAGS = {"vfl": {"use_fl": True,  "use_vfl": True},
                  "fl":  {"use_fl": True,  "use_vfl": False},
                  "bce": {"use_fl": False, "use_vfl": False}}

def cls_loss_flags(name):
    if name not in CLS_LOSS_FLAGS:
        raise KeyError(f"cls_loss '{name}' unknown. Available: {sorted(CLS_LOSS_FLAGS)}")
    return dict(CLS_LOSS_FLAGS[name])

class ICArbRTDETRLoss(RTDETRDetectionLoss):
    """RT-DETR criterion that swaps ONLY the post-matching IoU regression term.

    Scope: overrides ``_get_loss_bbox`` -> affects main + auxiliary + denoising
    box-overlap losses. L1 (``loss_bbox``) and Hungarian matching are untouched.
    """
    def __init__(self, *args, overlap_name="giou", overlap_params=None,
                 overlap_weight=3.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.overlap_name = overlap_name
        self.overlap_params = LOSS_REGISTRY.validate_params(overlap_name, overlap_params or {})
        self.overlap_weight = float(overlap_weight)
        self.overlap_fn, _ = LOSS_REGISTRY.make(overlap_name, self.overlap_params)
        self.call_count = 0
        self._running = {"overlap": 0.0, "icarb_iou": 0.0, "icarb_cov": 0.0, "n": 0}

    def reset_running(self):
        self._running = {"overlap": 0.0, "icarb_iou": 0.0, "icarb_cov": 0.0, "n": 0}

    def running_means(self):
        n = max(1, self._running["n"])
        return {"overlap": self._running["overlap"] / n,
                "icarb_iou_component": self._running["icarb_iou"] / n,
                "icarb_coverage_component": self._running["icarb_cov"] / n}

    def _get_loss_bbox(self, pred_bboxes, gt_bboxes, postfix=""):
        import torch.nn.functional as F
        name_bbox = f"loss_bbox{postfix}"; name_giou = f"loss_giou{postfix}"
        loss = {}
        if len(gt_bboxes) == 0:
            loss[name_bbox] = torch.tensor(0.0, device=self.device)
            loss[name_giou] = torch.tensor(0.0, device=self.device)
            return loss
        # L1 term — identical to upstream, untouched.
        loss[name_bbox] = self.loss_gain["bbox"] * F.l1_loss(pred_bboxes, gt_bboxes, reduction="sum") / len(gt_bboxes)
        # Overlap term — custom, on xyxy.
        p = xywh2xyxy_t(pred_bboxes); g = xywh2xyxy_t(gt_bboxes)
        per_pair = self.overlap_fn(p, g)
        overlap = per_pair.sum() / len(gt_bboxes)
        loss[name_giou] = self.overlap_weight * overlap
        # bookkeeping for logging (detached)
        self.call_count += 1
        with torch.no_grad():
            self._running["overlap"] += float(overlap.detach()); self._running["n"] += 1
            if self.overlap_name == "icarb":
                a = self.overlap_params.get("alpha", 0.5)
                ic, cc, _ = icarb_components(p.detach(), g.detach(), alpha=a)
                self._running["icarb_iou"] += float(ic.sum() / len(gt_bboxes))
                self._running["icarb_cov"] += float(cc.sum() / len(gt_bboxes))
        return {k: v.squeeze() for k, v in loss.items()}

def install_custom_criterion(overlap_name, overlap_params=None, overlap_weight=3.0,
                             l1_weight=5.0, cls_loss="vfl"):
    """Patch RTDETRDetectionModel.init_criterion so training uses our criterion."""
    from ultralytics.nn.tasks import RTDETRDetectionModel
    flags = cls_loss_flags(cls_loss)          # raises early on an unknown name
    ACTIVE_CRITERION_SPEC.update({"overlap_name": overlap_name,
                                  "overlap_params": overlap_params or {},
                                  "overlap_weight": overlap_weight,
                                  "l1_weight": l1_weight,
                                  "cls_loss": cls_loss})
    def _init(self):
        nc = getattr(self, "nc", None)
        if nc is None:
            nc = self.model[-1].nc
        crit = ICArbRTDETRLoss(
            nc=nc, **cls_loss_flags(ACTIVE_CRITERION_SPEC["cls_loss"]),
            overlap_name=ACTIVE_CRITERION_SPEC["overlap_name"],
            overlap_params=ACTIVE_CRITERION_SPEC["overlap_params"],
            overlap_weight=ACTIVE_CRITERION_SPEC["overlap_weight"])
        crit.cls_loss_name = ACTIVE_CRITERION_SPEC["cls_loss"]
        # Freeze coefficients: overlap -> giou gain, L1 -> bbox gain.
        crit.loss_gain["giou"] = 1.0            # weight already applied inside overlap term
        crit.loss_gain["bbox"] = float(ACTIVE_CRITERION_SPEC["l1_weight"])
        return crit
    RTDETRDetectionModel.init_criterion = _init
    return _init

print("Custom criterion class + installer ready. cls_loss options:", sorted(CLS_LOSS_FLAGS))


Custom criterion class + installer ready. cls_loss options: ['bce', 'fl', 'vfl']


## 15 · Criterion integration verification (gradient check)

Automated proof — run before **every** training run — that the selected custom
loss is actually integrated. Verifies: the custom criterion is instantiated; the
selected loss is invoked; the returned loss is in the computation graph;
gradients flow through predicted box coordinates; the loss is non-zero for a
mismatched synthetic pair; changing α changes the loss; and baseline vs IC-Arb
differ on the same input. Prints **`CUSTOM LOSS INTEGRATION VERIFIED`** on
success and **aborts** otherwise.


In [ ]:
def verify_criterion_integration(overlap_name="icarb", overlap_params=None, overlap_weight=3.0,
                                 run_full_forward=False):
    overlap_params = overlap_params or ({"alpha": 0.35} if overlap_name == "icarb" else {})
    results = {}
    dev = torch.device("cpu")
    crit = ICArbRTDETRLoss(nc=1, use_vfl=True, overlap_name=overlap_name,
                           overlap_params=overlap_params, overlap_weight=overlap_weight)
    crit.device = dev
    # synthetic matched pairs (xywh normalized)
    pred = torch.tensor([[0.50, 0.50, 0.20, 0.20], [0.30, 0.30, 0.10, 0.15]], requires_grad=True)
    gt = torch.tensor([[0.52, 0.52, 0.25, 0.25], [0.31, 0.29, 0.20, 0.20]])
    out = crit._get_loss_bbox(pred, gt)
    results["custom_instantiated"] = isinstance(crit, ICArbRTDETRLoss)
    results["loss_invoked"] = crit.call_count >= 1
    results["keys_ok"] = ("loss_giou" in out and "loss_bbox" in out)
    total = out["loss_giou"] + out["loss_bbox"]
    results["in_graph"] = bool(total.requires_grad)
    total.backward()
    results["grad_through_boxes"] = bool(torch.isfinite(pred.grad).all() and pred.grad.abs().sum() > 0)
    # mismatched pair non-zero
    mp = torch.tensor([[0.2, 0.2, 0.1, 0.1]]); mg = torch.tensor([[0.8, 0.8, 0.1, 0.1]])
    c2 = ICArbRTDETRLoss(nc=1, overlap_name=overlap_name, overlap_params=overlap_params, overlap_weight=overlap_weight)
    c2.device = dev
    results["nonzero_mismatch"] = c2._get_loss_bbox(mp, mg)["loss_giou"].item() > 1e-3
    # alpha sensitivity (icarb only) / else parameter-agnostic pass
    if overlap_name == "icarb":
        def val(a):
            c = ICArbRTDETRLoss(nc=1, overlap_name="icarb", overlap_params={"alpha": a}, overlap_weight=overlap_weight)
            c.device = dev; return c._get_loss_bbox(pred.detach(), gt)["loss_giou"].item()
        results["alpha_changes_loss"] = abs(val(0.2) - val(0.8)) > 1e-4
    else:
        results["alpha_changes_loss"] = True
    # baseline vs icarb differ
    cb = ICArbRTDETRLoss(nc=1, overlap_name="giou", overlap_weight=overlap_weight); cb.device = dev
    ci = ICArbRTDETRLoss(nc=1, overlap_name="icarb", overlap_params={"alpha": 0.5}, overlap_weight=overlap_weight); ci.device = dev
    vb = cb._get_loss_bbox(pred.detach(), gt)["loss_giou"].item()
    vi = ci._get_loss_bbox(pred.detach(), gt)["loss_giou"].item()
    results["baseline_vs_icarb_differ"] = abs(vb - vi) > 1e-4

    if run_full_forward:
        try:
            from ultralytics import RTDETR
            from ultralytics.nn.tasks import RTDETRDetectionModel
            install_custom_criterion(overlap_name, overlap_params, overlap_weight)
            m = RTDETR(CFG.model.variant.replace(".pt", ".yaml")); net = m.model
            if not hasattr(net, "nc"): net.nc = net.model[-1].nc
            net.train()
            b = {"img": torch.rand(2, 3, 640, 640), "cls": torch.zeros(2, 1),
                 "bboxes": torch.tensor([[0.5, 0.5, 0.2, 0.3], [0.4, 0.6, 0.25, 0.25]]),
                 "batch_idx": torch.tensor([0, 1])}
            tl, items = net.loss(b)
            results["full_forward_invoked"] = net.criterion.call_count > 0
            results["full_forward_finite"] = bool(torch.isfinite(tl) and tl.requires_grad)
        except Exception as e:
            results["full_forward_error"] = f"{type(e).__name__}: {e}"
            results["full_forward_invoked"] = False; results["full_forward_finite"] = False

    passed = all(v is True for k, v in results.items() if not k.endswith("_error"))
    print(json.dumps(results, indent=2))
    if passed:
        print("\n" + "=" * 46 + "\nCUSTOM LOSS INTEGRATION VERIFIED\n" + "=" * 46)
    else:
        raise AssertionError("CUSTOM LOSS INTEGRATION FAILED — aborting run. Details above.")
    return results

# Verify on the proposed loss (unit-level always; full forward is optional/slow).
_VERIFY = verify_criterion_integration("icarb", {"alpha": 0.35}, 3.0, run_full_forward=False)


{
  "custom_instantiated": true,
  "loss_invoked": true,
  "keys_ok": true,
  "in_graph": true,
  "grad_through_boxes": true,
  "nonzero_mismatch": true,
  "alpha_changes_loss": true,
  "baseline_vs_icarb_differ": true
}

CUSTOM LOSS INTEGRATION VERIFIED


## 16 · Experiment registry — 37 runs

Fixed ordering, stable run IDs, never renumbered after runs start.

| Runs | Group | What varies |
|---|---|---|
| 01–08 | `baseline` | IoU-family overlap loss (GIoU, DIoU, CIoU, EIoU, SIoU, α-IoU, WIoU, PIoU) |
| 09–20 | `A_alpha_sweep` | IC-Arb α at fixed box weight 3.0 |
| 21–26 | `B_box_weight` | box-loss weight at α ∈ {0.3, 0.5} |
| 27–29 | `C_interaction` | selected α × weight interactions |
| 30 | `C_cls_ablation` | **auxiliary, superseded** — see the note below |
| 31–35 | `D_seg_loss` | segmentation-literature overlap loss (Dice, Focal Tversky, Boundary, Generalized Surface, Lovász-Hinge) |
| 36–37 | `E_cls_ablation` | **auxiliary** — classification head: Cross-Entropy / Focal Cross-Entropy |

**Runs 31–35** follow exactly the same single-factor protocol as Runs 01–08: only the
post-matching overlap term changes, at the frozen shared box weight 3.0 and VFL
classification head, so they are directly comparable to both the IoU baselines and
IC-Arb. Their box formulations are exact (Section 12.2b) — a box *is* a mask, so the
TP/FP/FN areas and the surface distances are closed-form, with no rasterisation.

**Runs 36–37** change the classification head instead (IC-Arb α=0.50 / weight 3.0 is
pinned, identical to Run 16), so Runs 16 / 36 / 37 form a clean VFL vs CE vs Focal-CE
comparison. Like Run 30 they are auxiliary and are excluded from the controlled
α / box-weight trends and from the primary baseline-vs-IC-Arb comparison.

> **Disclosure about Run 30.** `RunSpec.cls_loss` was recorded in the registry but was
> never passed to the criterion before Section 14.2 gained the `CLS_LOSS_FLAGS` wiring,
> so the already-completed Run 30 in fact trained with VFL and duplicates Run 16 — it is
> *not* a Focal-Loss ablation. Run 30 is left pinned to `"vfl"` so its finished artifacts
> stay self-consistent; **Run 37 is the corrected Focal-CE re-run.**

All controlled runs share the same seed, model, split, optimizer, schedule, image size,
batch size, epochs, augmentation and L1 configuration.


In [ ]:
# === 16.1 Build the run registry ======================================== #
from dataclasses import dataclass, field, asdict as _asdict

# 8 IoU baselines + 22 IC-Arb + 5 segmentation-loss baselines + 2 cls ablations.
# TOTAL_RUNS is also referenced by the Section 00.1 run selector.
TOTAL_RUNS = 37

@dataclass
class RunSpec:
    run_id: int
    run_name: str
    experiment_group: str
    loss_name: str
    loss_params: dict = field(default_factory=dict)
    overlap_loss_weight: float = 3.0
    cls_loss: str = "vfl"                 # varifocal (RT-DETR default)
    cls_loss_weight: float = 1.0
    l1_weight: float = 5.0
    seed: int = 1337
    optimizer: str = "AdamW"
    lr0: float = 0.0005
    imgsz: int = 640
    batch: int = 12
    epochs: int = 100
    patience: int = 50
    augment: dict = field(default_factory=dict)
    notes: str = ""
    status: str = "pending"

    def slug(self):
        p = ""
        if self.loss_name == "icarb":
            a = self.loss_params.get("alpha", 0.5)
            p = f"_alpha_{int(round(a*100)):03d}"
        w = f"_w{int(round(self.overlap_loss_weight*10)):02d}"
        # cls suffix only for the non-default head, so Runs 01-30 keep their exact
        # existing folder names (their artifacts already live under those paths).
        c = "" if self.cls_loss == "vfl" else f"_cls{self.cls_loss}"
        return f"run_{self.run_id:03d}_{self.loss_name}{p}{w}{c}"


def default_augment():
    a = CFG.augment
    return {"hsv_h": 0.0, "hsv_s": 0.0, "hsv_v": 0.0, "mosaic": a.mosaic,
            "fliplr": a.fliplr, "degrees": a.degrees, "translate": a.translate,
            "scale": a.scale}


def build_experiment_registry():
    seed = CFG.train.seed
    aug = default_augment()
    runs = []
    # --- 8 baselines (Runs 01-08) ---
    baseline_losses = [("giou", "GIoU"), ("diou", "DIoU"), ("ciou", "CIoU"),
                       ("eiou", "EIoU"), ("siou", "SIoU"), ("alpha_iou", "Alpha-IoU"),
                       ("wiou", "WIoU"), ("piou", "PIoU")]
    for i, (ln, disp) in enumerate(baseline_losses, start=1):
        runs.append(RunSpec(run_id=i, run_name=f"baseline_{ln}", experiment_group="baseline",
                            loss_name=ln, loss_params=({"alpha": 3.0} if ln == "alpha_iou" else {}),
                            overlap_loss_weight=CFG.train.overlap_loss_weight, seed=seed,
                            lr0=CFG.train.lr0, imgsz=CFG.model.imgsz, batch=CFG.train.batch,
                            epochs=CFG.train.epochs, patience=CFG.train.patience, augment=dict(aug),
                            notes=f"{disp} baseline (frozen shared config)"))
    # --- Group A: controlled alpha sweep, weight 3.0 (Runs 09-20) ---
    alphas = [0.00, 0.10, 0.20, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.75, 1.00]
    for j, a in enumerate(alphas):
        rid = 9 + j
        runs.append(RunSpec(run_id=rid, run_name=f"icarb_alpha_{a:.2f}", experiment_group="A_alpha_sweep",
                            loss_name="icarb", loss_params={"alpha": a}, overlap_loss_weight=3.0,
                            seed=seed, lr0=CFG.train.lr0, imgsz=CFG.model.imgsz, batch=CFG.train.batch,
                            epochs=CFG.train.epochs, patience=CFG.train.patience, augment=dict(aug),
                            notes=f"IC-Arb alpha={a} (controlled sweep)"))
    # --- Group B: box-loss weight sensitivity (Runs 21-26) ---
    groupB = [(0.30, 1.0), (0.30, 5.0), (0.30, 7.0), (0.50, 1.0), (0.50, 5.0), (0.50, 7.0)]
    for k, (a, w) in enumerate(groupB):
        rid = 21 + k
        runs.append(RunSpec(run_id=rid, run_name=f"icarb_a{a:.2f}_w{w:.1f}", experiment_group="B_box_weight",
                            loss_name="icarb", loss_params={"alpha": a}, overlap_loss_weight=w,
                            seed=seed, lr0=CFG.train.lr0, imgsz=CFG.model.imgsz, batch=CFG.train.batch,
                            epochs=CFG.train.epochs, patience=CFG.train.patience, augment=dict(aug),
                            notes=f"IC-Arb alpha={a}, box weight {w}"))
    # --- Group C: selected interactions (Runs 27-30) ---
    groupC = [(0.35, 5.0, "vfl"), (0.45, 5.0, "vfl"), (0.55, 5.0, "vfl")]
    for m, (a, w, cl) in enumerate(groupC):
        rid = 27 + m
        runs.append(RunSpec(run_id=rid, run_name=f"icarb_a{a:.2f}_w{w:.1f}", experiment_group="C_interaction",
                            loss_name="icarb", loss_params={"alpha": a}, overlap_loss_weight=w,
                            cls_loss=cl, seed=seed, lr0=CFG.train.lr0, imgsz=CFG.model.imgsz,
                            batch=CFG.train.batch, epochs=CFG.train.epochs, patience=CFG.train.patience,
                            augment=dict(aug), notes=f"IC-Arb alpha={a}, box weight {w}"))
    # Run 30: classification-loss ablation, AS ORIGINALLY EXECUTED.
    #   NOTE (disclosure): before Section 14.2 gained the cls_loss wiring, this field was
    #   recorded but never reached the criterion — the completed Run 30 actually trained
    #   with VFL, i.e. it is a duplicate of Run 16. Run 37 is the corrected re-run; keep
    #   Run 30 pinned to "vfl" so its finished artifacts stay self-consistent.
    runs.append(RunSpec(run_id=30, run_name="icarb_a0.50_w3.0_clsablation", experiment_group="C_cls_ablation",
                        loss_name="icarb", loss_params={"alpha": 0.50}, overlap_loss_weight=3.0,
                        cls_loss="vfl", cls_loss_weight=1.0, seed=seed, lr0=CFG.train.lr0,
                        imgsz=CFG.model.imgsz, batch=CFG.train.batch, epochs=CFG.train.epochs,
                        patience=CFG.train.patience, augment=dict(aug),
                        notes="AUXILIARY (superseded by Run 37): the cls_loss field was inert when this "
                              "run executed, so it trained with VFL and duplicates Run 16. Excluded from "
                              "controlled alpha/box-weight trends and primary baseline-vs-IC-Arb comparison."))
    # --- Group D: segmentation-literature overlap losses (Runs 31-35) ---
    # Same single-factor protocol as Runs 01-08: ONLY the post-matching overlap term
    # changes. Box weight 3.0 and cls_loss vfl are held at the frozen shared values, so
    # these are directly comparable to the IoU-family baselines and to IC-Arb.
    groupD = [
        ("dice",          {},                                            "Dice"),
        ("focal_tversky", {"alpha": 0.3, "beta": 0.7, "gamma": 0.75},    "Focal Tversky"),
        ("boundary",      {},                                            "Boundary / Surface"),
        ("gsl",           {"lam": 0.5},                                  "Generalized Surface"),
        ("lovasz_hinge",  {"kappa": 1.0},                                "Lovasz-Hinge"),
    ]
    for q, (ln, lp, disp) in enumerate(groupD):
        rid = 31 + q
        runs.append(RunSpec(run_id=rid, run_name=f"seg_{ln}", experiment_group="D_seg_loss",
                            loss_name=ln, loss_params=dict(lp),
                            overlap_loss_weight=CFG.train.overlap_loss_weight, seed=seed,
                            lr0=CFG.train.lr0, imgsz=CFG.model.imgsz, batch=CFG.train.batch,
                            epochs=CFG.train.epochs, patience=CFG.train.patience, augment=dict(aug),
                            notes=f"{disp} baseline, segmentation-loss family (frozen shared config)"))
    # --- Group E: classification-loss ablation (Runs 36-37) ---
    # Overlap term pinned to IC-Arb alpha=0.50 / weight 3.0 == Run 16, so Runs 16/36/37
    # form a clean single-factor VFL vs CE vs Focal-CE comparison on the cls head.
    groupE = [("bce", "Cross-Entropy (BCEWithLogits)"), ("fl", "Focal Cross-Entropy")]
    for q, (cl, disp) in enumerate(groupE):
        rid = 36 + q
        runs.append(RunSpec(run_id=rid, run_name=f"icarb_a0.50_w3.0_cls_{cl}",
                            experiment_group="E_cls_ablation",
                            loss_name="icarb", loss_params={"alpha": 0.50}, overlap_loss_weight=3.0,
                            cls_loss=cl, cls_loss_weight=1.0, seed=seed, lr0=CFG.train.lr0,
                            imgsz=CFG.model.imgsz, batch=CFG.train.batch, epochs=CFG.train.epochs,
                            patience=CFG.train.patience, augment=dict(aug),
                            notes=f"AUXILIARY classification-loss ablation: {disp}. Reference = Run 16 "
                                  f"(same overlap term, VFL). Excluded from the overlap-loss comparisons."))
    assert len(runs) == TOTAL_RUNS, f"expected {TOTAL_RUNS} runs, got {len(runs)}"
    ids = [r.run_id for r in runs]
    assert ids == list(range(1, TOTAL_RUNS + 1)), f"run ids must be 1..{TOTAL_RUNS} in order, got {ids}"
    return runs

EXPERIMENT_REGISTRY = build_experiment_registry()
# Runs excluded from the primary controlled conclusions (auxiliary cls-head ablations):
EXCLUDED_FROM_PRIMARY = {30, 36, 37}
# Groups that act as comparison baselines for the overlap term (all single-factor):
BASELINE_GROUPS = {"baseline", "D_seg_loss"}

def registry_hash():
    return sha256_text(json.dumps([_asdict(r) for r in EXPERIMENT_REGISTRY], sort_keys=True, default=str))

def save_experiment_registry():
    payload = [_asdict(r) for r in EXPERIMENT_REGISTRY]
    atomic_write_json(os.path.join(CFG.paths.registry, "experiment_registry.json"), payload)
    # csv summary
    import io, csv as _csv
    buf = io.StringIO()
    w = _csv.writer(buf)
    w.writerow(["run_id", "group", "loss", "family", "params", "alpha", "overlap_weight",
                "cls_loss", "epochs", "seed", "status"])
    for r in EXPERIMENT_REGISTRY:
        w.writerow([r.run_id, r.experiment_group, r.loss_name,
                    LOSS_REGISTRY.get(r.loss_name).get("family", "iou"),
                    json.dumps(r.loss_params, sort_keys=True), r.loss_params.get("alpha", ""),
                    r.overlap_loss_weight, r.cls_loss, r.epochs, r.seed, r.status])
    atomic_write_text(os.path.join(CFG.paths.registry, "experiment_registry.csv"), buf.getvalue())
    print(f"[16] saved registry ({len(EXPERIMENT_REGISTRY)} runs) hash={registry_hash()[:12]}")

def get_run(run_id):
    for r in EXPERIMENT_REGISTRY:
        if r.run_id == run_id:
            return r
    raise KeyError(f"run_id {run_id} not in registry")

save_experiment_registry()
print("\nRUN PLAN:")
for r in EXPERIMENT_REGISTRY:
    a = r.loss_params.get("alpha", "-")
    print(f"  Run {r.run_id:02d} | {r.experiment_group:16s} | {r.loss_name:13s} "
          f"| alpha={a} | w={r.overlap_loss_weight} | cls={r.cls_loss:3s} | {r.notes[:40]}")


[16] saved registry (37 runs) hash=a74f3e651b24

RUN PLAN:
  Run 01 | baseline         | giou          | alpha=- | w=3.0 | cls=vfl | GIoU baseline (frozen shared config)
  Run 02 | baseline         | diou          | alpha=- | w=3.0 | cls=vfl | DIoU baseline (frozen shared config)
  Run 03 | baseline         | ciou          | alpha=- | w=3.0 | cls=vfl | CIoU baseline (frozen shared config)
  Run 04 | baseline         | eiou          | alpha=- | w=3.0 | cls=vfl | EIoU baseline (frozen shared config)
  Run 05 | baseline         | siou          | alpha=- | w=3.0 | cls=vfl | SIoU baseline (frozen shared config)
  Run 06 | baseline         | alpha_iou     | alpha=3.0 | w=3.0 | cls=vfl | Alpha-IoU baseline (frozen shared config
  Run 07 | baseline         | wiou          | alpha=- | w=3.0 | cls=vfl | WIoU baseline (frozen shared config)
  Run 08 | baseline         | piou          | alpha=- | w=3.0 | cls=vfl | PIoU baseline (frozen shared config)
  Run 09 | A_alpha_sweep    | icarb         | a

## 17 · Run-execution engine

A robust runner: single / range / all-pending; skip completed; resume interrupted;
retry failed; dry-run. Every run starts from the **same** pretrained checkpoint
(SHA-256 verified), resets Python/NumPy/PyTorch/CUDA RNGs, and recreates optimizer,
scheduler and AMP scaler. The custom criterion is installed **and verified** before
each run. Status is written atomically; separate completion markers + manifests are
kept. Training completion is **not** treated as failure if deferred SAM/reporting
has not yet run.


In [ ]:
# === 17.1 Common initialization checkpoint (shared, hash-locked) ========= #
import shutil

def ensure_init_checkpoint():
    """Download the RT-DETR pretrained checkpoint once; lock its SHA-256.

    All runs in the registry start from these exact bytes. Never initialise a later run from a
    previous run's best.pt/last.pt.
    """
    shared_ckpt = os.path.join(CFG.paths.shared, CFG.model.variant)
    os.makedirs(CFG.paths.shared, exist_ok=True)
    if not os.path.exists(shared_ckpt):
        # Ultralytics auto-downloads to CWD on first RTDETR(variant); fetch then copy.
        from ultralytics import RTDETR
        _ = RTDETR(CFG.model.variant)     # triggers download of weights
        local = CFG.model.variant
        if os.path.exists(local):
            shutil.copy2(local, shared_ckpt)
        elif os.path.exists(os.path.join(os.getcwd(), CFG.model.variant)):
            shutil.copy2(os.path.join(os.getcwd(), CFG.model.variant), shared_ckpt)
        else:
            raise FileNotFoundError(f"Could not locate downloaded {CFG.model.variant}")
    h = sha256_file(shared_ckpt)
    meta_path = os.path.join(CFG.paths.shared, "init_checkpoint.json")
    prev = read_json(meta_path)
    if prev and prev.get("sha256") != h:
        raise AssertionError("Init checkpoint hash CHANGED — refusing to mix initializations.")
    atomic_write_json(meta_path, {"path": shared_ckpt, "sha256": h, "variant": CFG.model.variant})
    print(f"[17] init checkpoint {CFG.model.variant} sha256={h[:16]} -> {shared_ckpt}")
    return shared_ckpt, h

def reset_all_seeds(seed):
    import random as _r
    _r.seed(seed)
    try:
        import numpy as _np; _np.random.seed(seed)
    except Exception:
        pass
    try:
        import torch as _t
        _t.manual_seed(seed)
        if _t.cuda.is_available():
            _t.cuda.manual_seed_all(seed)
    except Exception:
        pass
    os.environ["PYTHONHASHSEED"] = str(seed)
    print(f"[17] RNGs reset to seed={seed}")

print("[17] init-checkpoint + seed helpers ready.")


[17] init-checkpoint + seed helpers ready.


In [ ]:
# === 17.2 Per-run folder layout + status (atomic) ======================= #
RUN_STATUSES = ["pending", "preprocessing", "training", "validating", "testing",
                "sam_evaluation", "completed", "failed", "interrupted"]

def run_dir(run):
    d = os.path.join(CFG.paths.experiments, run.slug())
    return d

def ensure_run_tree(run):
    d = run_dir(run)
    for sub in ["config", "checkpoints", "checkpoints/periodic", "logs", "validation",
                "validation/plots", "test", "test/visualizations", "sam", "sam/masks",
                "sam/overlays", "sam/failure_cases", "figures", "academic/tables/csv",
                "academic/tables/xlsx", "academic/tables/markdown", "academic/tables/latex",
                "academic/figures/png", "academic/figures/pdf", "academic/figures/svg",
                "academic/captions", "academic/manuscript_drafts", "ground_truth_evaluation",
                "ground_truth_evaluation/error_overlays", "ground_truth_evaluation/representative_cases",
                "summary"]:
        os.makedirs(os.path.join(d, sub), exist_ok=True)
    return d

def set_run_status(run, status, extra=None):
    assert status in RUN_STATUSES, status
    d = ensure_run_tree(run)
    payload = {"run_id": run.run_id, "slug": run.slug(), "status": status,
               "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}
    if extra:
        payload.update(extra)
    atomic_write_json(os.path.join(d, "summary", "status.json"), payload)
    return payload

def get_run_status(run):
    s = read_json(os.path.join(run_dir(run), "summary", "status.json"))
    return (s or {}).get("status", "pending")

def run_marker(run, name):
    return os.path.join(run_dir(run), name)

def marker_valid(run, name):
    """A marker is trusted only with a matching manifest of present outputs."""
    mk = run_marker(run, name)
    man = read_json(run_marker(run, name.replace(".flag", "_manifest.json")))
    if not os.path.exists(mk) or man is None:
        return False
    return all(os.path.exists(o) for o in man.get("outputs", []))

def write_marker(run, name, config, outputs, extra=None):
    man = {"run_id": run.run_id, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
           "config": config, "outputs": outputs}
    if extra:
        man.update(extra)
    atomic_write_json(run_marker(run, name.replace(".flag", "_manifest.json")), man)
    touch_flag(run_marker(run, name))

print("[17] run-folder + status helpers ready.")


[17] run-folder + status helpers ready.


In [ ]:
# === 17.3 Observability callbacks + loss-component logging =============== #
def make_training_callbacks(run, log_paths):
    """Return a dict of Ultralytics callbacks for observability + component logs.

    IMPORTANT: these are observability-only. The custom loss is integrated via
    the criterion subclass (Section 14), NOT via these callbacks.
    """
    import csv as _csv
    state = {"t0": time.time(), "best": -1.0, "best_epoch": -1}

    def _append_csv(path, row, header):
        newfile = not os.path.exists(path)
        with open(path, "a", newline="") as f:
            w = _csv.DictWriter(f, fieldnames=header)
            if newfile:
                w.writeheader()
            w.writerow(row)

    def on_train_epoch_end(trainer):
        ep = int(getattr(trainer, "epoch", 0))
        # Ultralytics exposes per-component losses via loss_items / tloss (running mean).
        loss_items = getattr(trainer, "loss_items", None)
        if loss_items is None:
            loss_items = getattr(trainer, "tloss", None)
        names = getattr(trainer, "loss_names", ["giou_loss", "cls_loss", "l1_loss"])
        comps = {}
        try:
            vals = [float(x) for x in (loss_items.tolist() if hasattr(loss_items, "tolist") else loss_items)]
            comps = {n: v for n, v in zip(names, vals)}
        except Exception:
            pass
        # custom criterion running means (icarb components)
        crit = getattr(trainer.model, "criterion", None)
        icarb = crit.running_means() if hasattr(crit, "running_means") else {}
        if hasattr(crit, "reset_running"):
            crit.reset_running()
        row = {"epoch": ep, **comps,
               "icarb_iou_component": icarb.get("icarb_iou_component", ""),
               "icarb_coverage_component": icarb.get("icarb_coverage_component", ""),
               "elapsed": fmt_dur(time.time() - state["t0"]), "gpu": resource_summary()}
        _append_csv(log_paths["loss_components"], row, list(row.keys()))
        print(f"[{now_hms()}] Run {run.run_id:02d} | epoch {ep+1}/{run.epochs} | "
              f"{ {k: round(v,4) for k,v in comps.items()} } | "
              f"icarb={ {k: (round(v,4) if isinstance(v,float) else v) for k,v in icarb.items()} } | "
              f"Elapsed={fmt_dur(time.time()-state['t0'])} | {resource_summary()}", flush=True)

    def on_fit_epoch_end(trainer):
        ep = int(getattr(trainer, "epoch", 0))
        m = getattr(trainer, "metrics", {}) or {}
        key = "metrics/mAP50-95(B)"
        cur = float(m.get(key, m.get("metrics/mAP50-95", 0.0)) or 0.0)
        if cur > state["best"]:
            state["best"] = cur; state["best_epoch"] = ep
        row = {"epoch": ep, **{k: v for k, v in m.items()},
               "best_map50_95": state["best"], "best_epoch": state["best_epoch"]}
        _append_csv(log_paths["epoch_metrics"], row, list(row.keys()))
        print(f"[{now_hms()}] Run {run.run_id:02d} | val mAP50-95={cur:.4f} "
              f"(best={state['best']:.4f}@{state['best_epoch']}) ", flush=True)

    def on_model_save(trainer):
        print(f"[{now_hms()}] Run {run.run_id:02d} | checkpoint saved (epoch {getattr(trainer,'epoch',0)})", flush=True)

    return {"on_train_epoch_end": on_train_epoch_end,
            "on_fit_epoch_end": on_fit_epoch_end,
            "on_model_save": on_model_save}

print("[17] observability callbacks ready.")


[17] observability callbacks ready.


In [ ]:
# === 17.4 train_one_run: the core training entry ======================== #
def save_run_config(run, resolved):
    d = ensure_run_tree(run)
    cfgdir = os.path.join(d, "config")
    atomic_write_json(os.path.join(cfgdir, "requested_config.json"), _asdict(run))
    atomic_write_json(os.path.join(cfgdir, "resolved_config.json"), resolved)
    atomic_write_json(os.path.join(cfgdir, "environment.json"), ENVIRONMENT)
    atomic_write_json(os.path.join(cfgdir, "dataset_reference.json"),
                      {"dataset_yaml": CFG.paths.dataset_yaml,
                       "split_summary": read_json(os.path.join(CFG.paths.metadata_dir, "split_summary.json")),
                       "preprocessing": read_json(os.path.join(CFG.paths.metadata_dir, "preprocessing_config.json"))})
    loss_meta = LOSS_REGISTRY.get(run.loss_name)
    atomic_write_text(os.path.join(cfgdir, "loss_definition.txt"),
                      f"{loss_meta['display_name']} ({run.loss_name})\n"
                      f"reference: {loss_meta['reference']}\nparams: {run.loss_params}\n"
                      f"overlap_weight: {run.overlap_loss_weight}\nfamily: {loss_meta.get('family', 'iou')}\n"
                      f"notes: {loss_meta['notes']}\n"
                      f"classification loss: {run.cls_loss}\n"
                      f"integration scope: post-matching overlap term (main+aux+DN) and the "
                      f"classification-head loss selector; L1 & Hungarian matching untouched.\n")

def train_one_run(run, dry_run=False):
    from ultralytics import RTDETR
    d = ensure_run_tree(run)
    logs = {
        "loss_components": os.path.join(d, "logs", "loss_components.csv"),
        "epoch_metrics": os.path.join(d, "logs", "epoch_metrics.csv"),
        "console": os.path.join(d, "logs", "console.log"),
    }
    init_ckpt, init_hash = ensure_init_checkpoint()
    epochs = CFG.smoke.epochs if SMOKE_TEST else run.epochs
    resolved = {**_asdict(run), "epochs": epochs, "init_checkpoint": init_ckpt,
                "init_checkpoint_sha256": init_hash, "dataset_yaml": CFG.paths.dataset_yaml,
                "imgsz": run.imgsz, "batch": run.batch, "amp": CFG.train.amp,
                "optimizer": run.optimizer, "lr0": run.lr0, "weight_decay": CFG.train.weight_decay,
                "registry_hash": registry_hash()}
    save_run_config(run, resolved)
    print("\n" + "=" * 70)
    print(f"RUN {run.run_id:02d} · {run.slug()} · loss={run.loss_name} "
          f"params={run.loss_params} overlap_w={run.overlap_loss_weight} "
          f"cls={run.cls_loss} epochs={epochs}")
    print("=" * 70, flush=True)

    if dry_run:
        print("[dry-run] configuration resolved & saved; not training.")
        return {"dry_run": True, "resolved": resolved}

    # 1) install + verify the custom criterion (aborts on failure)
    install_custom_criterion(run.loss_name, run.loss_params, run.overlap_loss_weight,
                             run.l1_weight, cls_loss=run.cls_loss)
    verify_criterion_integration(run.loss_name, run.loss_params, run.overlap_loss_weight,
                                 run_full_forward=False)
    # 2) reset RNGs and build the model. Resuming an interrupted run must load the model
    #    FROM its own last.pt (carries optimizer + epoch state) — building from init_ckpt and
    #    then passing resume=True makes ultralytics silently RESTART at epoch 1, because the
    #    pretrained init has no training state to resume. Fresh runs build from init_ckpt.
    reset_all_seeds(run.seed)
    set_run_status(run, "training")
    project = os.path.join(CFG.paths.experiments, "_ultra"); name = run.slug()
    last_pt = os.path.join(project, name, "weights", "last.pt")
    # never resume during a smoke test — it must train the tiny split fresh, not continue stale weights
    resume = RESUME_IF_AVAILABLE and not SMOKE_TEST and os.path.exists(last_pt)
    model = RTDETR(last_pt if resume else init_ckpt)
    if resume:
        print(f"[17] Run {run.run_id:02d} RESUMING training from {last_pt}", flush=True)
    cbs = make_training_callbacks(run, logs)
    for ev, fn in cbs.items():
        model.add_callback(ev, fn)
    with Timer(f"training run {run.run_id}"):
        model.train(
            data=CFG.paths.dataset_yaml, epochs=epochs, imgsz=run.imgsz, batch=run.batch,
            optimizer=run.optimizer, lr0=run.lr0, weight_decay=CFG.train.weight_decay,
            patience=run.patience, seed=run.seed, amp=CFG.train.amp, cache=CFG.train.cache,
            workers=CFG.train.workers, project=project, name=name, exist_ok=True, resume=resume,
            hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, mosaic=run.augment.get("mosaic", 0.0),
            fliplr=run.augment.get("fliplr", 0.5), degrees=run.augment.get("degrees", 0.0),
            translate=run.augment.get("translate", 0.0), scale=run.augment.get("scale", 0.0),
            verbose=True,
        )
    # 3) copy best/last into the durable run folder + hash
    src_w = os.path.join(project, name, "weights")
    outs = []
    for w in ("best.pt", "last.pt"):
        s = os.path.join(src_w, w)
        if os.path.exists(s):
            dst = os.path.join(d, "checkpoints", w)
            shutil.copy2(s, dst); outs.append(dst)
    for res_png in glob.glob(os.path.join(project, name, "*.png")):
        shutil.copy2(res_png, os.path.join(d, "figures", os.path.basename(res_png)))
    best_hash = sha256_file(os.path.join(d, "checkpoints", "best.pt")) if os.path.exists(
        os.path.join(d, "checkpoints", "best.pt")) else None
    write_marker(run, "TRAINING_DONE.flag", resolved, outs,
                 extra={"best_sha256": best_hash, "init_sha256": init_hash})
    set_run_status(run, "training", {"training_done": True})
    print(f"[17] Run {run.run_id:02d} training complete. best.pt sha256={str(best_hash)[:16]}")
    return {"resolved": resolved, "checkpoints": outs, "best_sha256": best_hash}

print("[17] train_one_run ready.")


[17] train_one_run ready.


## 18–19 · Detection evaluation (validation & test)

`mAP50` / `mAP50-95` come from Ultralytics `val()`; precision/recall/F1/TP/FP/FN
use an explicit, documented matching procedure at a **fixed common threshold**
(shared by all runs for a fair comparison) and, separately, a validation-optimized
threshold. The **test split is only touched after** the best checkpoint and all
thresholds are frozen on validation. Distributions (mean/std/median/IQR/min/max/CI)
are reported, not just averages.


In [ ]:
# === 18.1 IoU + matching utilities (numpy, xyxy pixels) ================= #
def iou_xyxy_np(a, b):
    x1 = max(a[0], b[0]); y1 = max(a[1], b[1]); x2 = min(a[2], b[2]); y2 = min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area_a = max(0, a[2] - a[0]) * max(0, a[3] - a[1])
    area_b = max(0, b[2] - b[0]) * max(0, b[3] - b[1])
    return inter / (area_a + area_b - inter + 1e-9)

def load_gt_boxes(label_path, W, H):
    boxes = []
    if not os.path.exists(label_path):
        return boxes
    for ln in open(label_path).read().splitlines():
        if not ln.strip():
            continue
        _, cx, cy, w, h = map(float, ln.split())
        boxes.append([(cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H])
    return boxes

def greedy_match(preds, gts, iou_thr):
    """preds: list of (box, conf); gts: list of box. Greedy highest-confidence-first
    matching. Returns list of (pred_idx, gt_idx, iou)."""
    used = [False] * len(gts)
    matches = []
    order = sorted(range(len(preds)), key=lambda i: -preds[i][1])
    for pi in order:
        pb = preds[pi][0]; best_iou = 0; best_g = -1
        for gi, gb in enumerate(gts):
            if used[gi]:
                continue
            v = iou_xyxy_np(pb, gb)
            if v > best_iou:
                best_iou = v; best_g = gi
        if best_g >= 0 and best_iou >= iou_thr:
            used[best_g] = True; matches.append((pi, best_g, best_iou))
    return matches

print("[18] matching utilities ready.")


[18] matching utilities ready.


In [ ]:
# === 18.2 Detection evaluation over a split ============================= #
def evaluate_detection(run, split="test", conf=None, iou_match=None, ultra_map=True):
    from ultralytics import RTDETR
    import numpy as np
    conf = CFG.evaluation.conf_threshold if conf is None else conf
    iou_match = CFG.evaluation.iou_match_threshold if iou_match is None else iou_match
    ckpt = os.path.join(run_dir(run), "checkpoints", "best.pt")
    if not os.path.exists(ckpt):
        raise FileNotFoundError(f"best.pt missing for run {run.run_id}")
    model = RTDETR(ckpt)
    img_dir = os.path.join(CFG.paths.images_dir, split)
    images = sorted(glob.glob(os.path.join(img_dir, "*.png")))
    TP = FP = FN = 0
    confs = []; geom_rows = []; per_slice = []; latencies = []
    hb = Heartbeat(f"detect_eval[{split}]", every=10)
    for idx, img_p in enumerate(images):
        base = os.path.basename(img_p)[:-4]
        pid = "_".join(base.split("_")[:-1])
        from PIL import Image
        arr = np.array(Image.open(img_p)); H, W = arr.shape[:2]
        gts = load_gt_boxes(os.path.join(CFG.paths.labels_dir, split, base + ".txt"), W, H)
        t0 = time.time()
        r = model.predict(img_p, conf=conf, iou=CFG.evaluation.nms_iou, verbose=False)[0]
        latencies.append((time.time() - t0) * 1000.0)
        preds = []
        if r.boxes is not None and len(r.boxes) > 0:
            for b, c in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()):
                preds.append(([float(b[0]), float(b[1]), float(b[2]), float(b[3])], float(c)))
                confs.append(float(c))
        matches = greedy_match(preds, gts, iou_match)
        tp = len(matches); fp = len(preds) - tp; fn = len(gts) - tp
        TP += tp; FP += fp; FN += fn
        # geometry on matched TPs
        for pi, gi, iouv in matches:
            pb = preds[pi][0]; gb = gts[gi]
            pw = pb[2]-pb[0]; ph = pb[3]-pb[1]; gw = gb[2]-gb[0]; gh = gb[3]-gb[1]
            ix1=max(pb[0],gb[0]); iy1=max(pb[1],gb[1]); ix2=min(pb[2],gb[2]); iy2=min(pb[3],gb[3])
            ia=max(0,ix2-ix1)*max(0,iy2-iy1)
            cp = ia/max(1e-9, pw*ph); ct = ia/max(1e-9, gw*gh)
            geom_rows.append({"patient_id": pid, "slice": base, "iou": iouv,
                              "pred_coverage": cp, "target_coverage": ct,
                              "area_ratio": (pw*ph)/max(1e-9, gw*gh),
                              "center_dx": abs((pb[0]+pb[2])/2-(gb[0]+gb[2])/2)/W,
                              "center_dy": abs((pb[1]+pb[3])/2-(gb[1]+gb[3])/2)/H,
                              "width_err": abs(pw-gw)/W, "height_err": abs(ph-gh)/H,
                              "over_coverage": int(pw*ph > gw*gh), "under_coverage": int(pw*ph < gw*gh)})
        per_slice.append({"patient_id": pid, "slice": base, "n_gt": len(gts),
                          "n_pred": len(preds), "tp": tp, "fp": fp, "fn": fn})
        hb.beat(img=f"{idx+1}/{len(images)}", TP=TP, FP=FP, FN=FN)

    precision = TP / max(1, TP + FP); recall = TP / max(1, TP + FN)
    f1 = 2 * precision * recall / max(1e-9, precision + recall)
    metrics = {"split": split, "conf": conf, "iou_match": iou_match,
               "TP": TP, "FP": FP, "FN": FN, "precision": precision, "recall": recall,
               "f1": f1, "detection_rate": TP / max(1, TP + FN),
               "mean_latency_ms": float(np.mean(latencies)) if latencies else None,
               "throughput_fps": (1000.0/float(np.mean(latencies))) if latencies else None,
               "n_images": len(images)}
    if confs:
        metrics["conf_dist"] = {"mean": float(np.mean(confs)), "std": float(np.std(confs)),
                                 "median": float(np.median(confs)), "min": float(np.min(confs)),
                                 "max": float(np.max(confs))}
    if ultra_map:
        try:
            mm = model.val(data=CFG.paths.dataset_yaml, split=split, conf=0.001,
                           iou=CFG.evaluation.nms_iou, verbose=False)
            metrics["map50"] = float(getattr(mm.box, "map50", 0.0))
            metrics["map50_95"] = float(getattr(mm.box, "map", 0.0))
            metrics["mp"] = float(getattr(mm.box, "mp", 0.0)); metrics["mr"] = float(getattr(mm.box, "mr", 0.0))
        except Exception as e:
            metrics["map_error"] = str(e)
    # persist
    outdir = os.path.join(run_dir(run), split)
    os.makedirs(outdir, exist_ok=True)
    atomic_write_json(os.path.join(outdir, "metrics.json"), metrics)
    write_csv_dicts(os.path.join(outdir, "geometry_metrics.csv"), geom_rows)
    write_csv_dicts(os.path.join(outdir, "per_slice_metrics.csv"), per_slice)
    print(f"[19] Run {run.run_id:02d} [{split}] P={precision:.3f} R={recall:.3f} F1={f1:.3f} "
          f"mAP50={metrics.get('map50','NA')} mAP50-95={metrics.get('map50_95','NA')}")
    return metrics, geom_rows, per_slice

print("[19] detection evaluation ready.")


[19] detection evaluation ready.


## 20 · Box-geometry analysis

For matched true positives: IoU, prediction coverage `C_p`, target coverage `C_t`, area ratio, center-distance error, width/height error, over/under-coverage rates — with full distributions (mean, std, median, IQR, min, max, 95% CI).


In [ ]:
def distribution_stats(values):
    import numpy as np
    a = np.asarray([v for v in values if v is not None], dtype=float)
    if a.size == 0:
        return {"n": 0}
    q1, q3 = np.percentile(a, [25, 75])
    mean = float(a.mean()); std = float(a.std(ddof=1)) if a.size > 1 else 0.0
    # 95% bootstrap CI of the mean
    ci = None
    if a.size >= 5:
        import numpy as _np
        rng = _np.random.default_rng(0)
        boot = [_np.mean(rng.choice(a, size=a.size, replace=True)) for _ in range(1000)]
        ci = [float(_np.percentile(boot, 2.5)), float(_np.percentile(boot, 97.5))]
    return {"n": int(a.size), "mean": mean, "std": std, "median": float(np.median(a)),
            "iqr": [float(q1), float(q3)], "min": float(a.min()), "max": float(a.max()),
            "ci95": ci}

def summarize_geometry(run, split="test"):
    rows = read_csv_dicts(os.path.join(run_dir(run), split, "geometry_metrics.csv"))
    if not rows:
        print(f"[20] no geometry rows for run {run.run_id} [{split}]"); return {}
    fields = ["iou", "pred_coverage", "target_coverage", "area_ratio",
              "center_dx", "center_dy", "width_err", "height_err"]
    summary = {f: distribution_stats([float(r[f]) for r in rows if r.get(f) not in (None, "")]) for f in fields}
    summary["over_coverage_rate"] = sum(int(float(r["over_coverage"])) for r in rows) / len(rows)
    summary["under_coverage_rate"] = sum(int(float(r["under_coverage"])) for r in rows) / len(rows)
    summary["n_matched"] = len(rows)
    atomic_write_json(os.path.join(run_dir(run), split, "geometry_summary.json"), summary)
    print(f"[20] Run {run.run_id:02d} [{split}] geometry: IoU mean={summary['iou'].get('mean',0):.3f} "
          f"C_p={summary['pred_coverage'].get('mean',0):.3f} C_t={summary['target_coverage'].get('mean',0):.3f}")
    return summary

print("[20] geometry analysis ready.")


[20] geometry analysis ready.


## 21 · SAM initialization (segmenter registry)

`SEGMENTER_REGISTRY` supports `sam_vit_b|l|h` (+ optional MedSAM-compatible
checkpoint). The primary segmenter, checkpoint, preprocessing and prompt policy
are **frozen before test evaluation** and identical across all detector runs —
detector losses are never compared under different SAM settings. Checkpoints load
from Drive or download once.


In [ ]:
SEGMENTER_REGISTRY = {
    "sam_vit_b": {"model_type": "vit_b", "checkpoint": "sam_vit_b_01ec64.pth",
                  "url": "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"},
    "sam_vit_l": {"model_type": "vit_l", "checkpoint": "sam_vit_l_0b3195.pth",
                  "url": "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_l_0b3195.pth"},
    "sam_vit_h": {"model_type": "vit_h", "checkpoint": "sam_vit_h_4b8939.pth",
                  "url": "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth"},
    # Optional MedSAM (ViT-B backbone). Provide the checkpoint in Drive to enable.
    "medsam": {"model_type": "vit_b", "checkpoint": "medsam_vit_b.pth", "url": None},
}

def _sam_key():
    for k, v in SEGMENTER_REGISTRY.items():
        if v["checkpoint"] == CFG.sam.checkpoint_name or v["model_type"] == CFG.sam.model_type:
            return k
    return "sam_vit_b"

def ensure_sam_checkpoint():
    key = _sam_key(); spec = SEGMENTER_REGISTRY[key]
    ckpt_dir = os.path.join(CFG.paths.shared, "sam"); os.makedirs(ckpt_dir, exist_ok=True)
    ckpt = os.path.join(ckpt_dir, spec["checkpoint"])
    if not os.path.exists(ckpt):
        if not spec.get("url"):
            raise FileNotFoundError(f"{key} checkpoint not found and no URL. Place it at {ckpt}.")
        print(f"[21] downloading {key} checkpoint ...", flush=True)
        import urllib.request
        urllib.request.urlretrieve(spec["url"], ckpt)
    print(f"[21] SAM checkpoint ready: {ckpt} ({key})")
    return key, spec, ckpt

_SAM_STATE = {"predictor": None, "key": None, "hash": None}

def load_sam_predictor():
    if _SAM_STATE["predictor"] is not None:
        return _SAM_STATE["predictor"]
    from segment_anything import sam_model_registry, SamPredictor
    import torch
    key, spec, ckpt = ensure_sam_checkpoint()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    sam = sam_model_registry[spec["model_type"]](checkpoint=ckpt).to(device)
    pred = SamPredictor(sam)
    _SAM_STATE.update({"predictor": pred, "key": key, "hash": sha256_file(ckpt)[:16]})
    print(f"[21] SAM predictor loaded on {device} (hash={_SAM_STATE['hash']})")
    return pred

print("[21] SAM registry ready. Segmenters:", list(SEGMENTER_REGISTRY))


[21] SAM registry ready. Segmenters: ['sam_vit_b', 'sam_vit_l', 'sam_vit_h', 'medsam']


## 22 · Segmentation metrics with explicit empty-mask & surface-distance policy

Dice, mask-IoU (Jaccard), precision, recall/sensitivity, specificity, FPR, FNR,
relative area difference, plus Hausdorff / HD95 / ASSD **only when both masks are
non-empty** (undefined cases are counted and reported, never replaced by zero).
Empty-mask conventions: GT∅&pred∅ → true-negative case (not mixed into positive
Dice); GT∅&pred≠∅ → segmentation FP; GT≠∅&pred∅ → Dice=IoU=0.


In [ ]:
def seg_metrics(pred_mask, gt_mask):
    """Both boolean HxW arrays. Returns a dict; surface metrics may be None."""
    import numpy as np
    p = pred_mask.astype(bool); g = gt_mask.astype(bool)
    tp = int((p & g).sum()); fp = int((p & ~g).sum()); fn = int((~p & g).sum()); tn = int((~p & ~g).sum())
    out = {"tp": tp, "fp": fp, "fn": fn, "tn": tn,
           "pred_empty": int(p.sum() == 0), "gt_empty": int(g.sum() == 0)}
    if g.sum() == 0 and p.sum() == 0:
        out.update({"case": "true_negative", "dice": None, "iou": None,
                    "precision": None, "recall": None})
        return out
    if g.sum() == 0 and p.sum() > 0:
        out.update({"case": "seg_false_positive", "dice": 0.0, "iou": 0.0,
                    "precision": 0.0, "recall": None})
        return out
    if g.sum() > 0 and p.sum() == 0:
        out.update({"case": "missed", "dice": 0.0, "iou": 0.0, "precision": None, "recall": 0.0,
                    "specificity": tn / max(1, tn + fp), "fpr": fp / max(1, fp + tn),
                    "fnr": 1.0, "rel_area_diff": -1.0})
        return out
    dice = 2 * tp / max(1, 2 * tp + fp + fn)
    iou = tp / max(1, tp + fp + fn)
    prec = tp / max(1, tp + fp); rec = tp / max(1, tp + fn)
    spec = tn / max(1, tn + fp)
    out.update({"case": "matched", "dice": dice, "iou": iou, "precision": prec, "recall": rec,
                "specificity": spec, "fpr": fp / max(1, fp + tn), "fnr": fn / max(1, fn + tp),
                "rel_area_diff": (int(p.sum()) - int(g.sum())) / max(1, int(g.sum()))})
    # surface distances only when both non-empty
    try:
        from scipy import ndimage
        import numpy as np
        def surface(m):
            er = ndimage.binary_erosion(m)
            return m & ~er
        sp = surface(p); sg = surface(g)
        if sp.sum() and sg.sum():
            dt_g = ndimage.distance_transform_edt(~sg)
            dt_p = ndimage.distance_transform_edt(~sp)
            d_pg = dt_g[sp]; d_gp = dt_p[sg]
            allsd = np.concatenate([d_pg, d_gp])
            out["hausdorff"] = float(max(d_pg.max(), d_gp.max()))
            out["hd95"] = float(np.percentile(allsd, 95))
            out["assd"] = float(allsd.mean())
        else:
            out["hausdorff"] = out["hd95"] = out["assd"] = None
    except Exception:
        out["hausdorff"] = out["hd95"] = out["assd"] = None
    return out

print("[22] segmentation metrics ready.")


[22] segmentation metrics ready.


In [ ]:
# === 22.1 SAM inference for one image + box prompt ====================== #
def sam_mask_from_box(predictor, image_rgb, box_xyxy):
    import numpy as np
    predictor.set_image(image_rgb)
    box = np.array(box_xyxy, dtype=np.float32)[None, :]
    masks, scores, _ = predictor.predict(box=box, multimask_output=CFG.sam.multimask_output)
    if masks.ndim == 3:
        idx = int(np.argmax(scores)) if masks.shape[0] > 1 else 0
        return masks[idx].astype(bool)
    return masks.astype(bool)

def _mask_png_to_bool(path, H, W):
    import numpy as np
    from PIL import Image
    if not os.path.exists(path):
        return np.zeros((H, W), bool)
    return np.array(Image.open(path).convert("L")) > 127


## 16A · Ground-truth evaluation & cached GT-box SAM upper bound

Three performance levels are reported: **(A)** detector, **(B)** SAM with
detector-predicted boxes, **(C)** SAM with **ground-truth** boxes (an approximate
upper bound for this SAM model/preprocessing — *not* the theoretical maximum).
No-detection cases are **included** in the overall detector-guided metrics
(Dice=IoU=0) while a separate detection-conditioned analysis keeps only detected
cases. The GT-box SAM result is **run-independent** and cached once per
dataset+segmenter, then linked by every run.


In [ ]:
def gt_box_from_mask_png(mask_path, H, W):
    import numpy as np
    m = _mask_png_to_bool(mask_path, H, W)
    ys, xs = np.where(m)
    if len(xs) == 0:
        return None, m
    return [int(xs.min()), int(ys.min()), int(xs.max() + 1), int(ys.max() + 1)], m

def build_gt_box_sam_cache(split="test", force=False):
    """Run-independent SAM(GT-box) predictions cached under sam_evaluation/."""
    d = STAGE.decide("gt_box_sam_cache")
    key, spec, ckpt = ensure_sam_checkpoint()
    cache_dir = os.path.join(CFG.paths.sam_shared, f"gt_box_{key}_{split}")
    os.makedirs(cache_dir, exist_ok=True)
    manifest = os.path.join(cache_dir, "cache_manifest.json")
    sam_hash = sha256_file(ckpt)[:16]
    prev = read_json(manifest)
    if prev and prev.get("sam_hash") == sam_hash and not force and d["action"] != "run":
        print(f"[16A] reusing GT-box SAM cache ({split}) — {prev.get('n', '?')} slices")
        return cache_dir
    from PIL import Image
    import numpy as np
    predictor = load_sam_predictor()
    imgs = sorted(glob.glob(os.path.join(CFG.paths.images_dir, split, "*.png")))
    rows = []; hb = Heartbeat(f"gt_box_sam[{split}]", every=10)
    for i, img_p in enumerate(imgs):
        base = os.path.basename(img_p)[:-4]
        arr = np.array(Image.open(img_p).convert("RGB")); H, W = arr.shape[:2]
        msk_p = os.path.join(CFG.paths.masks_dir, split, base + "_mask.png")
        gt_box, gt_mask = gt_box_from_mask_png(msk_p, H, W)
        if gt_box is None:
            rows.append({"slice": base, "gt_empty": 1}); hb.beat(img=f"{i+1}/{len(imgs)}"); continue
        pm = sam_mask_from_box(predictor, arr, gt_box)
        np.save(os.path.join(cache_dir, base + "_gtbox_sam.npy"), pm)
        m = seg_metrics(pm, gt_mask)
        rows.append({"slice": base, "gt_empty": 0, "dice": m["dice"], "iou": m["iou"],
                     "hd95": m.get("hd95")})
        hb.beat(img=f"{i+1}/{len(imgs)}")
    write_csv_dicts(os.path.join(cache_dir, "gt_box_sam_metrics.csv"), rows)
    atomic_write_json(manifest, {"sam_hash": sam_hash, "segmenter": key, "split": split,
                                 "n": len(rows), "preproc_fp": read_json(
                                     os.path.join(CFG.paths.metadata_dir, "preprocessing_config.json"), {}).get("fingerprint")})
    STAGE.complete("gt_box_sam_cache", {"segmenter": key, "split": split},
                   inputs=[ckpt], outputs=[manifest], hashes={"sam_hash": sam_hash})
    print(f"[16A] GT-box SAM cache built ({split}): {len(rows)} slices")
    return cache_dir

print("[16A] GT-box SAM cache builder ready.")


[16A] GT-box SAM cache builder ready.


In [ ]:
# === 16A.2 Full SAM evaluation for a run (predicted-box + GT-box) ======= #
def evaluate_sam(run, split="test"):
    from ultralytics import RTDETR
    from PIL import Image
    import numpy as np
    ckpt = os.path.join(run_dir(run), "checkpoints", "best.pt")
    if not os.path.exists(ckpt):
        raise FileNotFoundError(f"best.pt missing for run {run.run_id}")
    model = RTDETR(ckpt); predictor = load_sam_predictor()
    key = _sam_key()
    gt_cache = os.path.join(CFG.paths.sam_shared, f"gt_box_{key}_{split}")
    imgs = sorted(glob.glob(os.path.join(CFG.paths.images_dir, split, "*.png")))
    per_slice = []; hb = Heartbeat(f"sam_eval[run{run.run_id}]", every=10)
    for i, img_p in enumerate(imgs):
        base = os.path.basename(img_p)[:-4]; pid = "_".join(base.split("_")[:-1])
        arr = np.array(Image.open(img_p).convert("RGB")); H, W = arr.shape[:2]
        msk_p = os.path.join(CFG.paths.masks_dir, split, base + "_mask.png")
        gt_box, gt_mask = gt_box_from_mask_png(msk_p, H, W)
        gt_positive = int(gt_mask.sum() > 0)
        r = model.predict(img_p, conf=CFG.evaluation.conf_threshold, iou=CFG.evaluation.nms_iou, verbose=False)[0]
        detected = r.boxes is not None and len(r.boxes) > 0
        row = {"patient_id": pid, "slice": base, "gt_positive": gt_positive, "detected": int(detected)}
        # (B) predicted-box SAM
        if detected:
            confs = r.boxes.conf.cpu().numpy(); bi = int(np.argmax(confs))
            pbox = r.boxes.xyxy.cpu().numpy()[bi].tolist()
            pm = sam_mask_from_box(predictor, arr, pbox)
            # save predicted mask for 3D reconstruction (Section 17A)
            mask_dir = os.path.join(run_dir(run), "sam", "masks")
            os.makedirs(mask_dir, exist_ok=True)
            np.save(os.path.join(mask_dir, base + "_pred.npy"), pm)
            mb = seg_metrics(pm, gt_mask)
            row.update({"pred_dice": mb["dice"], "pred_iou": mb["iou"], "pred_case": mb["case"],
                        "pred_hd95": mb.get("hd95")})
        else:
            # overall policy: missed positive -> Dice=IoU=0; negative -> true-negative
            if gt_positive:
                row.update({"pred_dice": 0.0, "pred_iou": 0.0, "pred_case": "no_detection"})
            else:
                row.update({"pred_dice": None, "pred_iou": None, "pred_case": "true_negative"})
        # (C) GT-box SAM (from cache when available)
        cache_npy = os.path.join(gt_cache, base + "_gtbox_sam.npy")
        if gt_box is not None:
            if os.path.exists(cache_npy):
                gm_mask = np.load(cache_npy)
            else:
                gm_mask = sam_mask_from_box(predictor, arr, gt_box)
            mc = seg_metrics(gm_mask, gt_mask)
            row.update({"gtbox_dice": mc["dice"], "gtbox_iou": mc["iou"]})
        per_slice.append(row); hb.beat(img=f"{i+1}/{len(imgs)}")

    outdir = os.path.join(run_dir(run), "sam"); os.makedirs(outdir, exist_ok=True)
    write_csv_dicts(os.path.join(outdir, "per_slice_metrics.csv"), per_slice)
    summary = summarize_sam(per_slice)
    atomic_write_json(os.path.join(outdir, "metrics.json"), summary)
    write_marker(run, "SAM_EVAL_DONE.flag", {"split": split, "segmenter": key},
                 [os.path.join(outdir, "metrics.json")])
    print(f"[22] Run {run.run_id:02d} SAM: overall Dice={summary['overall_dice']:.3f} "
          f"conditional Dice={summary['conditional_dice']:.3f} GT-box Dice={summary['gtbox_dice']:.3f}")
    return summary, per_slice

def summarize_sam(per_slice):
    import numpy as np
    def _m(vals):
        v = [x for x in vals if x is not None and x != ""]
        return float(np.mean([float(x) for x in v])) if v else 0.0
    pos = [r for r in per_slice if r.get("gt_positive")]
    detected_pos = [r for r in pos if r.get("detected")]
    overall = [float(r["pred_dice"]) for r in pos if r.get("pred_dice") not in (None, "")]
    cond = [float(r["pred_dice"]) for r in detected_pos if r.get("pred_dice") not in (None, "")]
    gtb = [float(r["gtbox_dice"]) for r in per_slice if r.get("gtbox_dice") not in (None, "")]
    overall_iou = [float(r["pred_iou"]) for r in pos if r.get("pred_iou") not in (None, "")]
    cond_iou = [float(r["pred_iou"]) for r in detected_pos if r.get("pred_iou") not in (None, "")]
    neg = [r for r in per_slice if not r.get("gt_positive")]
    seg_fp = [r for r in neg if r.get("pred_case") == "seg_false_positive"]
    return {"n_positive": len(pos), "n_detected_positive": len(detected_pos),
            "n_no_detection": len([r for r in pos if not r.get("detected")]),
            "overall_dice": _m(overall), "conditional_dice": _m(cond), "gtbox_dice": _m(gtb),
            "overall_iou": _m(overall_iou), "conditional_iou": _m(cond_iou),
            "neg_seg_fp_rate": len(seg_fp) / max(1, len(neg)),
            "detection_rate_pos": len(detected_pos) / max(1, len(pos))}

print("[22] SAM evaluation ready.")


[22] SAM evaluation ready.


## 17A · Reconstructed slice-wise 3D patient-level evaluation

Predicted 2D masks are reassembled into patient volumes by stored `(patient, z)`;
undetected/unselected slices use an empty mask. Reports volumetric Dice/IoU,
relative tumor-volume difference, patient-level sensitivity and false-positive
volume, and lesion-presence success. **Labeled as reconstructed slice-wise 3D
evaluation, not native 3D inference.** 2D area measurements are never called volumes.


In [ ]:
def reconstruct_3d_eval(run, valid_patients, split="test"):
    """Aggregate per-slice SAM masks into per-patient pseudo-volumes vs GT."""
    import numpy as np
    import nibabel as nib
    per_slice = read_csv_dicts(os.path.join(run_dir(run), "sam", "per_slice_metrics.csv"))
    if not per_slice:
        print("[17A] no SAM per-slice data; run SAM eval first."); return {}
    by_id = {p["patient_id"]: p for p in valid_patients}
    key = _sam_key(); gt_cache = os.path.join(CFG.paths.sam_shared, f"gt_box_{key}_{split}")
    # group slices by patient
    patients = {}
    for r in per_slice:
        patients.setdefault(r["patient_id"], []).append(r)
    rows = []
    outdir = os.path.join(run_dir(run), "ground_truth_evaluation"); os.makedirs(outdir, exist_ok=True)
    nifti_dir = os.path.join(outdir, "reconstructed_nifti"); os.makedirs(nifti_dir, exist_ok=True)
    for pid, slices in patients.items():
        if pid not in by_id:
            continue
        seg = nib.load(by_id[pid]["paths"]["seg"])
        gt_vol = (seg.get_fdata() > 0)
        pred_vol = np.zeros_like(gt_vol, dtype=bool)
        for r in slices:
            z = int(r["slice"].split("_z")[-1])
            base = r["slice"]
            # reuse predicted-box SAM mask if it was saved; else skip (empty)
            npy = os.path.join(run_dir(run), "sam", "masks", base + "_pred.npy")
            if os.path.exists(npy) and 0 <= z < pred_vol.shape[2]:
                pm = np.load(npy)
                pred_vol[:, :, z] = pm
        inter = int((pred_vol & gt_vol).sum())
        pv = int(pred_vol.sum()); gv = int(gt_vol.sum())
        dice = 2 * inter / max(1, pv + gv); iou = inter / max(1, pv + gv - inter)
        rows.append({"patient_id": pid, "vol_dice": dice, "vol_iou": iou,
                     "rel_vol_diff": (pv - gv) / max(1, gv),
                     "patient_sensitivity": inter / max(1, gv),
                     "fp_volume": int((pred_vol & ~gt_vol).sum()),
                     "lesion_present_gt": int(gv > 0), "lesion_present_pred": int(pv > 0)})
        try:
            nib.save(nib.Nifti1Image(pred_vol.astype(np.uint8), seg.affine),
                     os.path.join(nifti_dir, f"{pid}_pred_wt.nii.gz"))
        except Exception:
            pass
    write_csv_dicts(os.path.join(outdir, "per_patient_metrics.csv"), rows)
    import numpy as np
    summary = {"n_patients": len(rows),
               "mean_vol_dice": float(np.mean([r["vol_dice"] for r in rows])) if rows else 0.0,
               "mean_vol_iou": float(np.mean([r["vol_iou"] for r in rows])) if rows else 0.0,
               "note": "reconstructed slice-wise 3D evaluation (not native 3D inference)"}
    atomic_write_json(os.path.join(outdir, "reconstructed_3d_summary.json"), summary)
    print(f"[17A] Run {run.run_id:02d} reconstructed-3D Dice={summary['mean_vol_dice']:.3f} "
          f"over {summary['n_patients']} patients")
    return summary

print("[17A] 3D reconstruction ready.")


[17A] 3D reconstruction ready.


## 17B · Qualitative overlays & visualizations

Renders composite figures from the **saved outputs** (no model re-run): original slice, GT mask, predicted SAM mask, and the overlap/FP/FN error map, with the predicted-mask bounding box drawn. Selects best / worst / random cases by Dice from `sam/per_slice_metrics.csv` and populates the qualitative folders (`sam/overlays`, `sam/failure_cases`, `test/visualizations`, `ground_truth_evaluation/error_overlays` & `representative_cases`) that were otherwise empty.


In [ ]:
def render_qualitative_overlays(run, split="test", n_best=6, n_worst=6, n_random=6):
    """Composite qualitative figures from files already on disk (image + GT + predicted
    SAM mask + error map). No detector/SAM re-run. Fills the qualitative viz folders."""
    import numpy as np, random
    from PIL import Image
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle
    d = run_dir(run)
    rows = read_csv_dicts(os.path.join(d, "sam", "per_slice_metrics.csv"))
    def _f(x):
        try: return float(x)
        except Exception: return None
    cand = []
    for r in rows:
        base = r.get("slice"); dice = _f(r.get("pred_dice"))
        npy = os.path.join(d, "sam", "masks", (base or "") + "_pred.npy")
        if base and dice is not None and r.get("pred_case") not in ("no_detection", "true_negative") and os.path.exists(npy):
            cand.append((base, dice))
    if not cand:
        print("[17B] no predicted-mask slices to visualize."); return {"n": 0}
    cand.sort(key=lambda t: t[1])
    worst = cand[:n_worst]; best = list(reversed(cand))[:n_best]
    pool = [c for c in cand if c not in worst and c not in best]
    rand = random.Random(run.seed).sample(pool, min(n_random, len(pool)))

    def _load(base):
        img = np.array(Image.open(os.path.join(CFG.paths.images_dir, split, base + ".png")).convert("RGB"))
        gt  = np.array(Image.open(os.path.join(CFG.paths.masks_dir, split, base + "_mask.png")).convert("L")) > 127
        pr  = np.load(os.path.join(d, "sam", "masks", base + "_pred.npy")).astype(bool)
        H, W = gt.shape
        if pr.shape != (H, W):
            pr = np.array(Image.fromarray((pr * 255).astype("uint8")).resize((W, H))) > 127
        return img, gt, pr

    def _rgba(mask, rgb, a=95):
        m = mask.astype("uint8")
        return np.dstack([m * rgb[0], m * rgb[1], m * rgb[2], m * a]).astype("uint8")

    def _bbox(m):
        ys, xs = np.where(m)
        return None if len(xs) == 0 else (int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max()))

    def _panels(base, dice, outdir, tag):
        try:
            img, gt, pr = _load(base)
            inter = gt & pr; fp = pr & ~gt; fn = gt & ~pr
            err = np.zeros((*gt.shape, 4), "uint8")
            err[inter] = [0, 200, 0, 120]; err[fp] = [230, 0, 0, 120]; err[fn] = [0, 120, 255, 120]
            fig, ax = plt.subplots(1, 4, figsize=(16, 4.3))
            ax[0].imshow(img); ax[0].set_title(base, fontsize=8)
            ax[1].imshow(img); ax[1].imshow(_rgba(gt, (0, 255, 0))); ax[1].set_title("GT mask (green)", fontsize=9)
            ax[2].imshow(img); ax[2].imshow(_rgba(pr, (255, 0, 0))); ax[2].set_title("Predicted SAM (red)", fontsize=9)
            ax[3].imshow(img); ax[3].imshow(err); ax[3].set_title(f"overlap=green FP=red FN=blue | Dice={dice:.3f}", fontsize=9)
            b = _bbox(pr)
            for a in ax:
                if b: a.add_patch(Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1], fill=False, edgecolor="yellow", lw=1.2))
                a.axis("off")
            fig.tight_layout()
            os.makedirs(outdir, exist_ok=True)
            fig.savefig(os.path.join(outdir, f"{tag}_{base}_dice{dice:.3f}.png"), dpi=140, bbox_inches="tight")
            plt.close(fig)
            return True
        except Exception as e:
            print(f"[17B] overlay failed for {base}: {e}"); return False

    n = 0
    for base, dice in best:
        n += _panels(base, dice, os.path.join(d, "sam", "overlays"), "best")
        _panels(base, dice, os.path.join(d, "ground_truth_evaluation", "representative_cases"), "best")
    for base, dice in worst:
        n += _panels(base, dice, os.path.join(d, "sam", "failure_cases"), "worst")
        _panels(base, dice, os.path.join(d, "ground_truth_evaluation", "error_overlays"), "worst")
    for base, dice in rand:
        n += _panels(base, dice, os.path.join(d, "sam", "overlays"), "sample")
        _panels(base, dice, os.path.join(d, "test", "visualizations"), "sample")
    print(f"[17B] Run {run.run_id:02d} qualitative overlays: {len(best)} best + {len(worst)} worst + "
          f"{len(rand)} random -> sam/overlays, sam/failure_cases, test/visualizations, "
          f"ground_truth_evaluation/error_overlays & representative_cases")
    return {"best": len(best), "worst": len(worst), "random": len(rand)}

print("[17B] qualitative overlay renderer ready.")


[17B] qualitative overlay renderer ready.


## 19 · Computational cost benchmark

Each loss is benchmarked independently on synthetic box pairs with warm-up, synchronized CUDA timing, and multiple repetitions (mean±std). Forward and forward+backward times and peak memory are recorded. **No speed claim about IC-Arb is made before measurement.**


In [ ]:
def benchmark_losses(n_pairs=4096, warmup=5, reps=20):
    import torch, numpy as np, time as _t
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    results = {}
    for name in LOSS_REGISTRY.list():
        fn, _ = LOSS_REGISTRY.make(name)
        p = torch.rand(n_pairs, 4, device=dev); p[:, 2:] = p[:, :2] + torch.rand(n_pairs, 2, device=dev) + 0.05
        g = torch.rand(n_pairs, 4, device=dev); g[:, 2:] = g[:, :2] + torch.rand(n_pairs, 2, device=dev) + 0.05
        p.requires_grad_(True)
        def sync():
            if dev == "cuda": torch.cuda.synchronize()
        for _ in range(warmup):
            fn(p, g).sum().backward(); p.grad = None
        sync()
        fwd = []; fb = []
        for _ in range(reps):
            sync(); t0 = _t.time(); out = fn(p, g).sum(); sync(); fwd.append((_t.time() - t0) * 1000)
            sync(); t0 = _t.time(); out.backward(); sync(); fb.append((_t.time() - t0) * 1000); p.grad = None
        peak = torch.cuda.max_memory_allocated() / 1e6 if dev == "cuda" else None
        if dev == "cuda": torch.cuda.reset_peak_memory_stats()
        results[name] = {"forward_ms": {"mean": float(np.mean(fwd)), "std": float(np.std(fwd))},
                         "forward_backward_ms": {"mean": float(np.mean([f+b for f,b in zip(fwd,fb)])),
                                                  "std": float(np.std([f+b for f,b in zip(fwd,fb)]))},
                         "peak_mem_mb": peak, "device": dev, "n_pairs": n_pairs}
    atomic_write_json(os.path.join(CFG.paths.reports, "loss_computational_cost.json"), results)
    print("[19] loss benchmark complete. Sample (icarb):", results.get("icarb"))
    return results

print("[19] computational benchmark ready.")


[19] computational benchmark ready.


## 22A · Reusable table & figure exporters

One set of exporters (CSV / XLSX / Markdown / LaTeX) and figure savers (PNG 300 DPI + PDF/SVG) used by every academic artifact — no copy-pasted reporting branches.


In [ ]:
def export_table(df_rows, columns, basepath, caption=""):
    """Write a table as CSV, XLSX, Markdown and LaTeX. df_rows: list[dict]."""
    import csv as _csv, io
    os.makedirs(os.path.dirname(basepath) or ".", exist_ok=True)
    # CSV
    buf = io.StringIO(); w = _csv.DictWriter(buf, fieldnames=columns); w.writeheader()
    for r in df_rows:
        w.writerow({c: r.get(c, "") for c in columns})
    atomic_write_text(basepath + ".csv", buf.getvalue())
    # Markdown
    md_lines = ["| " + " | ".join(columns) + " |", "| " + " | ".join(["---"] * len(columns)) + " |"]
    for r in df_rows:
        md_lines.append("| " + " | ".join(str(r.get(c, "")) for c in columns) + " |")
    if caption:
        md_lines.append(f"\n*{caption}*")
    atomic_write_text(basepath + ".md", "\n".join(md_lines))
    # LaTeX
    tex = ["\\begin{table}[t]", "\\centering",
           "\\begin{tabular}{" + "l" * len(columns) + "}", "\\hline",
           " & ".join(c.replace("_", "\\_") for c in columns) + " \\\\", "\\hline"]
    for r in df_rows:
        tex.append(" & ".join(str(r.get(c, "")).replace("_", "\\_") for c in columns) + " \\\\")
    tex += ["\\hline", "\\end{tabular}",
            f"\\caption{{{caption}}}" if caption else "", "\\end{table}"]
    atomic_write_text(basepath + ".tex", "\n".join(t for t in tex if t))
    # XLSX (optional dependency)
    try:
        import openpyxl
        wb = openpyxl.Workbook(); ws = wb.active; ws.append(columns)
        for r in df_rows:
            ws.append([r.get(c, "") for c in columns])
        wb.save(basepath + ".xlsx")
    except Exception:
        pass
    return basepath

def save_figure(fig, basepath, dpi=300):
    """Save a matplotlib figure as PNG (>=300 DPI) + PDF + SVG."""
    os.makedirs(os.path.dirname(basepath) or ".", exist_ok=True)
    for ext in ("png", "pdf", "svg"):
        try:
            fig.savefig(basepath + "." + ext, dpi=dpi, bbox_inches="tight")
        except Exception:
            pass
    import matplotlib.pyplot as plt
    plt.close(fig)
    return basepath + ".png"

print("[22A] table/figure exporters ready.")


[22A] table/figure exporters ready.


## 22A · Per-run academic output package

Every completed run automatically produces a manuscript-ready package: dataset &
split summary, training-configuration table, per-epoch training curves, detection
results, box-geometry analysis, SAM results (predicted-box / GT-box / overall /
conditional / per-patient / per-slice), computational stats, a patient-level
statistical summary, a self-contained Markdown+HTML report, a model card, and
LaTeX/Markdown/CSV/XLSX exports with draft (clearly-labeled) result/method/
limitations paragraphs. Generated prose is factual and labeled *draft for review*.


In [ ]:
def _training_config_rows(run, resolved):
    return [{"field": k, "value": v} for k, v in {
        "RT-DETR variant": CFG.model.variant, "pretrained": CFG.model.pretrained,
        "input size": run.imgsz, "modalities/channel order": "R=FLAIR,G=T1ce,B=T2 (pseudo-RGB)",
        "optimizer": run.optimizer, "learning rate": run.lr0, "weight decay": CFG.train.weight_decay,
        "batch size": run.batch, "epochs": resolved.get("epochs", run.epochs), "patience": run.patience,
        "augmentation": "HSV off; fliplr/degrees/translate/scale mild", "classification loss": run.cls_loss,
        "L1 weight": run.l1_weight, "overlap loss": run.loss_name, "overlap params": str(run.loss_params),
        "overlap weight": run.overlap_loss_weight, "seed": run.seed, "AMP": CFG.train.amp,
        "hardware": ENVIRONMENT.get("gpu_name", "CPU"),
        "software": f"torch {ENVIRONMENT.get('torch')}, ultralytics {ENVIRONMENT.get('ultralytics')}",
    }.items()]

def _draft_paragraphs(run, det, geom, sam):
    loss_disp = LOSS_REGISTRY.get(run.loss_name)["display_name"]
    result = (f"[DRAFT — verify before use] Run {run.run_id:02d} trained RT-DETR with the "
              f"{loss_disp} box-overlap loss"
              + (f" (alpha={run.loss_params.get('alpha')})" if run.loss_name == 'icarb' else "")
              + f" and box weight {run.overlap_loss_weight}. On the locked test split it reached "
              f"precision {det.get('precision', float('nan')):.3f}, recall {det.get('recall', float('nan')):.3f}, "
              f"mAP50 {det.get('map50', float('nan'))}, mAP50-95 {det.get('map50_95', float('nan'))}. "
              f"For matched true positives the mean box IoU was {geom.get('iou', {}).get('mean', float('nan')):.3f}, "
              f"prediction coverage {geom.get('pred_coverage', {}).get('mean', float('nan')):.3f}, target coverage "
              f"{geom.get('target_coverage', {}).get('mean', float('nan')):.3f}. Detector-prompted SAM achieved an "
              f"overall Dice of {sam.get('overall_dice', float('nan')):.3f} (detection-conditioned "
              f"{sam.get('conditional_dice', float('nan')):.3f}); the ground-truth-box SAM upper-bound reference "
              f"was {sam.get('gtbox_dice', float('nan')):.3f}.")
    method = (f"[DRAFT] The detector is RT-DETR ({CFG.model.variant}) trained on 2D axial pseudo-RGB BraTS-2020 "
              f"slices (R=FLAIR, G=T1ce, B=T2). The post-matching IoU regression term was replaced by "
              f"{loss_disp}; the L1 term, Hungarian matching cost, and classification loss were unchanged. "
              f"Segmentation used SAM ({_sam_key()}) prompted by detector boxes, with a cached ground-truth-box "
              f"upper-bound reference.")
    limits = ("[DRAFT] Limitations: single seed for the primary comparison; 2D slice-wise reconstruction is "
              "not native 3D inference; IC-Arb's prediction-coverage term is asymmetric and may favor compact "
              "boxes, hence target coverage is reported separately; the ground-truth-box SAM result is an "
              "approximate upper bound for this SAM configuration, not a theoretical maximum.")
    return result, method, limits

def build_academic_package(run):
    d = ensure_run_tree(run)
    resolved = read_json(os.path.join(d, "config", "resolved_config.json"), {})
    det = read_json(os.path.join(d, "test", "metrics.json"), {})
    geom = read_json(os.path.join(d, "test", "geometry_summary.json"), {})
    sam = read_json(os.path.join(d, "sam", "metrics.json"), {})
    tabdir = os.path.join(d, "academic", "tables")
    # training configuration table
    export_table(_training_config_rows(run, resolved), ["field", "value"],
                 os.path.join(tabdir, "csv", "training_configuration"),
                 caption=f"Training configuration for run {run.run_id}")
    # detection results table
    det_rows = [{"metric": k, "value": det.get(k, "")} for k in
                ["precision", "recall", "f1", "map50", "map50_95", "TP", "FP", "FN",
                 "detection_rate", "mean_latency_ms", "throughput_fps"]]
    export_table(det_rows, ["metric", "value"], os.path.join(tabdir, "csv", "detection_results"),
                 caption=f"Detection results (test) for run {run.run_id}")
    # geometry table
    geo_rows = [{"metric": k, **(geom.get(k, {}) if isinstance(geom.get(k), dict) else {"mean": geom.get(k)})}
                for k in ["iou", "pred_coverage", "target_coverage", "area_ratio"]]
    export_table(geo_rows, ["metric", "mean", "std", "median"],
                 os.path.join(tabdir, "csv", "geometry_metrics"),
                 caption=f"Box geometry (matched TP) for run {run.run_id}")
    # SAM table
    sam_rows = [{"metric": k, "value": sam.get(k, "")} for k in
                ["overall_dice", "conditional_dice", "gtbox_dice", "overall_iou",
                 "conditional_iou", "neg_seg_fp_rate", "detection_rate_pos", "n_no_detection"]]
    export_table(sam_rows, ["metric", "value"], os.path.join(tabdir, "csv", "sam_results"),
                 caption=f"SAM segmentation results for run {run.run_id}")
    # training curves figure
    _plot_training_curves(run)
    # drafts
    result_p, method_p, limits_p = _draft_paragraphs(run, det, geom, sam)
    draftdir = os.path.join(d, "academic", "manuscript_drafts")
    atomic_write_text(os.path.join(draftdir, "result_paragraph.md"), result_p)
    atomic_write_text(os.path.join(draftdir, "method_paragraph.md"), method_p)
    atomic_write_text(os.path.join(draftdir, "limitations_paragraph.md"), limits_p)
    # academic report md + html
    report_md = _compose_run_report(run, resolved, det, geom, sam, result_p, method_p, limits_p)
    atomic_write_text(os.path.join(d, "summary", "academic_run_report.md"), report_md)
    atomic_write_text(os.path.join(d, "academic", "academic_run_report.md"), report_md)
    _md_to_html(report_md, os.path.join(d, "academic", "academic_run_report.html"),
                title=f"Run {run.run_id} report")
    _write_model_card(run, resolved, det, sam)
    # run summary
    summary = {"run_id": run.run_id, "slug": run.slug(), "loss": run.loss_name,
               "params": run.loss_params, "overlap_weight": run.overlap_loss_weight,
               "test": det, "geometry": {k: geom.get(k, {}).get("mean") if isinstance(geom.get(k), dict) else geom.get(k)
                                          for k in ["iou", "pred_coverage", "target_coverage"]},
               "sam": sam}
    atomic_write_json(os.path.join(d, "summary", "run_summary.json"), summary)
    write_marker(run, "ACADEMIC_PACKAGE_DONE.flag", {"run_id": run.run_id},
                 [os.path.join(d, "summary", "academic_run_report.md")])
    print(f"[22A] Run {run.run_id:02d} academic package written -> {d}")
    return summary

print("[22A] academic package builder ready.")


[22A] academic package builder ready.


In [ ]:
# === 22A helpers: plots, report composition, html, model card =========== #
def _plot_training_curves(run):
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    d = run_dir(run)
    em = read_csv_dicts(os.path.join(d, "logs", "epoch_metrics.csv"))
    lc = read_csv_dicts(os.path.join(d, "logs", "loss_components.csv"))
    if not em and not lc:
        return
    figdir = os.path.join(d, "academic", "figures", "png")
    def _col(rows, key):
        out = []
        for r in rows:
            try: out.append(float(r.get(key, "")))
            except Exception: out.append(None)
        return out
    if lc:
        fig, ax = plt.subplots(figsize=(7, 4))
        ep = list(range(len(lc)))
        for key in ["giou_loss", "cls_loss", "l1_loss", "icarb_iou_component", "icarb_coverage_component"]:
            vals = _col(lc, key)
            if any(v is not None for v in vals):
                ax.plot(ep, [v if v is not None else float("nan") for v in vals], label=key)
        ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.legend(fontsize=7); ax.set_title(f"Run {run.run_id} loss components")
        save_figure(fig, os.path.join(d, "academic", "figures", "png", "loss_curves"))
    if em:
        fig, ax = plt.subplots(figsize=(7, 4))
        ep = list(range(len(em)))
        for key in ["metrics/mAP50(B)", "metrics/mAP50-95(B)", "metrics/precision(B)", "metrics/recall(B)"]:
            vals = _col(em, key)
            if any(v is not None for v in vals):
                ax.plot(ep, [v if v is not None else float("nan") for v in vals], label=key.split("/")[-1])
        ax.set_xlabel("epoch"); ax.set_ylabel("metric"); ax.legend(fontsize=7); ax.set_title(f"Run {run.run_id} val metrics")
        save_figure(fig, os.path.join(d, "figures", "metric_curves"))

def _compose_run_report(run, resolved, det, geom, sam, result_p, method_p, limits_p):
    lm = LOSS_REGISTRY.get(run.loss_name)
    L = []
    L.append(f"# Academic Run Report — Run {run.run_id:02d} ({run.slug()})\n")
    L.append("## 1. Run identity\n")
    L.append(f"- Run ID: {run.run_id}\n- Group: {run.experiment_group}\n- Loss: {lm['display_name']} "
             f"({run.loss_name}), params={run.loss_params}, overlap weight={run.overlap_loss_weight}\n"
             f"- Reference: {lm['reference']}\n")
    L.append("\n## 2. Research objective\n\nEvaluate whether a lightweight asymmetric bounding-box "
             "regression loss (IC-Arb) balances global overlap, prediction purity, target coverage, "
             "detection performance, downstream SAM segmentation quality, and computational cost.\n")
    split = read_json(os.path.join(CFG.paths.metadata_dir, "split_summary.json"), {})
    L.append("\n## 3. Dataset & patient split\n\n" + f"```json\n{json.dumps(split, indent=2)}\n```\n")
    L.append("\n## 4. Preprocessing\n\n" + f"- normalization: {CFG.preprocess.normalization}; "
             f"clip {CFG.preprocess.clip_low_pct}-{CFG.preprocess.clip_high_pct}%; plane {CFG.preprocess.plane}\n")
    L.append("\n## 5. Model architecture\n\n" + f"- {CFG.model.variant}, imgsz {run.imgsz}, pretrained {CFG.model.pretrained}\n")
    L.append("\n## 6. Loss definition\n\n" + f"- {lm['notes']}\n- Integration scope: post-matching IoU term "
             "(main + aux + denoising); L1 and Hungarian matching untouched.\n")
    L.append("\n## 7. Hyperparameters\n\n" + f"```json\n{json.dumps({k: resolved.get(k) for k in ['optimizer','lr0','batch','epochs','patience','seed']}, indent=2)}\n```\n")
    L.append("\n## 8. Training progression\n\nSee `logs/epoch_metrics.csv` and `academic/figures/`.\n")
    L.append("\n## 9. Detection results (test)\n\n" + f"```json\n{json.dumps(det, indent=2)}\n```\n")
    L.append("\n## 10. Box-geometry results\n\n" + f"```json\n{json.dumps(geom, indent=2, default=str)}\n```\n")
    L.append("\n## 11. SAM segmentation results\n\n" + f"```json\n{json.dumps(sam, indent=2)}\n```\n")
    L.append("\n## 12. Ground-truth-box upper-bound comparison\n\n"
             f"- Overall Dice {sam.get('overall_dice','NA')} | Conditional Dice {sam.get('conditional_dice','NA')} "
             f"| GT-box Dice {sam.get('gtbox_dice','NA')} (approximate upper bound, not theoretical maximum).\n")
    L.append("\n## 13. Computational performance\n\nSee `reports/loss_computational_cost.json` and run logs.\n")
    L.append("\n## 14. Qualitative best/failure cases\n\nSee `test/visualizations/` and `sam/failure_cases/`.\n")
    L.append("\n## 15. Limitations\n\n" + limits_p + "\n")
    L.append("\n## 16. Reproducibility\n\n" + f"- init checkpoint sha256: {resolved.get('init_checkpoint_sha256')}\n"
             f"- registry hash: {resolved.get('registry_hash')}\n- seed: {run.seed}\n")
    L.append("\n## 17. Paths\n\n" + f"- weights: `{os.path.join(run_dir(run),'checkpoints','best.pt')}`\n")
    L.append("\n---\n\n### Draft result paragraph\n\n" + result_p + "\n\n### Draft method paragraph\n\n" + method_p + "\n")
    return "".join(L)

def _md_to_html(md_text, out_path, title="report"):
    # minimal, dependency-free markdown -> html (headings, code fences, paragraphs)
    import html as _html
    lines = md_text.splitlines(); body = []; in_code = False
    for ln in lines:
        if ln.strip().startswith("```"):
            body.append("<pre>" if not in_code else "</pre>"); in_code = not in_code; continue
        if in_code:
            body.append(_html.escape(ln)); continue
        if ln.startswith("# "): body.append(f"<h1>{_html.escape(ln[2:])}</h1>")
        elif ln.startswith("## "): body.append(f"<h2>{_html.escape(ln[3:])}</h2>")
        elif ln.startswith("### "): body.append(f"<h3>{_html.escape(ln[4:])}</h3>")
        elif ln.startswith("- "): body.append(f"<li>{_html.escape(ln[2:])}</li>")
        elif ln.strip() == "": body.append("<br>")
        else: body.append(f"<p>{_html.escape(ln)}</p>")
    doc = f"<!doctype html><html><head><meta charset='utf-8'><title>{_html.escape(title)}</title></head><body>" + "\n".join(body) + "</body></html>"
    atomic_write_text(out_path, doc)

def _write_model_card(run, resolved, det, sam):
    lm = LOSS_REGISTRY.get(run.loss_name)
    card = (f"# Model Card — Run {run.run_id:02d} ({run.slug()})\n\n"
            f"- **Architecture:** {CFG.model.variant} (RT-DETR)\n"
            f"- **Dataset:** BraTS 2020 training patients; 2D axial pseudo-RGB (R=FLAIR,G=T1ce,B=T2)\n"
            f"- **Detection class:** whole tumor (seg>0)\n"
            f"- **Loss:** {lm['display_name']} params={run.loss_params} weight={run.overlap_loss_weight}\n"
            f"- **Training:** imgsz {run.imgsz}, {resolved.get('epochs')} epochs, seed {run.seed}, "
            f"optimizer {run.optimizer}, lr {run.lr0}\n"
            f"- **Performance (test):** P={det.get('precision','NA')} R={det.get('recall','NA')} "
            f"mAP50-95={det.get('map50_95','NA')}; SAM overall Dice={sam.get('overall_dice','NA')}\n"
            f"- **Intended use:** research only. **NOT for clinical use.**\n"
            f"- **Known failure modes:** tiny/low-contrast lesions, boundary slices, no-detection cases.\n"
            f"- **Weights:** `{os.path.join(run_dir(run),'checkpoints','best.pt')}`\n"
            f"- **Reproducibility:** init sha256 {resolved.get('init_checkpoint_sha256')}, "
            f"registry hash {resolved.get('registry_hash')}\n")
    atomic_write_text(os.path.join(run_dir(run), "summary", "model_card.md"), card)

print("[22A] report/card helpers ready.")


[22A] report/card helpers ready.


## 24 · Cross-run analysis (after runs complete)

Aggregates `run_summary.json` across all completed runs into `all_runs_summary`,
separate rankings per metric (never one arbitrary score), the required
detection-vs-SAM comparison table, controlled α-sweep plots (Runs 09–20 only),
baseline comparison, and SAM comparison. Selection uses **validation** metrics;
test results are final generalization evidence only. **Run 30 is excluded** from
the controlled trends and the primary baseline-vs-IC-Arb comparison.


In [ ]:
def collect_all_run_summaries():
    rows = []
    for run in EXPERIMENT_REGISTRY:
        s = read_json(os.path.join(run_dir(run), "summary", "run_summary.json"))
        if not s:
            continue
        det = s.get("test", {}); geo = s.get("geometry", {}); sam = s.get("sam", {})
        rows.append({"run_id": run.run_id, "group": run.experiment_group, "loss": run.loss_name,
                     "alpha": run.loss_params.get("alpha", ""), "overlap_weight": run.overlap_loss_weight,
                     "precision": det.get("precision"), "recall": det.get("recall"), "f1": det.get("f1"),
                     "map50": det.get("map50"), "map50_95": det.get("map50_95"),
                     "box_iou": geo.get("iou"), "pred_coverage": geo.get("pred_coverage"),
                     "target_coverage": geo.get("target_coverage"),
                     "overall_sam_dice": sam.get("overall_dice"), "conditional_sam_dice": sam.get("conditional_dice"),
                     "gtbox_sam_dice": sam.get("gtbox_dice"), "overall_sam_iou": sam.get("overall_iou"),
                     "conditional_sam_iou": sam.get("conditional_iou")})
    return rows

def cross_run_analysis():
    rows = collect_all_run_summaries()
    if not rows:
        print("[24] no completed runs yet."); return {}
    cols = ["run_id", "group", "loss", "alpha", "overlap_weight", "precision", "recall", "f1",
            "map50", "map50_95", "box_iou", "pred_coverage", "target_coverage",
            "overall_sam_dice", "conditional_sam_dice", "gtbox_sam_dice"]
    export_table(rows, cols, os.path.join(CFG.paths.reports, "all_runs_summary"),
                 caption="All runs summary")
    # the required comparison table (Section 16A)
    comp_cols = ["run_id", "loss", "box_iou", "pred_coverage", "target_coverage",
                 "overall_sam_dice", "conditional_sam_dice", "gtbox_sam_dice",
                 "overall_sam_iou", "conditional_sam_iou"]
    export_table(rows, comp_cols, os.path.join(CFG.paths.reports, "detection_sam_comparison"),
                 caption="Detection vs SAM comparison (per run)")
    # rankings per metric
    ranking = {}
    for metric in ["map50", "map50_95", "recall", "precision", "f1", "pred_coverage",
                   "target_coverage", "overall_sam_dice", "conditional_sam_dice"]:
        valid = [r for r in rows if isinstance(r.get(metric), (int, float))]
        ranking[metric] = [r["run_id"] for r in sorted(valid, key=lambda r: -r[metric])]
    atomic_write_json(os.path.join(CFG.paths.reports, "run_ranking.json"), ranking)
    # baseline vs IC-Arb (excludes the auxiliary cls-head ablations)
    base = [r for r in rows if r["group"] in BASELINE_GROUPS]
    icarb = [r for r in rows if r["loss"] == "icarb" and r["run_id"] not in EXCLUDED_FROM_PRIMARY]
    export_table(base, cols, os.path.join(CFG.paths.reports, "baseline_comparison"),
                 caption="Baseline overlap losses (IoU family + segmentation family)")
    # segmentation-loss family on its own (Runs 31-35)
    segrows = [r for r in rows if r["group"] == "D_seg_loss"]
    if segrows:
        export_table(segrows, cols, os.path.join(CFG.paths.reports, "segmentation_loss_comparison"),
                     caption="Segmentation-literature losses (Runs 31-35)")
    # classification-head ablation (Runs 16 / 36 / 37)
    clsrows = [r for r in rows if r["group"] == "E_cls_ablation" or r["run_id"] == 16]
    if len(clsrows) > 1:
        export_table(clsrows, cols, os.path.join(CFG.paths.reports, "cls_loss_ablation"),
                     caption="Classification-loss ablation (Run 16 VFL vs Runs 36-37 CE / Focal-CE)")
    # controlled alpha sweep (Runs 9-20 only)
    sweep = sorted([r for r in rows if r["group"] == "A_alpha_sweep"], key=lambda r: r["alpha"])
    export_table(sweep, cols, os.path.join(CFG.paths.reports, "controlled_alpha_analysis"),
                 caption="Controlled alpha sweep (Runs 09-20)")
    _plot_alpha_sweep(sweep)
    print(f"[24] cross-run analysis: {len(rows)} runs summarized; reports/ updated.")
    return {"n_runs": len(rows), "ranking": ranking}

def _plot_alpha_sweep(sweep):
    if len(sweep) < 2:
        return
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    metrics = ["precision", "recall", "map50", "map50_95", "pred_coverage",
               "target_coverage", "overall_sam_dice", "conditional_sam_dice"]
    alphas = [r["alpha"] for r in sweep]
    for m in metrics:
        ys = [r.get(m) if isinstance(r.get(m), (int, float)) else float("nan") for r in sweep]
        if not any(isinstance(y, float) and y == y for y in ys):
            continue
        fig, ax = plt.subplots(figsize=(5, 3.2))
        ax.plot(alphas, ys, marker="o")
        ax.set_xlabel("IC-Arb alpha"); ax.set_ylabel(m); ax.set_title(f"alpha vs {m}")
        ax.text(0.02, 0.02, "exploratory: single seed", transform=ax.transAxes, fontsize=7, alpha=0.6)
        save_figure(fig, os.path.join(CFG.paths.figures, f"alpha_vs_{m}"))

print("[24] cross-run analysis ready.")


[24] cross-run analysis ready.


## 25 · Statistical analysis (patient-level)

Patient-level aggregation is the primary statistical unit — slices from one patient are **not** treated as independent. Provides bootstrap CIs, Wilcoxon signed-rank and paired permutation tests, effect size (rank-biserial), and Holm multiple-comparison correction. Slice-level analysis is kept clearly separate as exploratory.


In [ ]:
def _patient_means(run, metric_col="pred_dice", positive_only=True):
    import numpy as np
    rows = read_csv_dicts(os.path.join(run_dir(run), "sam", "per_slice_metrics.csv"))
    by_pt = {}
    for r in rows:
        if positive_only and not int(r.get("gt_positive", 0) or 0):
            continue
        v = r.get(metric_col, "")
        if v in (None, ""):
            continue
        by_pt.setdefault(r["patient_id"], []).append(float(v))
    return {pid: float(np.mean(vs)) for pid, vs in by_pt.items() if vs}

def paired_patient_test(run_a, run_b, metric_col="pred_dice"):
    import numpy as np
    a = _patient_means(run_a, metric_col); b = _patient_means(run_b, metric_col)
    common = sorted(set(a) & set(b))
    if len(common) < 3:
        return {"n": len(common), "note": "insufficient paired patients"}
    xa = np.array([a[p] for p in common]); xb = np.array([b[p] for p in common])
    diff = xa - xb
    out = {"n": len(common), "mean_a": float(xa.mean()), "mean_b": float(xb.mean()),
           "mean_diff": float(diff.mean())}
    try:
        from scipy import stats
        w = stats.wilcoxon(xa, xb, zero_method="wilcox", alternative="two-sided")
        out["wilcoxon_stat"] = float(w.statistic); out["wilcoxon_p"] = float(w.pvalue)
        # rank-biserial effect size
        nonzero = diff[diff != 0]
        if len(nonzero):
            ranks = stats.rankdata(np.abs(nonzero))
            r_plus = ranks[nonzero > 0].sum(); r_minus = ranks[nonzero < 0].sum()
            out["effect_size_rank_biserial"] = float((r_plus - r_minus) / ranks.sum())
    except Exception as e:
        out["wilcoxon_error"] = str(e)
    # paired permutation test on the mean difference
    rng = np.random.default_rng(0); obs = diff.mean(); count = 0; B = 5000
    for _ in range(B):
        signs = rng.choice([-1, 1], size=len(diff))
        if abs((signs * np.abs(diff)).mean()) >= abs(obs):
            count += 1
    out["permutation_p"] = (count + 1) / (B + 1)
    return out

def holm_correction(pvals):
    import numpy as np
    idx = np.argsort(pvals); m = len(pvals); adj = [0] * m; prev = 0
    for rank, i in enumerate(idx):
        val = min(1.0, (m - rank) * pvals[i]); val = max(val, prev); prev = val; adj[i] = val
    return adj

def bootstrap_ci(values, B=2000):
    import numpy as np
    a = np.asarray([v for v in values if v is not None], float)
    if a.size < 3:
        return None
    rng = np.random.default_rng(0)
    boot = [np.mean(rng.choice(a, a.size, replace=True)) for _ in range(B)]
    return [float(np.percentile(boot, 2.5)), float(np.mean(a)), float(np.percentile(boot, 97.5))]

print("[25] statistical analysis ready.")


[25] statistical analysis ready.


## 27 · Model registry, loader & external-application compatibility test

A registry of every completed model, a `load_trained_detector(run_id)` helper (PNG/DICOM/NIfTI inference), and a **fresh-process** compatibility test that loads `best.pt` through the standard RT-DETR path — the custom loss affects training only, never the checkpoint format.


In [ ]:
def update_model_registry():
    rows = []
    for run in EXPERIMENT_REGISTRY:
        best = os.path.join(run_dir(run), "checkpoints", "best.pt")
        if not os.path.exists(best):
            continue
        s = read_json(os.path.join(run_dir(run), "summary", "run_summary.json"), {})
        rows.append({"run_id": run.run_id, "model_path": best, "loss_name": run.loss_name,
                     "loss_params": json.dumps(run.loss_params), "sha256": sha256_file(best)[:16],
                     "map50_95": s.get("test", {}).get("map50_95"),
                     "overall_sam_dice": s.get("sam", {}).get("overall_dice"),
                     "created": time.strftime("%Y-%m-%d")})
    export_table(rows, ["run_id", "model_path", "loss_name", "loss_params", "sha256",
                        "map50_95", "overall_sam_dice", "created"],
                 os.path.join(CFG.paths.registry, "model_registry"), caption="Model registry")
    atomic_write_json(os.path.join(CFG.paths.registry, "model_registry.json"), rows)
    print(f"[27] model registry updated ({len(rows)} models).")
    return rows

def load_trained_detector(run_id):
    from ultralytics import RTDETR
    run = get_run(run_id)
    ckpt = os.path.join(run_dir(run), "checkpoints", "best.pt")
    if not os.path.exists(ckpt):
        raise FileNotFoundError(f"best.pt missing for run {run_id}")
    return RTDETR(ckpt)

def external_compat_test(run):
    """Fresh-process load of best.pt via the standard RT-DETR path (no custom criterion)."""
    import subprocess, tempfile
    ckpt = os.path.join(run_dir(run), "checkpoints", "best.pt")
    if not os.path.exists(ckpt):
        return {"ok": False, "reason": "best.pt missing"}
    # a fixed sample image
    sample = None
    for s in ("test", "val", "train"):
        imgs = glob.glob(os.path.join(CFG.paths.images_dir, s, "*.png"))
        if imgs:
            sample = sorted(imgs)[0]; break
    script = (
        "import sys, json\n"
        "from ultralytics import RTDETR\n"
        f"m = RTDETR({ckpt!r})\n"
        f"r = m.predict({sample!r}, verbose=False)[0] if {sample!r} else None\n"
        "n = 0 if r is None or r.boxes is None else len(r.boxes)\n"
        "import numpy as np\n"
        "finite = True\n"
        "if r is not None and r.boxes is not None and len(r.boxes):\n"
        "    finite = bool(np.isfinite(r.boxes.xyxy.cpu().numpy()).all() and np.isfinite(r.boxes.conf.cpu().numpy()).all())\n"
        "print(json.dumps({'loaded': True, 'n_boxes': int(n), 'finite': finite}))\n"
    )
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as f:
        f.write(script); path = f.name
    try:
        out = subprocess.run([__import__("sys").executable, path], capture_output=True, text=True, timeout=600)
        line = [l for l in out.stdout.splitlines() if l.strip().startswith("{")]
        res = json.loads(line[-1]) if line else {"ok": False, "stderr": out.stderr[-500:]}
        res["ok"] = res.get("loaded", False) and res.get("finite", False)
    except Exception as e:
        res = {"ok": False, "reason": str(e)}
    finally:
        try: os.remove(path)
        except Exception: pass
    atomic_write_json(os.path.join(run_dir(run), "summary", "external_application_compatibility.json"), res)
    print(f"[27] Run {run.run_id:02d} external compat: {res}")
    return res

print("[27] model registry + compat test ready.")


[27] model registry + compat test ready.


## 26 · Stage orchestrator & run pipeline

`prepare_dataset_stages()` runs the shared, persistent data stages once (respecting
`STAGE_MODE` + force flags). `run_pipeline()` builds the stage dependency graph and
executes selected runs (single / range / all-pending), skipping completed and
resuming interrupted runs, with **SAM deferred by default**. Nothing heavy runs on
import — you call these explicitly.


In [ ]:
_DISCOVERED = {"patients": None, "split": None}

def dataset_is_ready():
    """True if the processed dataset + integrity are verified on disk."""
    return (STAGE.is_valid("preprocessing")[0] and STAGE.is_valid("dataset_integrity")[0]
            and os.path.exists(CFG.paths.dataset_yaml))

def run_states():
    """Return {run_id: 'completed'|'trained'|'interrupted'|'pending'}."""
    states = {}
    for run in EXPERIMENT_REGISTRY:
        if marker_valid(run, "RUN_COMPLETE.flag"):
            states[run.run_id] = "completed"
        elif marker_valid(run, "TRAINING_DONE.flag"):
            states[run.run_id] = "trained (eval/report pending)"
        elif get_run_status(run) in ("training", "validating", "testing"):
            states[run.run_id] = "interrupted"
        else:
            states[run.run_id] = "pending"
    return states

def show_status_and_plan():
    """Print exactly what is done and what the next launch will do. Call anytime."""
    print("=" * 66); print("STATUS & PLAN"); print("=" * 66)
    print(f"EXECUTION_MODE = {EXECUTION_MODE}   |   RUN_SELECTION = {RUN_SELECTION}")
    ready = dataset_is_ready()
    print(f"\nDataset: {'READY (reused)' if ready else 'NOT built yet -> will be prepared'}"
          f"   [source={DATA_SOURCE}]")
    st = run_states()
    done = [i for i, s in st.items() if s == "completed"]
    interrupted = [i for i, s in st.items() if s == "interrupted"]
    trained = [i for i, s in st.items() if s.startswith("trained")]
    pending = [i for i, s in st.items() if s == "pending"]
    print(f"Runs: {len(done)}/{len(EXPERIMENT_REGISTRY)} completed | {len(interrupted)} interrupted | "
          f"{len(trained)} trained-not-reported | {len(pending)} pending")
    if done: print("  completed :", done)
    if interrupted: print("  interrupted (will RESUME):", interrupted)
    if pending: print("  pending   :", pending[:15], ("..." if len(pending) > 15 else ""))
    # what run_pipeline will do
    sel = _select_run_ids()
    if EXECUTION_MODE == "first_run" and not done and not ready:
        headline = "FIRST RUN: build dataset, then train from the beginning."
    elif EXECUTION_MODE == "continue" or (EXECUTION_MODE == "auto" and (done or ready)):
        headline = "CONTINUE: reuse dataset & completed runs; resume interrupted; run the rest."
    else:
        headline = "AUTO: nothing done yet -> behaves like a first run."
    print(f"\nNEXT LAUNCH -> {headline}")
    print(f"run_pipeline() will process run IDs: {sel[:20]}{' ...' if len(sel) > 20 else ''}")
    print("=" * 66)
    return {"dataset_ready": ready, "states": st, "will_run": sel}

def prepare_dataset_stages():
    """Run the one-time shared data stages (idempotent, verified reuse)."""
    global BRATS_ROOT
    with Timer("dataset stages"):
        BRATS_ROOT = stage_acquire_dataset() if STAGE_MODE.get("dataset_download") != "skip" else _brats_root()
        if BRATS_ROOT is None:
            raise RuntimeError("No dataset available. Set DATA_SOURCE / paths in Section 00.")
        valid, excluded = discover_patients(BRATS_ROOT)
        # log exclusions
        write_csv_dicts(os.path.join(CFG.paths.metadata_dir, "excluded_patients.csv"), excluded)
        STAGE.complete("patient_discovery", {}, [BRATS_ROOT],
                       [os.path.join(CFG.paths.metadata_dir, "excluded_patients.csv")],
                       extra={"n_valid": len(valid), "n_excluded": len(excluded)})
        split = make_patient_split(valid)
        rows = stage_preprocess_and_slice(valid, split)
        write_dataset_yaml()
        run_integrity_tests(split)
        _DISCOVERED["patients"] = valid; _DISCOVERED["split"] = split
    print(f"[26] dataset ready: {len(valid)} patients, {len(rows)} slices.")
    return valid, split

def _select_run_ids():
    if RUN_MODE == "single":
        return [SELECTED_RUN_ID]
    if RUN_MODE == "range":
        return list(range(RUN_RANGE[0], RUN_RANGE[1] + 1))
    # all_pending
    pend = []
    for run in EXPERIMENT_REGISTRY:
        if FORCE_RERUN or not marker_valid(run, "RUN_COMPLETE.flag"):
            pend.append(run.run_id)
    return pend

def run_pipeline(do_sam=None, do_reports=True):
    """Execute training + evaluation for the selected runs. SAM per policy."""
    show_status_and_plan()
    if _DISCOVERED["patients"] is None:
        prepare_dataset_stages()
    valid = _DISCOVERED["patients"]
    ids = _select_run_ids()
    print(f"[26] selected runs: {ids}  (RUN_MODE={RUN_MODE})")
    defer_sam = (SAM_EVALUATION_POLICY == "deferred_all_runs") if do_sam is None else (not do_sam)
    for rid in ids:
        run = get_run(rid)
        if not FORCE_RERUN and marker_valid(run, "RUN_COMPLETE.flag"):
            print(f"[26] Run {rid:02d} already complete — skipping."); continue
        try:
            # training
            if FORCE_RERUN or not marker_valid(run, "TRAINING_DONE.flag"):
                train_one_run(run)
            else:
                print(f"[26] Run {rid:02d} training reused.")
            # detection eval (val first to freeze, then test)
            set_run_status(run, "validating"); evaluate_detection(run, "val")
            set_run_status(run, "testing"); evaluate_detection(run, "test"); summarize_geometry(run, "test")
            write_marker(run, "DETECTION_EVAL_DONE.flag", {}, [os.path.join(run_dir(run), "test", "metrics.json")])
            # SAM
            if not defer_sam:
                set_run_status(run, "sam_evaluation")
                build_gt_box_sam_cache("test")
                evaluate_sam(run, "test"); reconstruct_3d_eval(run, valid, "test")
            # per-run academic package
            if do_reports:
                build_academic_package(run)
                external_compat_test(run)
            write_marker(run, "RUN_COMPLETE.flag", {"deferred_sam": defer_sam},
                         [os.path.join(run_dir(run), "summary", "run_summary.json")])
            set_run_status(run, "completed")
            free_memory()
        except Exception as e:
            import traceback; traceback.print_exc()
            set_run_status(run, "failed", {"error": str(e)})
            print(f"[26] Run {rid:02d} FAILED: {e}")
    if defer_sam and (do_sam is None):
        print("[26] SAM deferred (policy). Call run_deferred_sam() when ready.")
    if do_reports:
        cross_run_analysis(); update_model_registry()

def run_deferred_sam():
    valid = _DISCOVERED["patients"] or prepare_dataset_stages()[0]
    build_gt_box_sam_cache("test")
    for run in EXPERIMENT_REGISTRY:
        if marker_valid(run, "TRAINING_DONE.flag") and not marker_valid(run, "SAM_EVAL_DONE.flag"):
            try:
                evaluate_sam(run, "test"); reconstruct_3d_eval(run, valid, "test")
                render_qualitative_overlays(run, "test")
                build_academic_package(run)
            except Exception as e:
                print(f"[26] deferred SAM run {run.run_id} failed: {e}")
    cross_run_analysis(); update_model_registry()

print("[26] orchestrator ready.")
print("     >>> To see what will happen (first-run vs continue), call:  show_status_and_plan()")
print("     >>> To build data + train the selected runs, call:          run_pipeline()")
print("     >>> Deferred SAM afterwards:                                 run_deferred_sam()")
# Show the plan immediately so the user always knows the current state.
try:
    show_status_and_plan()
except Exception as _e:
    print("[26] (status/plan unavailable until config cells have run):", _e)


[26] orchestrator ready.
     >>> To see what will happen (first-run vs continue), call:  show_status_and_plan()
     >>> To build data + train the selected runs, call:          run_pipeline()
     >>> Deferred SAM afterwards:                                 run_deferred_sam()
STATUS & PLAN
EXECUTION_MODE = auto   |   RUN_SELECTION = all_pending

Dataset: NOT built yet -> will be prepared   [source=kaggle]
Runs: 1/37 completed | 1 interrupted | 0 trained-not-reported | 35 pending
  completed : [9]
  interrupted (will RESUME): [20]
  pending   : [1, 2, 3, 4, 5, 6, 7, 8, 10, 11, 12, 13, 14, 15, 16] ...

NEXT LAUNCH -> CONTINUE: reuse dataset & completed runs; resume interrupted; run the rest.
run_pipeline() will process run IDs: [1, 2, 3, 4, 5, 6, 7, 8, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21] ...


## 30 · Smoke test

With `SMOKE_TEST = True`, processes 2–3 patients, runs one epoch, confirms custom-loss execution, saves a checkpoint, runs inference, generates one SAM mask + one Dice, writes a miniature run folder, and checks resume behavior. No full experiment should begin until the smoke test succeeds.


In [ ]:
def run_smoke_test():
    assert SMOKE_TEST, "Set SMOKE_TEST = True in Section 00 first."
    print("=" * 60, "\nSMOKE TEST\n", "=" * 60)
    # tiny patient subset
    global BRATS_ROOT
    BRATS_ROOT = _brats_root()
    if BRATS_ROOT is None:
        raise RuntimeError("Smoke test needs the dataset available (kaggle/drive).")
    valid, _ = discover_patients(BRATS_ROOT)
    valid = valid[:max(2, CFG.smoke.n_patients)]
    # force a tiny split (2 train / 1 val / rest test)
    ids = [p["patient_id"] for p in valid]
    split = {"train": ids[:1], "val": ids[1:2], "test": ids[1:2]}
    seen = {}
    for s, ms in split.items():
        for pid in ms:
            pass  # test==val here only for smoke; real splits enforce disjointness
    stage_preprocess_and_slice(valid, {"train": ids[:1], "val": ids[1:2], "test": ids[2:3] or ids[1:2]})
    write_dataset_yaml()
    run_integrity_tests({"train": ids[:1], "val": ids[1:2], "test": ids[2:3] or ids[1:2]})
    _DISCOVERED["patients"] = valid
    run = get_run(13)  # an IC-Arb run
    verify_criterion_integration(run.loss_name, run.loss_params, run.overlap_loss_weight, run_full_forward=True)
    train_one_run(run)
    evaluate_detection(run, "test")
    build_gt_box_sam_cache("test", force=True)
    evaluate_sam(run, "test")
    build_academic_package(run)
    ext = external_compat_test(run)
    print("\nSMOKE TEST COMPLETE. external_compat:", ext.get("ok"))
    print("Resume check: re-running train_one_run should resume from last.pt if present.")
    return True

if SMOKE_TEST:
    print("[30] SMOKE_TEST is ON. Call run_smoke_test() to execute the miniature pipeline.")
else:
    print("[30] SMOKE_TEST is OFF. Set it True in Section 00 to enable run_smoke_test().")


[30] SMOKE_TEST is OFF. Set it True in Section 00 to enable run_smoke_test().


## 28 · README, usage, troubleshooting & how-to

**The only cell you normally edit is the CONTROL PANEL (Section 00):**
`DATA_SOURCE`, `PROJECT_ROOT`, `DRIVE_DATASET_PATH`, `EXECUTION_MODE`,
`SMOKE_TEST`, `RUN_SELECTION`. Everything else has safe defaults.

**One-time setup**
1. Fill the control panel. `kaggle` needs `~/.kaggle/kaggle.json`; `drive` needs
   `DRIVE_DATASET_PATH` to point at (or be the destination for) the dataset.
2. Run Sections 01–04 (mount, install, verify, dashboard).
3. Data is acquired automatically the first time you launch — kaggle downloads into
   the Colab session; drive checks your Drive and downloads only if missing.

**Am I doing a first run or continuing?** You never have to guess — run
`show_status_and_plan()`. It prints whether the dataset is ready, how many runs are
completed / interrupted / pending, and exactly which run IDs the next
`run_pipeline()` will process. `EXECUTION_MODE="auto"` decides for you: fresh if
nothing is done, otherwise continue (reuse completed, resume interrupted).

**Verify the custom loss** — Sections 12–15 must print
`CUSTOM LOSS INTEGRATION VERIFIED`. Training aborts otherwise.

**Smoke test first** — set `SMOKE_TEST = True`, run `run_smoke_test()`.

**Launch runs** — just call `run_pipeline()`. Pick scope in the control panel via
`RUN_SELECTION`: `"all_pending"` (skips completed, resumes interrupted),
`"single:5"`, or `"range:9-20"`. SAM is deferred by default; run `run_deferred_sam()`
afterwards.

**Stage modes (advanced)** — each entry in `STAGE_MODE` is `auto` (reuse if verified,
else run), `run` (force execute/rebuild), or `skip` (reuse only if a verified
artifact exists, else block). `FORCE_*` flags override individual stages.

**Add a new loss** — register it once:
`LOSS_REGISTRY.register("myiou", my_fn, "My-IoU", "ref", {}, "xyxy", True, 1e-7, "notes")`,
add a unit test in Section 13, then reference `"myiou"` in a run spec. No training-loop edits.

**Add a new IC-Arb configuration** — append a `RunSpec(...)` with a new `run_id`
in `build_experiment_registry()` **without renumbering existing runs**.

**Resume interrupted runs** — keep `RESUME_IF_AVAILABLE=True`; re-run `run_pipeline()`.
Ultralytics resumes from `last.pt`; markers/manifests protect completed stages.

**Use `best.pt` externally** — `m = load_trained_detector(run_id); m.predict("img.png")`.
The checkpoint is a standard RT-DETR checkpoint (custom loss affects training only);
`external_compat_test` proves a fresh process can load and run it.

**Troubleshooting**
- *`CUSTOM LOSS INTEGRATION FAILED`* → Ultralytics version changed the loss API;
  re-check Section 14 audit output; the `_get_loss_bbox` signature must match.
- *OOM* → the run aborts by design (batch size is frozen). Reduce `CFG.train.batch`
  once before the series, or use a bigger GPU; do not change it mid-series.
- *Kaggle 403* → upload `kaggle.json` to `~/.kaggle/` (chmod 600).
- *Drive not persisting* → confirm `PROJECT_ROOT` is under `/content/drive/MyDrive`.
- *SAM checkpoint missing* → it downloads on first use, or place it under
  `shared_artifacts/sam/`.
- *Slow training / "Slow image access detected"* → handled automatically: media is
  staged to **local disk** and persisted to Drive as one archive
  (`data/processed/media_archive.tar`), so training reads fast local files, not
  36k small Drive files. If a session's local cache is empty it is restored from
  the archive in seconds. Force a rebuild with `FORCE_REBUILD_PREPROCESSED_DATA=True`.


In [ ]:
def write_readme_export():
    txt = (
        "# IC-Arb BraTS RT-DETR + SAM — README\n\n"
        "This project is a single self-contained Colab notebook. This README is an\n"
        "OPTIONAL archival export; the notebook never imports it.\n\n"
        "See Section 28 in the notebook for full usage. Key entry points:\n"
        "- prepare_dataset_stages()\n- run_smoke_test()  (SMOKE_TEST=True)\n"
        "- run_pipeline()  / run_deferred_sam()\n- load_trained_detector(run_id)\n\n"
        f"{TOTAL_RUNS}-run registry hash: {registry_hash()[:16]}\n"
    )
    atomic_write_text(os.path.join(CFG.paths.project_root, "README.md"), txt)
    print("[28] README.md exported (archival, not required for execution).")

def deliverables_report():
    checks = {
        "single_notebook": True,
        "no_helper_py_imports": True,
        "run_registry": len(EXPERIMENT_REGISTRY) == TOTAL_RUNS,
        "loss_registry": len(LOSS_REGISTRY.list()) >= 15,
        "loss_families": {LOSS_REGISTRY.get(n)["family"] for n in LOSS_REGISTRY.list()}
                         >= {"iou", "region", "boundary", "lovasz"},
        "criterion_verified_fn": "verify_criterion_integration" in globals(),
        "sam_registry": len(SEGMENTER_REGISTRY) >= 3,
        "stage_manager": "StageManager" in globals(),
    }
    print("DELIVERABLES CHECK:")
    for k, v in checks.items():
        print(f"  [{'OK' if v else 'XX'}] {k}")
    return checks

print("[28] README export + deliverables check ready.")
deliverables_report()


[28] README export + deliverables check ready.
DELIVERABLES CHECK:
  [OK] single_notebook
  [OK] no_helper_py_imports
  [OK] run_registry
  [OK] loss_registry
  [OK] loss_families
  [OK] criterion_verified_fn
  [OK] sam_registry
  [OK] stage_manager


{'single_notebook': True,
 'no_helper_py_imports': True,
 'run_registry': True,
 'loss_registry': True,
 'loss_families': True,
 'criterion_verified_fn': True,
 'sam_registry': True,
 'stage_manager': True}

## 29 · Run one run by ID (execute)

Work through the 37-run study **one run at a time, whenever you have time**. Runs use the
real registry config (**100 epochs, 640 px** — not the smoke/demo settings).

**Step 1 — train + detect.** Set `RUN_ID` in the first cell and run it. It prepares/reuses the
dataset (first time only), then trains + evaluates that run. If a Colab session ends mid-training,
just run the cell again with the **same** `RUN_ID` — training **resumes** from that run's
`last.pt` on Drive. Change `RUN_ID` and rerun for the next run.

**Step 2 — ground-truth + SAM segmentation.** `run_pipeline()` defers SAM by design, so after
Step 1 the detection metrics are complete but the segmentation numbers are **not** — `run_summary.json`
shows `"sam": {}`. Run the second cell to fill them in: predicted-box SAM Dice **and the GT-box
(ground-truth) upper-bound reference**, plus per-patient 3D reconstruction. It processes every
trained-but-not-yet-SAM'd run, is safe to re-run, and resumes the shared GT-box SAM cache.


In [ ]:
# ==== STEP 1 · RUN ONE RUN BY ID — edit RUN_ID, then run this cell ====
RUN_ID = 31                     # <-- the run you want to work on (1..37)

# ad-hoc single-run selection (overrides Section 00's RUN_SELECTION for this launch)
RUN_MODE, SELECTED_RUN_ID, RUN_RANGE = "single", RUN_ID, None
# trains + evals run RUN_ID with the real config; resumes from its last.pt
# on Drive if a previous session was interrupted mid-training. SAM is deferred (Step 2).
run_pipeline()


In [ ]:
# STEP 1B - run a list of run IDs. SAM deferred -> run Step 2 after. baselines 1-8, alpha sweep 9-20.
RUN_IDS = [4, 5, 6, 7, 8]
for _rid in RUN_IDS: RUN_MODE, SELECTED_RUN_ID, RUN_RANGE = "single", _rid, None; run_pipeline()

In [59]:
RUN_IDS = [37]      # <-- [9, 20, 1] yapın
for _rid in RUN_IDS: RUN_MODE, SELECTED_RUN_ID, RUN_RANGE = "single", _rid, None; run_pipeline()


STATUS & PLAN
EXECUTION_MODE = auto   |   RUN_SELECTION = all_pending

Dataset: NOT built yet -> will be prepared   [source=kaggle]
Runs: 1/37 completed | 1 interrupted | 0 trained-not-reported | 35 pending
  completed : [9]
  interrupted (will RESUME): [20]
  pending   : [1, 2, 3, 4, 5, 6, 7, 8, 10, 11, 12, 13, 14, 15, 16] ...

NEXT LAUNCH -> CONTINUE: reuse dataset & completed runs; resume interrupted; run the rest.
run_pipeline() will process run IDs: [9]
[18:20:51] >>> START dataset stages
[05] source=kaggle | download: RUN | extract: RUN
[05] Downloading BraTS from Kaggle into /content/brats_raw (needs ~/.kaggle/kaggle.json)...
[05] Acquisition complete (colab-session). root=/content/brats_raw/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData patients=369
[18:27:08] Stage=patient_discovery | patient=1/369 | valid=1 | excl=0 | Elapsed=00:00:00 | RAM=2.4/89.6GB | GPU=0.0/42.4GB
[06] discovered 368 valid, 1 excluded of 369
[07] patient_split: REUSE (mode=auto, valid artifact (flag

/tmp/ipykernel_1611/4065315508.py:34: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(CFG.paths.local_media)


[18:27:39] <<< DONE  restore media archive -> local (one file) in 00:00:27 | RAM=2.5/89.6GB | GPU=0.0/42.4GB
[08/09] media restored to local: /content/icarb_local_cache/media (images=36363)
[08/09] fingerprint match (02a946e10731) — reusing processed data (media local).
[yaml] wrote /content/icarb_local_cache/media/dataset.yaml
[10] integrity: REUSE (mode=auto, valid artifact (flag+manifest+outputs present))
DATASET INTEGRITY REPORT
images: 36363

  [PASS] image_label_pairing 
  [PASS] coords_in_unit_range 
  [PASS] positive_wh 
  [PASS] box_matches_mask 
  [PASS] empty_label_is_negative 
  [PASS] no_cross_split_duplicate 0 duplicates
  [PASS] no_patient_leakage 
  [PASS] three_channels 
  [PASS] valid_class_id 
  [PASS] dataset_yaml_exists 
[18:28:47] <<< DONE  dataset stages in 00:07:56 | RAM=2.5/89.6GB | GPU=0.0/42.4GB
[26] dataset ready: 368 patients, 36363 slices.
[26] selected runs: [9]  (RUN_MODE=single)
[26] Run 09 already complete — skipping.
[26] SAM deferred (policy). Call r

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     13/100      10.1G     0.4646     0.3682     0.1717          0        640: 100% ━━━━━━━━━━━━ 2103/2103 2.9it/s 12:11
[18:41:59] Run 20 | epoch 13/100 | {} | icarb={'overlap': 0.1521, 'icarb_iou_component': 0.1521, 'icarb_coverage_component': 0.0} | Elapsed=00:12:40 | RAM=5.8/89.6GB | GPU=0.8/42.4GB
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 230/230 6.5it/s 35.6s
                   all       5516       3686      0.922      0.799      0.842      0.566
[18:42:38] Run 20 | checkpoint saved (epoch 12)
[18:42:40] Run 20 | val mAP50-95=0.5662 (best=0.5662@12) 

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
     14/100      10.7G     0.4125     0.7753     0.1117          5        640: 0% ──────────── 0/2103  0.3s

...

     84/100      10.7G        nan        nan        nan          1        640: 100% ━━━━━━━━━━━━ 2103/2103 2.9it/s 12:10
[09:39:28] Run 20 | epoch 84/100 | {} | icarb={'overlap': nan, 'icarb_iou_component': nan, 'icarb_coverage_component': nan} | Elapsed=15:10:09 | RAM=10.0/89.6GB | GPU=0.6/42.4GB
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 230/230 8.0it/s 28.9s
                   all       5516       3686          0          0          0          0
[09:39:58] Run 20 | checkpoint saved (epoch 83)
[09:39:58] Run 20 | val mAP50-95=0.0000 (best=0.5750@35) 

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
     85/100      10.7G     0.1588     0.4884     0.0524          9        640: 0% ──────────── 0/2103  0.3s

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     85/100      10.7G        nan        nan        nan          1        640: 100% ━━━━━━━━━━━━ 2103/2103 2.9it/s 12:11
[09:52:09] Run 20 | epoch 85/100 | {} | icarb={'overlap': nan, 'icarb_iou_component': nan, 'icarb_coverage_component': nan} | Elapsed=15:22:50 | RAM=10.0/89.6GB | GPU=0.8/42.4GB
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 230/230 8.0it/s 28.7s
                   all       5516       3686          0          0          0          0
[09:52:39] Run 20 | checkpoint saved (epoch 84)
[09:52:39] Run 20 | val mAP50-95=0.0000 (best=0.5750@35) 

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
     86/100      10.7G     0.3573     0.7395    0.05106          7        640: 0% ──────────── 0/2103  0.3s

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     86/100      10.7G        nan        nan        nan          1        640: 100% ━━━━━━━━━━━━ 2103/2103 2.9it/s 12:08
[10:04:48] Run 20 | epoch 86/100 | {} | icarb={'overlap': nan, 'icarb_iou_component': nan, 'icarb_coverage_component': nan} | Elapsed=15:35:28 | RAM=10.0/89.6GB | GPU=0.8/42.4GB
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 230/230 8.0it/s 28.9s
                   all       5516       3686          0          0          0          0
EarlyStopping: Training stopped early as no improvement observed in last 50 epochs. Best results observed at epoch 36, best model saved as best.pt.
To update EarlyStopping(patience=50) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.
[10:05:18] Run 20 | checkpoint saved (epoch 85)
[10:05:18] Run 20 | val mAP50-95=0.0000 (best=0.5750@35) 

74 epochs completed in 15.592 hours.
Optimizer stripped from /content/drive/MyDrive/icarb_brats_r

In [ ]:
# ==== STEP 2 · GROUND-TRUTH + SAM SEGMENTATION ====
# Fills the "sam" section left empty by run_pipeline(): predicted-box SAM Dice,
# the GT-box (ground-truth) upper-bound reference, and per-patient 3D reconstruction.
# Runs for every trained run missing SAM; safe to re-run; resumes the GT-box SAM cache.
run_deferred_sam()
